# **Task 1**


##  -- KB Parse Engine --

In [1]:
import re
import json
import logging
import argparse
import os
from markdown_it import MarkdownIt

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# INPUT LOADER  —  handles .md / .pdf / .txt → clean markdown string

# Known H2 section titles (normalised, used for heuristic heading detection)
_KNOWN_H2 = [
    "company overview", "north star metric", "north star",
    "target audience", "core features", "features",
    "allowed communication tones", "communication tones", "tones",
    "hook taxonomy", "hooks",
]
_KNOWN_H3_PATTERNS = [
    r"^ai tutor", r"^gamification", r"^progress reports",
    r"^role.based practice", r"^leaderboard\b",
    r"^approved tones?", r"^disallowed tones?",
    r"^allowed hooks?", r"^disallowed hooks?",
    r"^\w.+\(drive \d+\)",
]


def _clean_cid(text: str) -> str:
    """Remove (cid:NNN) PDF font artifacts, replacing bullet-like ones with '-'."""
    return re.sub(r'\(cid:\d+\)\s*', '- ', text)


def _heading_level(line: str) -> int:
    """Return 1/2/3 if line looks like a heading, else 0."""
    stripped = line.strip()
    if not stripped or len(stripped) > 120:
        return 0
    low = stripped.lower()

    if low in ("speakx company knowledge bank", "speakx kb", "knowledge bank"):
        return 1
    if any(low.startswith(h) for h in _KNOWN_H2):
        return 2
    for pat in _KNOWN_H3_PATTERNS:
        if re.match(pat, low):
            return 3

    if stripped == stripped.upper() and 3 < len(stripped) < 80:
        return 2 if len(stripped) < 40 else 3
    return 0


def _reconstruct_md(raw: str) -> str:
    raw   = _clean_cid(raw)
    lines = raw.splitlines()
    out   = []
    prev_blank = True

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if not prev_blank:
                out.append("")
            prev_blank = True
            continue

        level = _heading_level(stripped)
        if level:
            if not prev_blank:
                out.append("")
            out.append("#" * level + " " + stripped)
            out.append("")
            prev_blank = True
        else:
            if re.match(r'^[•\-*]\s+', stripped):
                out.append("- " + stripped.lstrip("•-*"))
            elif re.match(r'^\w.{0,60}:\s+\S', stripped) and len(stripped) < 120:
                out.append("- " + stripped)
            else:
                out.append(stripped)
            prev_blank = False

    return "\n".join(out)


def _pdf_to_md(path: str) -> str:
    try:
        import pdfplumber
    except ImportError:
        raise ImportError("pdfplumber required for PDF input: pip install pdfplumber")
    pages_text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2, y_tolerance=2)
            if text:
                pages_text.append(text)
    if not pages_text:
        raise ValueError("PDF appears to be scanned/image-based. Please provide a text-based PDF, .md, or .txt file.")
    return _reconstruct_md("\n".join(pages_text))


def _safe_read_text(path: str) -> str:
    """Read a text file trying multiple encodings before giving up."""
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    # Absolute fallback: read bytes and replace bad chars
    with open(path, "rb") as f:
        return f.read().decode("utf-8", errors="replace")


def _is_pdf(path: str) -> bool:
    """Detect PDF by magic bytes, not just extension."""
    try:
        with open(path, "rb") as f:
            return f.read(4) == b"%PDF"
    except Exception:
        return False


def load_any(path: str) -> str:
    """Load .md / .pdf / .txt (or any extension) → markdown-compatible string."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input file not found: {path}")

    ext = os.path.splitext(path)[1].lower()

    # Always detect PDF by magic bytes — never try to open it as text
    if ext == ".pdf" or _is_pdf(path):
        log.info("Loading PDF → reconstructing markdown: %s", path)
        return _pdf_to_md(path)

    if ext == ".md":
        log.info("Loading markdown: %s", path)
        return _safe_read_text(path)

    # .txt or unknown extension → safe text read
    if ext == ".txt":
        log.info("Loading TXT → reconstructing markdown: %s", path)
    else:
        log.warning("Unknown extension '%s', attempting plain-text load", ext)

    return _reconstruct_md(_safe_read_text(path))

# FUZZY NORMALISER  —  "NortStar", "North - - _Star", "north_star" → "north star"

def normalize(text: str) -> str:
    """
    Aggressive normalisation so typos / separators don't break matching.
    Steps:
      1. camelCase split         : NorthStar  → North Star
      2. strip non-alphanumeric  : North--_Star → North Star
      3. lowercase + collapse    : NORTH  STAR → north star
      4. common abbreviations    : ns → north star  (for section titles)
    """
    # camelCase → spaced
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', text)
    # replace any non-alphanumeric run with a single space
    text = re.sub(r'[^a-zA-Z0-9]+', ' ', text)
    text = text.lower().strip()
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text)
    return text


def _fuzzy_contains(haystack: str, needle: str, threshold: float = 0.75) -> bool:
    """
    True if needle fuzzy-matches anywhere inside haystack.
    Handles typos (NortStar), separators (Nort--_Star), casing, camelCase.
    Strategy:
      1. Strip everything except alphanumerics, lowercase both
      2. Substring check (fast path)
      3. SequenceMatcher ratio >= threshold (typo path)
    """
    import difflib

    def _alphanum(t: str) -> str:
        t = re.sub(r'([a-z])([A-Z])', r'\1 \2', t)
        t = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', t)
        return re.sub(r'[^a-zA-Z0-9]+', '', t).lower()

    h = _alphanum(haystack)
    n = _alphanum(needle)

    if not n:
        return False
    if n in h or h in n:
        return True
    return difflib.SequenceMatcher(None, h, n).ratio() >= threshold


# SCHEMA VALIDATOR

class SchemaValidator:
    SCHEMAS = {
        "north_star":  ["north_star", "justification", "measurable_proxy"],
        "feature_map": ["features"],
        "tone_hook":   ["allowed_tones", "disallowed_tones", "hooks"],
    }

    def validate(self, name: str, data: dict, strict: bool = True):
        missing = [f for f in self.SCHEMAS[name] if not data.get(f)]
        if missing:
            if strict:
                raise ValueError(f"[{name}] Missing required fields: {missing}")
            log.warning("[%s] missing fields: %s", name, missing)
        else:
            log.info("[%s] validation passed", name)


validator = SchemaValidator()

# SECTION BUILDER

def preprocess(text: str) -> str:
    lines = []
    for line in text.splitlines():
        line = re.sub(r'^(#{1,6})([^#\s])', r'\1 \2', line)
        line = re.sub(r'^(\s*[-*])([^\s\-*])', r'\1 \2', line)
        line = line.replace('\t', '    ')
        lines.append(line)
    return "\n".join(lines)


def get_sections(text: str) -> list:
    md = MarkdownIt()
    tokens = md.parse(preprocess(text))

    sections = []
    current = None
    in_list = False
    i = 0

    while i < len(tokens):
        tok = tokens[i]

        if tok.type == "heading_open":
            if current:
                sections.append(current)
            level = int(tok.tag[1])
            title = _strip_md(tokens[i + 1].content) if tokens[i + 1].type == "inline" else ""
            current = {"level": level, "title": title, "paragraphs": [], "items": []}
            i += 3
            continue

        if current is None:
            i += 1
            continue

        if tok.type in ("bullet_list_open", "ordered_list_open"):
            in_list = True
        elif tok.type in ("bullet_list_close", "ordered_list_close"):
            in_list = False
        elif tok.type == "inline" and tok.content.strip():
            if in_list:
                current["items"].append(tok.content.strip())
            else:
                current["paragraphs"].append(tok.content.strip())

        i += 1

    if current:
        sections.append(current)

    return sections


def _strip_md(text: str) -> str:
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'\*(.+?)\*',     r'\1', text)
    text = re.sub(r'__(.+?)__',     r'\1', text)
    return text.strip()


def find_section(sections: list, *keywords) -> dict | None:
    """
    Find a section whose title fuzzy-matches ANY of the given keywords.
    Tolerates typos, separators, casing, camelCase.
    """
    for s in sections:
        for kw in keywords:
            if _fuzzy_contains(s["title"], kw):
                return s
    return None


def child_sections(sections: list, parent: dict) -> list:
    result, found = [], False
    for s in sections:
        if s is parent:
            found = True
            continue
        if found:
            if s["level"] <= parent["level"]:
                break
            if s["level"] == parent["level"] + 1:
                result.append(s)
    return result


def label_value(text: str, *labels) -> str:
    for label in labels:
        m = re.search(rf'{re.escape(label)}\s*[:\-\u2013\u2014]\s*(.+)', text, re.IGNORECASE)
        if m:
            return _strip_md(m.group(1).strip())
    return ""


def backtick_value(text: str) -> str:
    m = re.search(r'`([^`]+)`', text)
    return m.group(1).strip() if m else ""


# PARSER 1  —  company_north_star.json

# All aliases that should resolve to "North Star"
_NORTH_STAR_ALIASES = [
    "north star", "northstar", "nortstar", "north_star",
    "ns metric", "nsm", "north star metric",
]

def parse_north_star(sections: list) -> dict:
    # Find section by fuzzy match on all known aliases
    sec = find_section(sections, *_NORTH_STAR_ALIASES)

    # Fallback: scan all section text for north-star keywords
    if not sec:
        for s in sections:
            combined = " ".join(s["paragraphs"] + s["items"])
            if any(_fuzzy_contains(combined, alias) for alias in _NORTH_STAR_ALIASES):
                log.warning("North Star section not found by title; using section '%s' by content match", s["title"])
                sec = s
                break

    if not sec:
        raise ValueError("North Star section not found in any format")

    all_text = sec["paragraphs"] + sec["items"]

    # ── North Star value ──
    north_star = ""

    # 1. Bold text on a line containing any north-star alias
    for line in all_text:
        m = re.search(r'\*\*(.+?)\*\*', line)
        if m and any(_fuzzy_contains(line, alias) for alias in _NORTH_STAR_ALIASES):
            north_star = m.group(1).strip()
            break

    # 2. "X is <value>" pattern
    if not north_star:
        for line in all_text:
            if any(_fuzzy_contains(line, alias) for alias in _NORTH_STAR_ALIASES):
                m = re.search(r'(?:is|=)\s+([A-Za-z][A-Za-z ]{2,60}?)(?:\s*[—\-–]|\.|,|$)', line)
                if m:
                    north_star = m.group(1).strip()
                    break

    # 3. Any bold text in the section
    if not north_star:
        for line in all_text:
            m = re.search(r'\*\*(.+?)\*\*', line)
            if m:
                north_star = m.group(1).strip()
                break

    # ── Measurable Proxy ──
    proxy = ""
    for line in all_text:
        proxy = backtick_value(line)
        if proxy:
            break
    if not proxy:
        for line in all_text:
            proxy = label_value(line, "Measurable Proxy", "Proxy", "Metric")
            if proxy:
                break

    if not proxy:
        for line in all_text:
            m = re.search(r'([a-z_]+\s*(?:>=|<=|==|>|<|!=)\s*\d+)', line)
            if m:
                proxy = m.group(1).strip()
                break

    # ── Justification ──
    justification = ""
    for line in all_text:
        justification = label_value(line, "Justification", "Why", "Rationale")
        if justification:
            break
    if not justification and len(sec["paragraphs"]) > 1:
        justification = _strip_md(sec["paragraphs"][-1])

    return {
        "north_star":       north_star,
        "justification":    justification,
        "measurable_proxy": proxy,
    }


# PARSER 2  —  feature_goal_map.json

def parse_feature_goal_map(sections: list) -> dict:
    parent = find_section(sections, "core features", "features", "corefeatures")
    if not parent:
        log.warning("Core Features section not found")
        return {"features": [], "lifecycle_map": {"trial": [], "paid": [], "churned": [], "inactive": []}}

    features = []
    lifecycle_map = {"trial": [], "paid": [], "churned": [], "inactive": []}

    for fs in child_sections(sections, parent):
        title = fs["title"]

        key_match    = re.search(r'\(([a-zA-Z0-9_,\s]+)\)', title)
        feature_key  = key_match.group(1).split(',')[0].strip() if key_match else re.sub(r'\W+', '_', title.lower()).strip('_')
        feature_name = re.sub(r'\s*\([^)]+\)', '', title).strip()

        lifecycle = []
        outcome   = ""
        primary_goal = ""

        for item in fs["items"]:
            lower = item.lower()
            if "lifecycle" in lower:
                val = label_value(item, "Lifecycle relevance", "Lifecycle")
                lifecycle = [x.strip().lower() for x in re.split(r'[;,]', val) if x.strip()]
            elif lower.startswith("outcome"):
                outcome = label_value(item, "Outcome")
            elif "primary goal" in lower or "primary" in lower:
                primary_goal = label_value(item, "Primary goal", "Primary")

        for stage in lifecycle:
            base = stage.split('(')[0].strip()
            if base in lifecycle_map:
                lifecycle_map[base].append(feature_name)

        features.append({
            "feature":      feature_name,
            "feature_key":  feature_key,
            "lifecycle":    lifecycle,
            "primary_goal": primary_goal,
            "outcome":      outcome,
        })

    return {"features": features, "lifecycle_map": lifecycle_map}


# PARSER 3  —  allowed_tone_hook_matrix.json


def parse_tone_hook_matrix(sections: list) -> dict:
    allowed_tones    = []
    disallowed_tones = []
    hooks            = []
    disallowed_hooks = []

    tone_parent = find_section(sections, "communication tone", "allowed communication", "tone")
    if tone_parent:
        for sub in child_sections(sections, tone_parent):
            is_dis = any(_fuzzy_contains(sub["title"], kw) for kw in ["disallow", "not allow", "forbidden"])
            for item in sub["items"]:
                m = re.match(r'\*\*(.+?)\*\*\s*[:\-]?\s*(.*)', item)
                if m:
                    tone_name = m.group(1).strip()
                    desc      = _strip_md(m.group(2).strip())
                else:
                    parts     = re.split(r'[:\-\u2013]', item, maxsplit=1)
                    tone_name = parts[0].strip()
                    desc      = parts[1].strip() if len(parts) > 1 else ""

                entry = {"tone": tone_name, "description": desc}
                if is_dis:
                    disallowed_tones.append(entry)
                else:
                    allowed_tones.append(entry)

    hook_parent = find_section(sections, "hook taxonomy", "hook", "octolysis")
    if hook_parent:
        for sub in child_sections(sections, hook_parent):
            is_dis = any(_fuzzy_contains(sub["title"], kw) for kw in ["disallow", "not allow", "forbidden"])

            if is_dis:
                disallowed_hooks.extend([_strip_md(i) for i in sub["items"]])
            else:
                dm = re.match(r'(.+?)\s*\(Drive\s*(\d+)\)', sub["title"])
                drive = {
                    "drive_id":      int(dm.group(2)) if dm else 0,
                    "drive_name":    dm.group(1).strip() if dm else sub["title"],
                    "example_hooks": [],
                    "use_context":   "",
                }
                for item in sub["items"]:
                    clean = _strip_md(item)
                    if re.match(r'hook use', clean, re.IGNORECASE):
                        drive["use_context"] = re.sub(r'^hook use\s*[:\-]?\s*', '', clean, flags=re.IGNORECASE)
                    else:
                        drive["example_hooks"].append(clean)
                hooks.append(drive)

    return {
        "allowed_tones":    allowed_tones,
        "disallowed_tones": disallowed_tones,
        "hooks": {
            "framework":  "Octolysis",
            "allowed":    hooks,
            "disallowed": disallowed_hooks,
        },
    }


# I/O


def save_json(data: dict, path: str):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    log.info("saved: %s", path)



# ENTRY POINT


def run(input_path: str, output_dir: str = "./output") -> dict:
    text     = load_any(input_path)
    sections = get_sections(text)

    log.info("input: %s", input_path)
    log.info("sections: %s", [s["title"] for s in sections])

    ns = parse_north_star(sections)
    validator.validate("north_star", ns)

    fm = parse_feature_goal_map(sections)
    validator.validate("feature_map", fm, strict=False)

    th = parse_tone_hook_matrix(sections)
    validator.validate("tone_hook", th, strict=False)

    save_json(ns, f"{output_dir}/company_north_star.json")
    save_json(fm, f"{output_dir}/feature_goal_map.json")
    save_json(th, f"{output_dir}/allowed_tone_hook_matrix.json")

    log.info("done - 3 files written to %s", output_dir)

    return {"north_star": ns, "feature_map": fm, "tone_hook": th}


if __name__ == "__main__":
    import os

    # 1. Define the extensions we are looking for
    valid_extensions = ('.txt', '.pdf', '.md')

    # 2. Scan the current directory (Colab's default is usually '/content')
    current_dir = "."
    found_files = [
        f for f in os.listdir(current_dir)
        if f.lower().endswith(valid_extensions) and os.path.isfile(f)
    ]

    # 3. Process the file if we found one
    if not found_files:
        log.error("No valid input files (.txt, .md, .pdf) found in the current directory.")
    else:
        # We'll just grab the first valid file we find
        input_file = found_files[0]
        output_directory = "./iteration_0_before_learning"

        print(f" Found valid file: {input_file}")
        print(f" Running with input: {input_file} and output directory: {output_directory}")

        try:
            run(input_file, output_directory)
        except Exception as e:
            log.error("Failed to process %s: %s", input_file, e)


 Found valid file: speakx_kb.md
 Running with input: speakx_kb.md and output directory: ./iteration_0_before_learning


In [ ]:
!ls -F

 classified_results.csv         segment_goals.csv
 communication_themes.csv       speakx_kb.txt
 experiment_results.csv         timing_recommendations.csv
 generation_log.csv	        user_behavioral_data.ipynb
 iteration_0_before_learning/  'user_behavioral_data - Sheet1 (1) (1).csv'
 iteration_1_after_learning/   'user_behavioral_data - Sheet1 (1).csv'
 message_templates.csv	       'user_behavioral_data - Sheet1.xls'
 output/		        user_notification_schedule.csv
 processed_users.csv	        user_segments.csv
 sample_data/


## -- Ingestion Engine --

In [2]:
from google.colab import files

uploaded = files.upload()

Saving user_behavioral_data - Sheet1 (1).csv to user_behavioral_data - Sheet1 (1).csv


In [3]:
import pandas as pd
import numpy as np
import os

# Get the name of the file you just uploaded
file_name = list(uploaded.keys())[0]

# Check the extension and read accordingly
if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
elif file_name.endswith(('.xls', '.xlsx')):
    df = pd.read_excel(file_name)
else:
    print("Unsupported file format!")

df.head()

,user_id,lifecycle_stage,days_since_signup,age_band,region,sessions_last_7d,exercises_completed_7d,streak_current,coins_balance,feature_ai_tutor_used,feature_leaderboard_viewed,feature_progress_checked,preferred_hour,notif_open_rate_30d,motivation_score
0,US_1,paid,119,25-34,tier2,5,16,22,453,True,False,True,7,0.304,0.65
1,US_2,churned,126,25-34,tier2,0,0,0,5,False,False,True,18,0.111,0.37
2,US_3,trial,3,25-34,tier2,9,9,5,90,True,True,True,9,0.435,0.51
3,US_4,paid,22,18-24,tier1,7,6,22,139,True,True,True,15,0.483,0.77
4,US_5,churned,117,25-34,tier2,0,0,0,40,False,False,False,9,0.129,0.39


In [4]:
required_columns = [
    'user_id',
    'age',
    'gender',
    'location',
    'lifecycle_stage',
    'sessions_last_7d',
    'exercises_completed_7d',
    'streak_current',
    'notif_open_rate_30d',
    'motivation_score',
    'feature_ai_tutor_used',
    'feature_leaderboard_used',
    'feature_social_used',
    'coins_balance',
    'preferred_hour'
]

missing_cols = [col for col in required_columns if col not in df.columns]

if missing_cols:
    print(" Missing columns:", missing_cols)
else:
    print(" Schema valid")


 Missing columns: ['age', 'gender', 'location', 'feature_leaderboard_used', 'feature_social_used']


In [5]:
# Numerical columns
num_cols = [
    'sessions_last_7d', 'exercises_completed_7d',
    'streak_current', 'notif_open_rate_30d',
    'motivation_score', 'coins_balance', 'preferred_hour'
]

# Categorical columns
cat_cols = ['gender', 'location', 'lifecycle_stage']

# Boolean columns
bool_cols = [
    'feature_ai_tutor_used',
    'feature_leaderboard_used',
    'feature_social_used'
]

# First, ensure all expected columns are present in the DataFrame.
# If a column is missing, add it with a default value (NaN for numerical/categorical, False for boolean).
import numpy as np

for col_list, default_val in [(num_cols, np.nan), (cat_cols, np.nan), (bool_cols, False)]:
    for col in col_list:
        if col not in df.columns:
            df[col] = default_val


# Numerical columns imputation
for col in num_cols:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# Categorical columns imputation
for col in cat_cols:
    if col in df.columns:
        # Check if mode() would return an empty Series (e.g., if all values are NaN after adding)
        if not df[col].mode().empty:
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            # Fallback for columns where mode cannot be calculated (e.g., all NaN), fill with 'Unknown'
            df[col].fillna('Unknown', inplace=True)

# Boolean columns imputation
for col in bool_cols:
    if col in df.columns:
        df[col].fillna(False, inplace=True)

print(" Missing values handled")

 Missing values handled


/tmp/ipykernel_223/2686877966.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_223/2686877966.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try us

In [6]:
def normalize(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn + 1e-9)


In [7]:
df['activeness_score'] = (
    0.4 * normalize(df['sessions_last_7d']) +
    0.3 * normalize(df['exercises_completed_7d']) +
    0.3 * normalize(df['streak_current'])
)


In [8]:
df['engagement_score'] = normalize(df['notif_open_rate_30d'])


In [9]:
df['churn_risk_score'] = (
    0.4 * (1 - df['activeness_score']) +
    0.3 * (1 - df['engagement_score']) +
    0.3 * (1 - normalize(df['motivation_score']))
)


In [10]:
df['gamification_score'] = (
    0.6 * normalize(df['coins_balance']) +
    0.4 * df['feature_leaderboard_viewed'].astype(int)
)

In [11]:
df['ai_tutor_score'] = (
    0.7 * df['feature_ai_tutor_used'].astype(int) +
    0.3 * normalize(df['exercises_completed_7d'])
)


In [12]:
df['leaderboard_score'] = (
    0.6 * df['feature_leaderboard_viewed'].astype(int) +
    0.4 * normalize(df['streak_current'])
)

In [13]:
df['social_score'] = (
    0.5 * df['feature_progress_checked'].astype(int) +
    0.3 * normalize(df['notif_open_rate_30d']) +
    0.2 * normalize(df['sessions_last_7d'])
)

In [14]:
def build_user_profile(row):
    return {
        "user_id": row['user_id'],

        # Demographics
        "demographics": {
            "gender": row['gender'],
            "location": row['location']
        },

        # Lifecycle
        "lifecycle": {
            "stage": row['lifecycle_stage']
        },

        # Behavior
        "behavior": {
            "sessions_last_7d": row['sessions_last_7d'],
            "exercises_completed_7d": row['exercises_completed_7d'],
            "streak": row['streak_current'],
            "notif_open_rate": row['notif_open_rate_30d']
        },

        # Derived Scores
        "scores": {
            "activeness": row['activeness_score'],
            "engagement": row['engagement_score'],
            "churn_risk": row['churn_risk_score'],
            "gamification": row['gamification_score'],
            "ai_tutor": row['ai_tutor_score'],
            "leaderboard": row['leaderboard_score'],
            "social": row['social_score']
        }
    }

df['user_profile'] = df.apply(build_user_profile, axis=1)


In [15]:
import json

print(json.dumps(df['user_profile'].iloc[0], indent=2))


{
  "user_id": "US_1",
  "demographics": {
    "gender": "Unknown",
    "location": "Unknown"
  },
  "lifecycle": {
    "stage": "paid"
  },
  "behavior": {
    "sessions_last_7d": 5,
    "exercises_completed_7d": 16,
    "streak": 22,
    "notif_open_rate": 0.304
  },
  "scores": {
    "activeness": 0.5795698924464356,
    "engagement": 0.49999999910394266,
    "churn_risk": 0.4170356798824433,
    "gamification": 0.4887272727263841,
    "ai_tutor": 0.9666666666518517,
    "leaderboard": 0.28387096773277837,
    "social": 0.6999999997286829
  }
}


In [16]:
import json
import os

profiles = df['user_profile'].tolist()

# Create the 'outputs' directory if it doesn't exist
output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "user_profiles.json"), "w") as f:
    json.dump(profiles, f, indent=2)

print(" Profiles saved")

 Profiles saved


In [17]:
df.to_csv("/content/processed_users.csv", index=False)

In [18]:
import pandas as pd
import numpy as np


In [19]:
df = pd.read_csv("/content/processed_users.csv")
df.head()


,user_id,lifecycle_stage,days_since_signup,age_band,region,sessions_last_7d,exercises_completed_7d,streak_current,coins_balance,feature_ai_tutor_used,...,feature_leaderboard_used,feature_social_used,activeness_score,engagement_score,churn_risk_score,gamification_score,ai_tutor_score,leaderboard_score,social_score,user_profile
0,US_1,paid,119,25-34,tier2,5,16,22,453,True,...,False,False,0.579570,0.500000,0.417036,0.488727,0.966667,0.283871,0.700000,"{'user_id': 'US_1', 'demographics': {'gender':..."
1,US_2,churned,126,25-34,tier2,0,0,0,5,False,...,False,False,0.000000,0.154122,0.848082,0.000000,0.000000,0.000000,0.546237,"{'user_id': 'US_2', 'demographics': {'gender':..."
2,US_3,trial,3,25-34,tier2,9,9,5,90,True,...,False,False,0.378387,0.734767,0.474806,0.492727,0.850000,0.664516,0.810430,"{'user_id': 'US_3', 'demographics': {'gender':..."
3,US_4,paid,22,18-24,tier1,7,6,22,139,True,...,False,False,0.452903,0.820789,0.330557,0.546182,0.800000,0.883871,0.816237,"{'user_id': 'US_4', 'demographics': {'gender':..."
4,US_5,churned,117,25-34,tier2,0,0,0,40,False,...,False,False,0.000000,0.186380,0.831586,0.038182,0.000000,0.000000,0.055914,"{'user_id': 'US_5', 'demographics': {'gender':..."


In [20]:
print(df.columns)


Index(['user_id', 'lifecycle_stage', 'days_since_signup', 'age_band', 'region',
       'sessions_last_7d', 'exercises_completed_7d', 'streak_current',
       'coins_balance', 'feature_ai_tutor_used', 'feature_leaderboard_viewed',
       'feature_progress_checked', 'preferred_hour', 'notif_open_rate_30d',
       'motivation_score', 'gender', 'location', 'feature_leaderboard_used',
       'feature_social_used', 'activeness_score', 'engagement_score',
       'churn_risk_score', 'gamification_score', 'ai_tutor_score',
       'leaderboard_score', 'social_score', 'user_profile'],
      dtype='object')


## -- Mece Segmentation Engine --

In [21]:

class MECESegmentationEngine:

    def __init__(self, df):
        self.df = df.copy()

    # ---------- BAND CREATION ----------
    def create_engagement_band(self):
        conditions = [
            self.df["activeness_score"] >= 0.7,
            (self.df["activeness_score"] >= 0.4) & (self.df["activeness_score"] < 0.7),
            self.df["activeness_score"] < 0.4
        ]
        labels = ["high", "moderate", "low"]
        self.df["engagement_band"] = np.select(conditions, labels, default="unknown")

    def create_churn_band(self):
        conditions = [
            self.df["churn_risk_score"] >= 0.7,
            (self.df["churn_risk_score"] >= 0.4) & (self.df["churn_risk_score"] < 0.7),
            self.df["churn_risk_score"] < 0.4
        ]
        labels = ["high_risk", "medium_risk", "low_risk"]
        self.df["churn_band"] = np.select(conditions, labels, default="unknown")

    # ---------- SEGMENT ASSIGNMENT ----------
    def assign_segments(self):

        def segment_logic(row):

            lifecycle = str(row.get("lifecycle_stage", "")).strip().lower()
            activeness = float(row.get("activeness_score", 0.0))
            churn_risk = float(row.get("churn_risk_score", 0.0))

            gamification = float(row.get("gamification_score", row.get("gamification_propensity", 0.0)))
            ai = float(row.get("ai_tutor_score", row.get("ai_tutor_propensity", 0.0)))
            leaderboard = float(row.get("leaderboard_score", row.get("leaderboard_propensity", 0.0)))
            social = float(row.get("social_score", row.get("social_propensity", 0.0)))

            # -------- CHURNED --------
            if lifecycle == "churned":
                return "Lost Customers"

            # -------- INACTIVE --------
            elif lifecycle == "inactive":
                return "Dormant Users"

            # -------- TRIAL USERS --------
            elif lifecycle == "trial":
                if churn_risk >= 0.5:
                    return "At-Risk Trialists"
                else:
                    return "High-Intent Trialists"

            # -------- PAID USERS & OTHERS --------
            else:
                if churn_risk >= 0.4:
                    return "Flight-Risk Subscribers"

                elif activeness < 0.5:
                    return "Slipping Subscribers"

                else:
                    propensities = {
                        "ai_driven": ai,
                        "community_driven": social,
                        "competition_driven": max(gamification, leaderboard)
                    }

                    dominant_trait = max(propensities, key=propensities.get)
                    sorted_scores = sorted(propensities.values(), reverse=True)
                    spread = sorted_scores[0] - sorted_scores[1]

                    if spread < 0.01:
                        return "Core Curriculum Users"

                    elif dominant_trait == "ai_driven":
                        return "AI-Powered Learners"

                    elif dominant_trait == "competition_driven":
                        return "Gamified Competitors"

                    elif dominant_trait == "community_driven":
                        return "Community Builders"

                    return "Core Curriculum Users"

        self.df["segment_id"] = self.df.apply(segment_logic, axis=1)

    # ---------- VALIDATION ----------
    def validate(self):
        if self.df["segment_id"].isnull().any():
            raise ValueError("Some users not assigned!")
        if (self.df["segment_id"] == "UNKNOWN").any():
            raise ValueError("Unknown segment found!")

    # ---------- RUN PIPELINE ----------
    def run(self):
        self.create_engagement_band()
        self.create_churn_band()
        self.assign_segments()
        self.validate()
        return self.df

In [22]:
import pandas as pd
df = pd.read_csv("/content/processed_users.csv")
engine = MECESegmentationEngine(df)
segmented_df = engine.run()

segmented_df.head()

,user_id,lifecycle_stage,days_since_signup,age_band,region,sessions_last_7d,exercises_completed_7d,streak_current,coins_balance,feature_ai_tutor_used,...,engagement_score,churn_risk_score,gamification_score,ai_tutor_score,leaderboard_score,social_score,user_profile,engagement_band,churn_band,segment_id
0,US_1,paid,119,25-34,tier2,5,16,22,453,True,...,0.500000,0.417036,0.488727,0.966667,0.283871,0.700000,"{'user_id': 'US_1', 'demographics': {'gender':...",moderate,medium_risk,Flight-Risk Subscribers
1,US_2,churned,126,25-34,tier2,0,0,0,5,False,...,0.154122,0.848082,0.000000,0.000000,0.000000,0.546237,"{'user_id': 'US_2', 'demographics': {'gender':...",low,high_risk,Lost Customers
2,US_3,trial,3,25-34,tier2,9,9,5,90,True,...,0.734767,0.474806,0.492727,0.850000,0.664516,0.810430,"{'user_id': 'US_3', 'demographics': {'gender':...",low,medium_risk,High-Intent Trialists
3,US_4,paid,22,18-24,tier1,7,6,22,139,True,...,0.820789,0.330557,0.546182,0.800000,0.883871,0.816237,"{'user_id': 'US_4', 'demographics': {'gender':...",moderate,low_risk,Slipping Subscribers
4,US_5,churned,117,25-34,tier2,0,0,0,40,False,...,0.186380,0.831586,0.038182,0.000000,0.000000,0.055914,"{'user_id': 'US_5', 'demographics': {'gender':...",low,high_risk,Lost Customers


In [23]:
segmented_df["segment_id"].value_counts()


,count
segment_id,
Flight-Risk Subscribers,12
High-Intent Trialists,10
Lost Customers,8
Dormant Users,8
Slipping Subscribers,6
At-Risk Trialists,6
AI-Powered Learners,4
Gamified Competitors,4
Community Builders,2


In [24]:
segmented_df.to_csv("/content/iteration_0_before_learning/user_segments.csv", index=False)

In [25]:
import pandas as pd

# Load the user segments data generated in the previous engine

try:
    df_segments = pd.read_csv('user_segments.csv')
    unique_segments = df_segments[['segment_id', 'lifecycle_stage']].drop_duplicates().reset_index(drop=True)
    print("Unique Segments and Lifecycle Stages Extracted:")
    display(unique_segments)
except FileNotFoundError:
    print("Error: 'user_segments.csv' not found. Please run the MECE Segmentation Engine first.")

Error: 'user_segments.csv' not found. Please run the MECE Segmentation Engine first.


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import warnings
warnings.filterwarnings("ignore")

print(" Libraries loaded.")

 Libraries loaded.


In [27]:
import pandas as pd
filename = '/content/iteration_0_before_learning/user_segments.csv'
segments_df = pd.read_csv(filename)

print(f"\n Loaded: {filename}")
print(f"   Rows: {len(segments_df)} | Columns: {len(segments_df.columns)}")
print(f"\nSegment distribution:")
print(segments_df["segment_id"].value_counts().to_string())

# Display the first 3 rows
segments_df.head(3)


 Loaded: /content/iteration_0_before_learning/user_segments.csv
   Rows: 61 | Columns: 30

Segment distribution:
segment_id
Flight-Risk Subscribers    12
High-Intent Trialists      10
Lost Customers              8
Dormant Users               8
Slipping Subscribers        6
At-Risk Trialists           6
AI-Powered Learners         4
Gamified Competitors        4
Community Builders          2
Core Curriculum Users       1


,user_id,lifecycle_stage,days_since_signup,age_band,region,sessions_last_7d,exercises_completed_7d,streak_current,coins_balance,feature_ai_tutor_used,...,engagement_score,churn_risk_score,gamification_score,ai_tutor_score,leaderboard_score,social_score,user_profile,engagement_band,churn_band,segment_id
0,US_1,paid,119,25-34,tier2,5,16,22,453,True,...,0.500000,0.417036,0.488727,0.966667,0.283871,0.700000,"{'user_id': 'US_1', 'demographics': {'gender':...",moderate,medium_risk,Flight-Risk Subscribers
1,US_2,churned,126,25-34,tier2,0,0,0,5,False,...,0.154122,0.848082,0.000000,0.000000,0.000000,0.546237,"{'user_id': 'US_2', 'demographics': {'gender':...",low,high_risk,Lost Customers
2,US_3,trial,3,25-34,tier2,9,9,5,90,True,...,0.734767,0.474806,0.492727,0.850000,0.664516,0.810430,"{'user_id': 'US_3', 'demographics': {'gender':...",low,medium_risk,High-Intent Trialists


## -- Goal Engine --

In [28]:
# 10 Segments × 3 journey phases each = 30 rows in segment_goals.csv
#
# Each phase defines:
#   primary_goal      — what the system is trying to achieve
#   sub_goals         — 2-3 supporting objectives
#   daily_theme       — emotional/behavioral intent per day window
#   notification_cta  — the single action we want the user to take
#   success_metric    — how we measure this journey worked

JOURNEY_KB = {

    # TRIAL SEGMENTS — D0-D7: conversion + early habit seed

    "At-Risk Trialists": [
        {
            "day_range":        "D0-D2",
            "primary_goal":     "Stop immediate drop-off",
            "sub_goals":        "Complete first lesson; Build first positive experience; Reduce perceived difficulty",
            "daily_theme":      "D0: Welcome + pick a simple topic (e.g. ordering food). D1: Celebrate completing any exercise. D2: 'You spoke English today — that's more than most people do.'",
            "notification_cta": "Open Sia and say your first sentence",
            "success_metric":   "First lesson completion rate",
        },
        {
            "day_range":        "D3-D5",
            "primary_goal":     "Demonstrate tangible progress before trial ends",
            "sub_goals":        "Show measurable improvement; Build 3-day streak; Trigger emotional investment",
            "daily_theme":      "D3: Mini progress card ('You used 12 new words this week'). D4: Streak reminder with bonus coins. D5: 'Your English is already better than Day 0 — here is proof.'",
            "notification_cta": "Check your progress report",
            "success_metric":   "3-day streak retention rate",
        },
        {
            "day_range":        "D6-D7",
            "primary_goal":     "Convert to paid before trial expires",
            "sub_goals":        "Create urgency; Surface loss of accumulated assets; Offer frictionless upgrade",
            "daily_theme":      "D6: 'Only 2 days left — your 5-day streak and 120 coins will be lost.' D7: Final warning showing exactly what they lose at midnight.",
            "notification_cta": "Upgrade now to keep your streak and coins",
            "success_metric":   "Trial-to-paid conversion rate",
        },
    ],

    "High-Intent Trialists": [
        {
            "day_range":        "D0-D1",
            "primary_goal":     "Secure first meaningful session within 24 hours",
            "sub_goals":        "Complete onboarding; Select learning goal; Start first AI Tutor conversation",
            "daily_theme":      "D0: 'Welcome! What do you want English to do for you — better job, travel, confidence?' D1: Reinforce their chosen goal back to them in the notification.",
            "notification_cta": "Continue your first lesson with Sia",
            "success_metric":   "Day-1 retention rate",
        },
        {
            "day_range":        "D2-D5",
            "primary_goal":     "Establish a consistent daily practice time",
            "sub_goals":        "Build 3-day streak; Lock in preferred session time; Introduce gamification rewards",
            "daily_theme":      "D2: Celebrate streak Day 2 with bonus coins. D3: '3 days straight — you are building a real habit.' D4: Introduce leaderboard. D5: Progress summary with encouragement.",
            "notification_cta": "Keep your streak alive — 5 minutes today",
            "success_metric":   "Sessions per user in D2-D5",
        },
        {
            "day_range":        "D6-D7",
            "primary_goal":     "Convert via aspiration not fear",
            "sub_goals":        "Reinforce sunk investment; Show cost-per-day framing; Tease premium scenarios",
            "daily_theme":      "D6: Personalized progress summary + premium feature tease. D7: 'You built a 7-day streak and learned 60+ words. Unlock full SpeakX for less than a coffee a day.'",
            "notification_cta": "Upgrade and keep your progress",
            "success_metric":   "Trial-to-paid conversion rate",
        },
    ],

    # PAID SEGMENTS — D8-D30: retention + feature deepening

    "Flight-Risk Subscribers": [
        {
            "day_range":        "D8-D12",
            "primary_goal":     "Restart daily sessions without triggering uninstall",
            "sub_goals":        "Get one session this week; Remind of existing assets; Remove all friction",
            "daily_theme":      "D8: Low-pressure nudge — 'Sia misses you. Just 2 minutes today?' D9-D10: No notification (breathing room). D11: Surface coins and streak stats. D12: Soft re-entry with micro-lesson.",
            "notification_cta": "Jump back in — your coins are waiting",
            "success_metric":   "Week-2 session restart rate",
        },
        {
            "day_range":        "D13-D20",
            "primary_goal":     "Rebuild habit before renewal decision point",
            "sub_goals":        "Complete 3 sessions; Reconnect with original learning goal; Show ROI of subscription",
            "daily_theme":      "D13: 'You joined to improve your English for work. Here is how far you have come.' D15-D18: Gentle daily reminders. D20: Progress comparison Day 1 vs now.",
            "notification_cta": "See how much you have improved",
            "success_metric":   "Churn rate reduction vs control group",
        },
        {
            "day_range":        "D21-D30",
            "primary_goal":     "Secure next billing cycle renewal",
            "sub_goals":        "Build 10-day streak; Introduce new feature; Create social proof",
            "daily_theme":      "D21: Introduce something new ('Have you tried role-play mode?'). D25: Mid-streak celebration. D30: '1 day from a 30-day badge — do not stop now.'",
            "notification_cta": "Try the new role-play scenario today",
            "success_metric":   "Month-2 renewal rate",
        },
    ],

    "Slipping Subscribers": [
        {
            "day_range":        "D8-D14",
            "primary_goal":     "Establish minimum viable daily practice (even 2 min counts)",
            "sub_goals":        "One session per day; Reduce session length expectation; Reward any engagement",
            "daily_theme":      "D8-D10: Push 2-minute micro-lessons only. D11: Coin bonus for 3 micro-lessons in a row. D12-D14: Gentle streak notifications at preferred hour.",
            "notification_cta": "2-minute practice — do it now",
            "success_metric":   "Daily active rate D8-D14",
        },
        {
            "day_range":        "D15-D22",
            "primary_goal":     "Increase session frequency from once to twice daily",
            "sub_goals":        "Introduce a second daily touchpoint; Connect to career goal; Build confidence",
            "daily_theme":      "D15: Introduce morning + evening split. D18: 'You completed 12 exercises this week — your best yet.' D22: Motivational message tied to their original goal.",
            "notification_cta": "Complete your evening practice session",
            "success_metric":   "Sessions per user per week",
        },
        {
            "day_range":        "D23-D30",
            "primary_goal":     "Lock in two daily sessions as the new normal",
            "sub_goals":        "Reach 7-day streak; Introduce new feature; Reward consistency",
            "daily_theme":      "D23-D26: Streak visual milestones. D27: Unlock a new conversation topic as reward. D28-D30: Month-end celebration and streak protection.",
            "notification_cta": "You are close to a 7-day streak — keep going",
            "success_metric":   "Month-end activeness score delta",
        },
    ],

    "AI-Powered Learners": [
        {
            "day_range":        "D8-D14",
            "primary_goal":     "Lock in a daily AI Tutor session at consistent time",
            "sub_goals":        "Complete 7 Sia conversations; Establish session timing habit; Introduce topic variety",
            "daily_theme":      "D8-D10: Daily check-in nudge at exact preferred_hour. D11: '4 conversations this week — try a new topic today.' D12-D14: Suggest a role-play they have not tried.",
            "notification_cta": "Start today's conversation with Sia",
            "success_metric":   "AI Tutor sessions per user per week",
        },
        {
            "day_range":        "D15-D22",
            "primary_goal":     "Move users from comfort topics to challenging role-plays",
            "sub_goals":        "Complete 2 high-stakes scenarios; Track vocabulary growth; Build confidence in unfamiliar topics",
            "daily_theme":      "D15: Introduce salary negotiation or interview scenario. D18: 'Your vocabulary grew by 80 words this month.' D20-D22: Encourage a completely new domain scenario.",
            "notification_cta": "Try the salary negotiation role-play today",
            "success_metric":   "Scenario variety index (unique scenarios tried)",
        },
        {
            "day_range":        "D23-D30",
            "primary_goal":     "User feels measurable personal growth and owns their progress",
            "sub_goals":        "Complete full month progress report; Save achievements; Re-commit for next month",
            "daily_theme":      "D23-D26: Deep progress analytics push. D27: '19 conversations with Sia this month — here is your vocabulary growth chart.' D28-D30: Month-end celebration + next month goal setting.",
            "notification_cta": "View your full monthly progress report",
            "success_metric":   "Monthly retention rate; Progress report open rate",
        },
    ],

    "Gamified Competitors": [
        {
            "day_range":        "D8-D10",
            "primary_goal":     "Get user placed and invested in the competitive leaderboard",
            "sub_goals":        "Complete placement exercises; Check leaderboard rank; Earn first weekly coins",
            "daily_theme":      "D8: 'Welcome to the Pro bracket — here is your starting rank.' D9: Show nearest competitor within striking distance. D10: Early week coins bonus for top-50% placement.",
            "notification_cta": "Check your leaderboard rank",
            "success_metric":   "Leaderboard check rate",
        },
        {
            "day_range":        "D11-D20",
            "primary_goal":     "Sustain daily sessions through rival-tracking notifications",
            "sub_goals":        "Maintain top-half leaderboard; Build 10-day streak; Complete daily challenges",
            "daily_theme":      "D11-D14: Mid-week rival update ('User_X just passed you — practice now to reclaim your rank'). D15: Top-25 entry badge. D16-D20: Daily competitive nudge at preferred hour.",
            "notification_cta": "Reclaim your rank — practice now",
            "success_metric":   "Daily sessions per user; Leaderboard engagement rate",
        },
        {
            "day_range":        "D21-D30",
            "primary_goal":     "End-of-month top-3 push for maximum emotional investment",
            "sub_goals":        "Reach top-10 leaderboard; Complete 30-day streak; Win end-of-month badge",
            "daily_theme":      "D21-D24: Countdown with live rank position. D25: 'Only 5 days left — 3 spots from the Top 10 badge.' D28-D30: Final sprint with daily urgent rank updates.",
            "notification_cta": "Final push — you are 2 spots from the Top 10 badge",
            "success_metric":   "Month-end leaderboard participation rate; Monthly retention",
        },
    ],

    "Community Builders": [
        {
            "day_range":        "D8-D14",
            "primary_goal":     "Connect user to at least 2 peers in the SpeakX community",
            "sub_goals":        "Follow 2 learners; Join a group challenge; Send or receive one social interaction",
            "daily_theme":      "D8: '3 people from your city joined SpeakX this week.' D10: Invite to a group speaking challenge. D12-D14: Peer activity updates ('Your friend completed 5 lessons today').",
            "notification_cta": "See what your community is learning today",
            "success_metric":   "Social feature engagement rate",
        },
        {
            "day_range":        "D15-D22",
            "primary_goal":     "Make user feel like a valued member not just a passive participant",
            "sub_goals":        "Complete a shared challenge; Share a milestone; Invite one friend",
            "daily_theme":      "D15: 'You are one of our most consistent learners — your streak inspires others.' D18: Invite a friend with bonus coins. D20-D22: Group challenge mid-point update.",
            "notification_cta": "Invite a friend and earn 100 bonus coins",
            "success_metric":   "Referral rate; Social sharing rate",
        },
        {
            "day_range":        "D23-D30",
            "primary_goal":     "User identifies as part of the SpeakX community — drives organic retention",
            "sub_goals":        "Complete month-end group challenge; Earn community badge; Re-commit for next month",
            "daily_theme":      "D23-D26: Community challenge final stretch. D27: 'You helped 3 friends start their English journey this month.' D28-D30: Community celebration with special badge.",
            "notification_cta": "Complete the month-end group challenge",
            "success_metric":   "Monthly retention; Community badge earn rate",
        },
    ],

    "Core Curriculum Users": [
        {
            "day_range":        "D8-D14",
            "primary_goal":     "Identify which feature creates strongest personal resonance",
            "sub_goals":        "Try AI Tutor, leaderboard, progress report at least once; Track highest follow-through feature",
            "daily_theme":      "D8-D10: AI Tutor deep dive. D11-D12: Progress report and achievements view. D13-D14: Light leaderboard intro ('See how you compare this week').",
            "notification_cta": "Explore a feature you have not tried yet",
            "success_metric":   "Feature variety index; Session follow-through rate",
        },
        {
            "day_range":        "D15-D22",
            "primary_goal":     "Double down on the feature that drove highest engagement in D8-D14",
            "sub_goals":        "Complete 5 sessions using dominant feature; Build consistent usage; Connect to learning goal",
            "daily_theme":      "D15: 'Based on your practice, you love [feature] — here is how to get even more from it.' D18: Progress snapshot tied to that feature. D22: Milestone celebration.",
            "notification_cta": "Continue your top feature session today",
            "success_metric":   "Dominant feature sessions per week",
        },
        {
            "day_range":        "D23-D30",
            "primary_goal":     "Tie all features into a complete daily routine",
            "sub_goals":        "Build 7-day streak; Complete cross-feature session; Month-end summary",
            "daily_theme":      "D23-D26: Cross-feature synthesis push. D27: 'Review your report, practice weak words with Sia, then check your rank.' D28-D30: Month-end wrap-up.",
            "notification_cta": "Complete your full daily routine today",
            "success_metric":   "Monthly retention; Feature diversity score",
        },
    ],

    # INACTIVE / CHURNED — Reactivation + Win-Back

    "Dormant Users": [
        {
            "day_range":        "D1-D5 (Post-Inactivity)",
            "primary_goal":     "Maintain brand warmth without triggering uninstall",
            "sub_goals":        "Stay top-of-mind without pressure; Deliver one value message; No conversion ask",
            "daily_theme":      "D1-D3: Complete silence. D4: One warm message — 'Learning takes time. Sia is here whenever you are ready.' D5: Optional — inspiring story from another learner.",
            "notification_cta": "Come back whenever you are ready — Sia is waiting",
            "success_metric":   "Uninstall rate on this window; D4 open rate",
        },
        {
            "day_range":        "D6-D14 (Post-Inactivity)",
            "primary_goal":     "One session restart with zero friction",
            "sub_goals":        "Complete one micro-lesson; Reconnect with personal goal; Trigger positive emotion",
            "daily_theme":      "D6: 'Just 1 minute with Sia today — no pressure, no streak reset.' D10: Nostalgia — show all-time best streak or top achievement. D14: Micro-commitment — '60 seconds, no speaking required.'",
            "notification_cta": "1-minute exercise — just to say hi to Sia",
            "success_metric":   "Reactivation rate (session restart %)",
        },
        {
            "day_range":        "D15+ (Post-Inactivity)",
            "primary_goal":     "Convert reactivated user into consistent weekly learner",
            "sub_goals":        "Complete 3 sessions in a week; Deliver exclusive incentive; Reset positive streak",
            "daily_theme":      "D15: '2x coins on every lesson this week — just for coming back.' D20: New feature or content they have not seen. D30+: If no response, reduce to 1 notification per week.",
            "notification_cta": "Claim your 2x coins welcome-back offer",
            "success_metric":   "Win-back conversion rate; Week-3 retention after reactivation",
        },
    ],

    "Lost Customers": [
        {
            "day_range":        "D1-D7 (Post-Churn)",
            "primary_goal":     "Trigger positive recall — remind them of their best moments",
            "sub_goals":        "Deliver one high-impact nostalgia message; No hard sell; Keep uninstall rate below 1%",
            "daily_theme":      "D1-D3: Silence. D4: 'You learned 500+ words with us. That knowledge is yours forever.' D7: Share personal best stats (longest streak, total exercises, words learned).",
            "notification_cta": "See everything you accomplished with SpeakX",
            "success_metric":   "Nostalgia message open rate; Uninstall rate on this window",
        },
        {
            "day_range":        "D8-D15 (Post-Churn)",
            "primary_goal":     "Remove the 'I already tried it' objection with genuinely new content",
            "sub_goals":        "Announce new feature or content update; Address likely churn reason; Offer frictionless re-entry",
            "daily_theme":      "D8: 'A lot has changed since you left — new role-plays, updated leaderboard, and more.' D12: Specific new feature deep-link. D15: 'Come back and see — no commitment needed.'",
            "notification_cta": "See what is new in SpeakX",
            "success_metric":   "Re-install or reactivation click rate",
        },
        {
            "day_range":        "D16-D30 (Post-Churn)",
            "primary_goal":     "Convert churned user to paid subscriber with targeted offer",
            "sub_goals":        "Deliver time-limited exclusive offer; Reduce price friction; Create urgency without pressure",
            "daily_theme":      "D16: '40% off your first month back — valid 72 hours only.' D20: Reminder if not clicked. D25: Social proof ('1,200 users came back this month'). D30+: If no response, 1 notification per month.",
            "notification_cta": "Claim your 40% win-back offer — 72 hours only",
            "success_metric":   "Churned user reactivation rate; Revenue recovered",
        },
    ],
}

print(" Journey Knowledge Base loaded.")
print(f"   Segments defined    : {len(JOURNEY_KB)}")
print(f"   Total journey phases: {sum(len(v) for v in JOURNEY_KB.values())}")
print(f"\nPhases per segment:")
for seg, phases in JOURNEY_KB.items():
    print(f"   {len(phases)} phases  →  {seg}")

 Journey Knowledge Base loaded.
   Segments defined    : 10
   Total journey phases: 30

Phases per segment:
   3 phases  →  At-Risk Trialists
   3 phases  →  High-Intent Trialists
   3 phases  →  Flight-Risk Subscribers
   3 phases  →  Slipping Subscribers
   3 phases  →  AI-Powered Learners
   3 phases  →  Gamified Competitors
   3 phases  →  Community Builders
   3 phases  →  Core Curriculum Users
   3 phases  →  Dormant Users
   3 phases  →  Lost Customers


In [29]:
# ── Lifecycle mapping
LIFECYCLE_MAP = {
    "At-Risk Trialists":       "trial",
    "High-Intent Trialists":   "trial",
    "Flight-Risk Subscribers": "paid",
    "Slipping Subscribers":    "paid",
    "AI-Powered Learners":     "paid",
    "Gamified Competitors":    "paid",
    "Community Builders":      "paid",
    "Core Curriculum Users":   "paid",
    "Dormant Users":           "inactive",
    "Lost Customers":          "churned",
}

# ── Flatten JOURNEY_KB into rows
rows = []
for segment_name, phases in JOURNEY_KB.items():
    for phase_data in phases:
        row = {"segment_id": segment_name}
        row["lifecycle_stage"] = LIFECYCLE_MAP.get(segment_name, "unknown")
        row.update(phase_data)
        rows.append(row)

segment_goals_df = pd.DataFrame(rows)

# ── Merge user counts from segments_df
seg_counts = (
    segments_df["segment_id"]
    .value_counts()
    .reset_index()
)
seg_counts.columns = ["segment_id", "user_count"]
segment_goals_df = segment_goals_df.merge(seg_counts, on="segment_id", how="left")
segment_goals_df["user_count"] = segment_goals_df["user_count"].fillna(0).astype(int)

# ── Final column order
FINAL_COLS = [
    "segment_id", "lifecycle_stage", "user_count",
    "day_range", "primary_goal", "sub_goals",
    "daily_theme", "success_metric",
]
segment_goals_df = segment_goals_df[FINAL_COLS]

print(f" segment_goals_df built")
print(f"   Rows    : {len(segment_goals_df)}")
print(f"   Columns : {len(segment_goals_df.columns)}")
print(f"\nPhases per segment:")
print(
    segment_goals_df.groupby("segment_id")["day_range"]
    .count()
    .sort_values(ascending=False)
    .to_string()
)
display(segment_goals_df.head(6))

 segment_goals_df built
   Rows    : 30
   Columns : 8

Phases per segment:
segment_id
AI-Powered Learners        3
At-Risk Trialists          3
Community Builders         3
Core Curriculum Users      3
Dormant Users              3
Flight-Risk Subscribers    3
Gamified Competitors       3
High-Intent Trialists      3
Lost Customers             3
Slipping Subscribers       3


,segment_id,lifecycle_stage,user_count,day_range,primary_goal,sub_goals,daily_theme,success_metric
0,At-Risk Trialists,trial,6,D0-D2,Stop immediate drop-off,Complete first lesson; Build first positive ex...,D0: Welcome + pick a simple topic (e.g. orderi...,First lesson completion rate
1,At-Risk Trialists,trial,6,D3-D5,Demonstrate tangible progress before trial ends,Show measurable improvement; Build 3-day strea...,D3: Mini progress card ('You used 12 new words...,3-day streak retention rate
2,At-Risk Trialists,trial,6,D6-D7,Convert to paid before trial expires,Create urgency; Surface loss of accumulated as...,D6: 'Only 2 days left — your 5-day streak and ...,Trial-to-paid conversion rate
3,High-Intent Trialists,trial,10,D0-D1,Secure first meaningful session within 24 hours,Complete onboarding; Select learning goal; Sta...,D0: 'Welcome! What do you want English to do f...,Day-1 retention rate
4,High-Intent Trialists,trial,10,D2-D5,Establish a consistent daily practice time,Build 3-day streak; Lock in preferred session ...,D2: Celebrate streak Day 2 with bonus coins. D...,Sessions per user in D2-D5
5,High-Intent Trialists,trial,10,D6-D7,Convert via aspiration not fear,Reinforce sunk investment; Show cost-per-day f...,D6: Personalized progress summary + premium fe...,Trial-to-paid conversion rate


In [30]:
# ── Lifecycle mapping
LIFECYCLE_MAP = {
    "At-Risk Trialists":       "trial",
    "High-Intent Trialists":   "trial",
    "Flight-Risk Subscribers": "paid",
    "Slipping Subscribers":    "paid",
    "AI-Powered Learners":     "paid",
    "Gamified Competitors":    "paid",
    "Community Builders":      "paid",
    "Core Curriculum Users":   "paid",
    "Dormant Users":           "inactive",
    "Lost Customers":          "churned",
}

# ── Flatten JOURNEY_KB into rows
rows = []
for segment_name, phases in JOURNEY_KB.items():
    for phase_data in phases:
        row = {"segment_id": segment_name}
        row["lifecycle_stage"] = LIFECYCLE_MAP.get(segment_name, "unknown")
        row.update(phase_data)
        rows.append(row)

segment_goals_df = pd.DataFrame(rows)

# ── Merge user counts from segments_df
seg_counts = (
    segments_df["segment_id"]
    .value_counts()
    .reset_index()
)
seg_counts.columns = ["segment_id", "user_count"]
segment_goals_df = segment_goals_df.merge(seg_counts, on="segment_id", how="left")
segment_goals_df["user_count"] = segment_goals_df["user_count"].fillna(0).astype(int)

# ── Final column order
FINAL_COLS = [
    "segment_id", "lifecycle_stage", "user_count",
    "day_range", "primary_goal", "sub_goals",
    "daily_theme", "notification_cta",
    "success_metric",
]
segment_goals_df = segment_goals_df[FINAL_COLS]

print(f" segment_goals_df built")
print(f"   Rows    : {len(segment_goals_df)}")
print(f"   Columns : {len(segment_goals_df.columns)}")
print(f"\nPhases per segment:")
print(
    segment_goals_df.groupby("segment_id")["day_range"] # Use day_range as a proxy for phase count
    .count()
    .sort_values(ascending=False)
    .to_string()
)
display(segment_goals_df.head(6))

 segment_goals_df built
   Rows    : 30
   Columns : 9

Phases per segment:
segment_id
AI-Powered Learners        3
At-Risk Trialists          3
Community Builders         3
Core Curriculum Users      3
Dormant Users              3
Flight-Risk Subscribers    3
Gamified Competitors       3
High-Intent Trialists      3
Lost Customers             3
Slipping Subscribers       3


,segment_id,lifecycle_stage,user_count,day_range,primary_goal,sub_goals,daily_theme,notification_cta,success_metric
0,At-Risk Trialists,trial,6,D0-D2,Stop immediate drop-off,Complete first lesson; Build first positive ex...,D0: Welcome + pick a simple topic (e.g. orderi...,Open Sia and say your first sentence,First lesson completion rate
1,At-Risk Trialists,trial,6,D3-D5,Demonstrate tangible progress before trial ends,Show measurable improvement; Build 3-day strea...,D3: Mini progress card ('You used 12 new words...,Check your progress report,3-day streak retention rate
2,At-Risk Trialists,trial,6,D6-D7,Convert to paid before trial expires,Create urgency; Surface loss of accumulated as...,D6: 'Only 2 days left — your 5-day streak and ...,Upgrade now to keep your streak and coins,Trial-to-paid conversion rate
3,High-Intent Trialists,trial,10,D0-D1,Secure first meaningful session within 24 hours,Complete onboarding; Select learning goal; Sta...,D0: 'Welcome! What do you want English to do f...,Continue your first lesson with Sia,Day-1 retention rate
4,High-Intent Trialists,trial,10,D2-D5,Establish a consistent daily practice time,Build 3-day streak; Lock in preferred session ...,D2: Celebrate streak Day 2 with bonus coins. D...,Keep your streak alive — 5 minutes today,Sessions per user in D2-D5
5,High-Intent Trialists,trial,10,D6-D7,Convert via aspiration not fear,Reinforce sunk investment; Show cost-per-day f...,D6: Personalized progress summary + premium fe...,Upgrade and keep your progress,Trial-to-paid conversion rate


In [31]:
output_path = "iteration_0_before_learning/segment_goals.csv"
segment_goals_df.to_csv(output_path, index=False)

# **Task 2**

## -- Theme Engine --

In [32]:

"""
THEME ENGINE — EdTech Communication System (Task 2)

Input:  user_segments.csv   (from Task 1 segmentation)
Output: communication_themes.csv
        theme_engine_debug_clusters.csv

Architecture:
  Layer 1 — Business-safe default mappings (always available)
  Layer 2 — K-means refinement inside each segment (conditional)
             → cluster profiling → rulebook → aggregation → safety overrides

Theme vocabulary: Octalysis 8 Core Drives (closed set)

"""

# 0. IMPORTS

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from copy import deepcopy


# 1. CONFIGURATION & CONSTANTS

RANDOM_STATE = 42
KMEANS_N_INIT = 20

# Octalysis 8 Core Drives — closed vocabulary
OCTALYSIS_THEMES = {
    "Epic Meaning",
    "Accomplishment",
    "Empowerment",
    "Ownership",
    "Social Influence",
    "Scarcity",
    "Unpredictability",
    "Loss Avoidance",
}

# ── Segment categories ──
RISK_SEGMENTS     = {"At-Risk Trialists", "Flight-Risk Subscribers"}
RECOVERY_SEGMENTS = {"Dormant Users", "Slipping Subscribers", "Lost Customers"}
GROWTH_SEGMENTS   = {
    "High-Intent Trialists",
    "AI-Powered Learners",
    "Gamified Competitors",
    "Community Builders",
    "Core Curriculum Users",
}

# ── K-means eligibility thresholds ──
KMEANS_MIN_SIZE      = 20
SMALL_CLUSTER_MIN    = 8
CLUSTER_SHARE_MIN    = 0.10

# ── Risk / energy thresholds ──
CHURN_HIGH   = 0.60
CHURN_MED    = 0.35
ENERGY_HIGH  = 0.65
ENERGY_MED   = 0.35

# ── Dominant motivation thresholds ──
DOM_TOP_MIN      = 0.45
DOM_SPREAD_MIN   = 0.07
COMP_LEADER_THR  = 0.55
COMP_SOCIAL_THR  = 0.45

# ── Confidence scoring weights ──
W_SIZE = 0.4
W_MOT  = 0.4
W_RULE = 0.2

CONF_HIGH_THR   = 0.85
CONF_MED_THR    = 0.65

CONF_MULT = {"High": 1.00, "Medium": 0.75, "Low": 0.50}


# 2. THEME-FAMILY → HOOK DICTIONARY  (back-fill / validation)


THEME_HOOKS = {
    "Epic Meaning":      ["future_self", "real_life_confidence", "meaningful_progress", "life_impact"],
    "Accomplishment":    ["streak_continuation", "milestone_progress", "points_milestone",
                          "weekly_target", "rank_jump"],
    "Empowerment":       ["easy_restart", "two_min_practice", "ai_personalized_practice",
                          "guided_practice", "choose_your_path", "ai_feedback"],
    "Ownership":         ["your_learning_path", "your_progress", "continue_where_left",
                          "continue_plan", "plan_waiting"],
    "Social Influence":  ["peer_progress", "group_challenge", "shared_milestone",
                          "community_streak", "learn_together"],
    "Scarcity":          ["time_bound_challenge", "today_only_bonus", "expiring_opportunity",
                          "deadline_finish"],
    "Unpredictability":  ["surprise_challenge", "today_special_prompt", "new_prompt", "new_twist"],
    "Loss Avoidance":    ["protect_streak", "protect_progress", "preserve_momentum",
                          "don't_miss_today", "protect_trial_momentum"],
}


# 3. DEFAULT SEGMENT MAPPINGS  (Layer 1 / fallback)


DEFAULT_MAPPINGS = {
    "Lost Customers": {
        "primary_theme":    "Epic Meaning",
        "secondary_theme":  "Empowerment",
        "tone_preferences": ["empathetic", "aspirational", "gentle"],
        "hooks":            ["future_self", "real_life_confidence", "restart_small", "rebuild_confidence"],
    },
    "Dormant Users": {
        "primary_theme":    "Unpredictability",
        "secondary_theme":  "Empowerment",
        "tone_preferences": ["curiosity-led", "supportive", "low-pressure"],
        "hooks":            ["surprise_challenge", "easy_restart", "two_min_practice", "new_prompt"],
    },
    "At-Risk Trialists": {
        "primary_theme":    "Loss Avoidance",
        "secondary_theme":  "Empowerment",
        "tone_preferences": ["urgent-but-supportive", "reassuring"],
        "hooks":            ["protect_trial_momentum", "small_restart", "don't_miss_today", "quick_ai_practice"],
    },
    "High-Intent Trialists": {
        "primary_theme":    "Accomplishment",
        "secondary_theme":  "Ownership",
        "tone_preferences": ["motivating", "progress-focused", "encouraging"],
        "hooks":            ["streak_start", "milestone_progress", "your_learning_path", "next_step"],
    },
    "Flight-Risk Subscribers": {
        "primary_theme":    "Loss Avoidance",
        "secondary_theme":  "Ownership",
        "tone_preferences": ["urgent-but-supportive", "reassuring", "continuity-focused"],
        "hooks":            ["protect_streak", "protect_progress", "continue_plan", "restart_small"],
    },
    "Slipping Subscribers": {
        "primary_theme":    "Empowerment",
        "secondary_theme":  "Ownership",
        "tone_preferences": ["supportive", "gentle", "encouraging"],
        "hooks":            ["easy_restart", "continue_where_left", "plan_waiting", "momentum_recovery"],
    },
    "AI-Powered Learners": {
        "primary_theme":    "Empowerment",
        "secondary_theme":  "Accomplishment",
        "tone_preferences": ["coaching", "progress-focused", "smart"],
        "hooks":            ["ai_personalized_practice", "ai_feedback", "fluency_progress", "skill_improvement"],
    },
    "Gamified Competitors": {
        "primary_theme":    "Accomplishment",
        "secondary_theme":  "Scarcity",
        "tone_preferences": ["energetic", "challenge-driven", "competitive"],
        "hooks":            ["streak_continuation", "points_milestone", "rank_jump", "time_bound_challenge"],
    },
    "Community Builders": {
        "primary_theme":    "Social Influence",
        "secondary_theme":  "Ownership",
        "tone_preferences": ["warm", "community-oriented", "encouraging"],
        "hooks":            ["peer_progress", "group_challenge", "community_streak", "shared_milestone"],
    },
    "Core Curriculum Users": {
        "primary_theme":    "Ownership",
        "secondary_theme":  "Accomplishment",
        "tone_preferences": ["clear", "structured", "encouraging"],
        "hooks":            ["your_learning_path", "next_step", "weekly_progress", "steady_milestone"],
    },
}

# ─────────────────────────────────────────────
# 4. GUARDRAILS — allowed themes per category
# ─────────────────────────────────────────────

RISK_PRIMARY_ALLOWED     = {"Loss Avoidance", "Empowerment", "Ownership"}
RISK_SECONDARY_ALLOWED   = {"Empowerment", "Ownership", "Accomplishment", "Loss Avoidance"}

RECOVERY_PRIMARY_ALLOWED = {
    "Dormant Users":      {"Empowerment", "Unpredictability", "Ownership"},
    "Slipping Subscribers": {"Empowerment", "Unpredictability", "Ownership"},
    "Lost Customers":     {"Epic Meaning", "Empowerment"},
}

GROWTH_PRIMARY_ALLOWED = {
    "High-Intent Trialists": {"Accomplishment", "Ownership", "Empowerment"},
    "AI-Powered Learners":   {"Empowerment", "Accomplishment"},
    "Gamified Competitors":  {"Accomplishment", "Scarcity", "Social Influence"},
    "Community Builders":    {"Social Influence", "Ownership", "Epic Meaning"},
    "Core Curriculum Users": {"Ownership", "Accomplishment", "Empowerment"},
}

# Themes never allowed as primary for entire recovery group
RECOVERY_BLOCKED_PRIMARY = {"Scarcity"}
# Lost Customers cannot use Loss Avoidance anywhere
LOST_CUSTOMERS_BLOCKED   = {"Loss Avoidance"}


# ─────────────────────────────────────────────────────────────
# 5. HELPER: infer segment category
# ─────────────────────────────────────────────────────────────

def get_segment_category(segment_id: str) -> str:
    """Return 'risk', 'recovery', or 'growth' for a given segment name."""
    if segment_id in RISK_SEGMENTS:
        return "risk"
    if segment_id in RECOVERY_SEGMENTS:
        return "recovery"
    if segment_id in GROWTH_SEGMENTS:
        return "growth"
    # Heuristic fallback for unknown segment names
    name_lower = segment_id.lower()
    if any(kw in name_lower for kw in ["risk", "trial"]):
        return "risk"
    if any(kw in name_lower for kw in ["dormant", "slipping", "lost", "inactive"]):
        return "recovery"
    return "growth"


# ─────────────────────────────────────────────────────────────
# 6. HELPER: generic fallback for unknown segments
# ─────────────────────────────────────────────────────────────

def generic_fallback(segment_id: str) -> dict:
    """
    Return a generic default mapping for segments not in DEFAULT_MAPPINGS.
    Infers category from segment name keywords.
    """
    cat = get_segment_category(segment_id)
    if cat == "risk":
        return {
            "primary_theme":    "Loss Avoidance",
            "secondary_theme":  "Empowerment",
            "tone_preferences": ["urgent-but-supportive", "reassuring"],
            "hooks":            ["protect_progress", "easy_restart", "your_learning_path", "next_step"],
        }
    elif cat == "recovery":
        return {
            "primary_theme":    "Empowerment",
            "secondary_theme":  "Ownership",
            "tone_preferences": ["supportive", "gentle", "encouraging"],
            "hooks":            ["easy_restart", "your_learning_path", "next_step", "continue_where_left"],
        }
    else:  # growth
        return {
            "primary_theme":    "Ownership",
            "secondary_theme":  "Accomplishment",
            "tone_preferences": ["motivating", "progress-focused", "encouraging"],
            "hooks":            ["your_learning_path", "next_step", "milestone_progress", "steady_milestone"],
        }


def get_default_mapping(segment_id: str) -> dict:
    """Return predefined default or generic fallback. Always returns a deep copy."""
    if segment_id in DEFAULT_MAPPINGS:
        return deepcopy(DEFAULT_MAPPINGS[segment_id])
    print(f"  [WARN] Segment '{segment_id}' not in predefined defaults → using generic fallback.")
    return generic_fallback(segment_id)


# ─────────────────────────────────────────────────────────────
# 7. FEATURE RESOLUTION
# ─────────────────────────────────────────────────────────────

CANDIDATE_FEATURES = [
    "activeness_score",
    "churn_risk_score",
    "engagement_score",
    # resolved below
]

FEATURE_ALIASES = {
    "ai":            ["ai_tutor_score", "ai_tutor_propensity"],
    "gamification":  ["gamification_score", "gamification_propensity"],
    "leaderboard":   ["leaderboard_score", "leaderboard_propensity"],
    "social":        ["social_score", "social_propensity"],
}


def resolve_feature_columns(df_columns: list) -> dict:
    """
    Resolve which actual column to use for each motivational feature.
    Returns a dict: {canonical_name: actual_column or None}.
    """
    resolved = {}
    for canonical, aliases in FEATURE_ALIASES.items():
        found = None
        for alias in aliases:
            if alias in df_columns:
                found = alias
                break
        resolved[canonical] = found
    return resolved


def get_clustering_features(df: pd.DataFrame) -> list:
    """
    Return list of valid numeric column names to use for K-means on this DataFrame slice.
    Applies fallback resolution, drops zero-variance and all-NaN columns.
    """
    resolved = resolve_feature_columns(df.columns.tolist())

    candidates = []
    # Base features
    for base_col in ["activeness_score", "churn_risk_score", "engagement_score"]:
        if base_col in df.columns:
            candidates.append(base_col)

    # Motivational features (resolved)
    for canonical, col in resolved.items():
        if col is not None and col in df.columns:
            candidates.append(col)

    # Keep only numeric columns
    numeric_candidates = [c for c in candidates if pd.api.types.is_numeric_dtype(df[c])]

    # Drop zero-variance features within this slice
    valid = []
    for col in numeric_candidates:
        series = df[col].dropna()
        if len(series) > 0 and series.std() > 1e-9:
            valid.append(col)

    return valid


# ─────────────────────────────────────────────────────────────
# 8. CLUSTER PROFILER
# ─────────────────────────────────────────────────────────────

def profile_cluster(cluster_df: pd.DataFrame, resolved_cols: dict,
                    cluster_id: int, segment_id: str, total_segment_size: int) -> dict:
    """
    Given a DataFrame slice for one cluster, compute all profiling metrics.
    resolved_cols: {canonical -> actual_col_or_None}
    """
    n = len(cluster_df)
    share = n / total_segment_size if total_segment_size > 0 else 0.0

    def safe_mean(col):
        if col and col in cluster_df.columns:
            vals = cluster_df[col].dropna()
            return float(vals.mean()) if len(vals) > 0 else 0.0
        return 0.0

    activeness_mean  = safe_mean("activeness_score")
    churn_mean       = safe_mean("churn_risk_score")
    engagement_mean  = safe_mean("engagement_score") if "engagement_score" in cluster_df.columns else None

    ai_mean           = safe_mean(resolved_cols.get("ai"))
    gamification_mean = safe_mean(resolved_cols.get("gamification"))
    leaderboard_mean  = safe_mean(resolved_cols.get("leaderboard"))
    social_mean       = safe_mean(resolved_cols.get("social"))

    competition_mean = max(gamification_mean, leaderboard_mean)

    # ── Energy score ──
    if engagement_mean is not None and "engagement_score" in cluster_df.columns:
        energy_score = 0.7 * activeness_mean + 0.3 * engagement_mean
    else:
        energy_score = activeness_mean

    if energy_score >= ENERGY_HIGH:
        energy_state = "high_energy"
    elif energy_score >= ENERGY_MED:
        energy_state = "medium_energy"
    else:
        energy_state = "low_energy"

    # ── Risk state ──
    if churn_mean >= CHURN_HIGH:
        risk_state = "high_risk"
    elif churn_mean >= CHURN_MED:
        risk_state = "medium_risk"
    else:
        risk_state = "low_risk"

    # ── Dominant motivation ──
    motivation_scores = {
        "ai":          ai_mean,
        "competition": competition_mean,
        "social":      social_mean,
    }
    sorted_mots = sorted(motivation_scores.items(), key=lambda x: x[1], reverse=True)
    top_name, top_score    = sorted_mots[0]
    _, second_score        = sorted_mots[1]
    motivation_spread      = top_score - second_score

    if top_score < DOM_TOP_MIN:
        dominant_motivation = "balanced"
    elif motivation_spread < DOM_SPREAD_MIN:
        dominant_motivation = "balanced"
    else:
        dominant_motivation = top_name

    # ── Competition subtype ──
    competition_subtype = None
    if dominant_motivation == "competition":
        if leaderboard_mean >= COMP_LEADER_THR and social_mean >= COMP_SOCIAL_THR:
            competition_subtype = "leaderboard_socialized_competition"
        else:
            competition_subtype = "solo_competition"

    return {
        "segment_id":          segment_id,
        "cluster_id":          cluster_id,
        "cluster_size":        n,
        "cluster_share":       round(share, 4),
        "activeness_mean":     round(activeness_mean, 4),
        "churn_mean":          round(churn_mean, 4),
        "engagement_mean":     round(engagement_mean, 4) if engagement_mean is not None else None,
        "ai_mean":             round(ai_mean, 4),
        "gamification_mean":   round(gamification_mean, 4),
        "leaderboard_mean":    round(leaderboard_mean, 4),
        "social_mean":         round(social_mean, 4),
        "competition_mean":    round(competition_mean, 4),
        "energy_score":        round(energy_score, 4),
        "energy_state":        energy_state,
        "risk_state":          risk_state,
        "dominant_motivation": dominant_motivation,
        "top_motivation_score":    round(top_score, 4),
        "second_motivation_score": round(second_score, 4),
        "motivation_spread":       round(motivation_spread, 4),
        "competition_subtype": competition_subtype,
    }


# ─────────────────────────────────────────────────────────────
# 9. RULEBOOK: cluster profile → themes / tones / hooks
# ─────────────────────────────────────────────────────────────

def apply_rulebook(profile: dict, segment_id: str, default_mapping: dict) -> dict:
    """
    Apply ordered rules to a cluster profile.
    Returns: {primary, secondary, tones, hooks, rule_id, rule_conf}
    """
    cat = get_segment_category(segment_id)

    risk_state         = profile["risk_state"]
    energy_state       = profile["energy_state"]
    dominant_motivation = profile["dominant_motivation"]
    leaderboard_mean   = profile["leaderboard_mean"]
    social_mean        = profile["social_mean"]
    top_motivation     = profile["top_motivation_score"]
    motivation_spread  = profile["motivation_spread"]

    # ── Rule 8: Lost Customers (always first for this segment) ──
    if segment_id == "Lost Customers":
        return {
            "primary_theme":  "Epic Meaning",
            "secondary_theme": "Empowerment",
            "tones": ["empathetic", "aspirational", "gentle"],
            "hooks": ["future_self", "real_life_confidence", "restart_small", "rebuild_confidence"],
            "rule_id":   8,
            "rule_conf": 1.0,
        }

    # ── Rule 1: Fragile high-risk pattern ──
    if risk_state == "high_risk" and energy_state in {"low_energy", "medium_energy"}:
        if segment_id == "At-Risk Trialists":
            primary, secondary = "Loss Avoidance", "Empowerment"
        elif segment_id == "Flight-Risk Subscribers":
            primary, secondary = "Loss Avoidance", "Ownership"
        else:
            primary, secondary = "Loss Avoidance", "Empowerment"
        return {
            "primary_theme":  primary,
            "secondary_theme": secondary,
            "tones": ["urgent-but-supportive", "reassuring"],
            "hooks": ["protect_streak", "protect_progress", "small_restart", "continue_plan"],
            "rule_id":   1,
            "rule_conf": 1.0,
        }

    # ── Rule 2: Low-energy recovery pattern (not high risk) ──
    if energy_state == "low_energy" and risk_state in {"low_risk", "medium_risk"}:
        primary   = "Empowerment"
        secondary = "Ownership"
        # Exception: Dormant Users with flat motivation
        if segment_id == "Dormant Users" and (top_motivation < DOM_TOP_MIN or motivation_spread < DOM_SPREAD_MIN):
            secondary = "Unpredictability"
        return {
            "primary_theme":  primary,
            "secondary_theme": secondary,
            "tones": ["supportive", "gentle", "low-pressure"],
            "hooks": ["easy_restart", "two_min_practice", "continue_where_left", "plan_waiting"],
            "rule_id":   2,
            "rule_conf": 0.8,
        }

    # ── Rule 3: AI mastery pattern ──
    if (dominant_motivation == "ai"
            and energy_state in {"medium_energy", "high_energy"}
            and risk_state != "high_risk"):
        return {
            "primary_theme":  "Empowerment",
            "secondary_theme": "Accomplishment",
            "tones": ["coaching", "progress-focused"],
            "hooks": ["ai_personalized_practice", "ai_feedback", "skill_improvement", "fluency_progress"],
            "rule_id":   3,
            "rule_conf": 1.0,
        }

    # ── Rule 4: Competitive high-energy pattern ──
    if (dominant_motivation == "competition"
            and energy_state == "high_energy"
            and risk_state in {"low_risk", "medium_risk"}):
        if leaderboard_mean >= COMP_LEADER_THR and social_mean >= COMP_SOCIAL_THR:
            secondary = "Social Influence"
        else:
            secondary = "Scarcity"
        return {
            "primary_theme":  "Accomplishment",
            "secondary_theme": secondary,
            "tones": ["energetic", "challenge-driven"],
            "hooks": ["streak_continuation", "points_milestone", "rank_jump", "time_bound_challenge"],
            "rule_id":   4,
            "rule_conf": 1.0,
        }

    # ── Rule 5: Competitive medium-energy pattern ──
    if (dominant_motivation == "competition"
            and energy_state == "medium_energy"
            and risk_state in {"low_risk", "medium_risk"}):
        # Recovery-like segments get Empowerment; others depend on social/leaderboard
        if cat == "recovery":
            secondary = "Empowerment"
        elif leaderboard_mean >= COMP_LEADER_THR and social_mean >= COMP_SOCIAL_THR:
            secondary = "Social Influence"
        else:
            secondary = "Scarcity"
        return {
            "primary_theme":  "Accomplishment",
            "secondary_theme": secondary,
            "tones": ["motivating", "encouraging", "challenge-lite"],
            "hooks": ["streak_recovery", "small_points_win", "next_rank_push", "daily_challenge"],
            "rule_id":   5,
            "rule_conf": 0.8,
        }

    # ── Rule 6: Social belonging pattern ──
    if dominant_motivation == "social" and risk_state in {"low_risk", "medium_risk"}:
        if segment_id == "Community Builders":
            secondary = "Ownership"
        elif energy_state == "high_energy":
            secondary = "Accomplishment"
        else:
            secondary = "Ownership"
        return {
            "primary_theme":  "Social Influence",
            "secondary_theme": secondary,
            "tones": ["warm", "community-oriented", "encouraging"],
            "hooks": ["peer_progress", "group_challenge", "shared_milestone", "community_streak"],
            "rule_id":   6,
            "rule_conf": 1.0,
        }

    # ── Rule 7: Balanced stable pattern ──
    if (dominant_motivation == "balanced"
            and energy_state in {"medium_energy", "high_energy"}
            and risk_state in {"low_risk", "medium_risk"}):
        return {
            "primary_theme":  "Ownership",
            "secondary_theme": "Accomplishment",
            "tones": ["clear", "structured", "encouraging"],
            "hooks": ["your_learning_path", "next_step", "weekly_progress", "steady_milestone"],
            "rule_id":   7,
            "rule_conf": 0.8,
        }

    # ── Rule 9: Fallback / uninterpretable ──
    if cat == "risk":
        primary = "Loss Avoidance"
        # subscriber risk segments → Ownership; trialist-ish → Empowerment
        secondary = "Ownership" if "Subscriber" in segment_id else "Empowerment"
    elif cat == "recovery":
        primary = "Empowerment"
        secondary = "Unpredictability" if segment_id == "Dormant Users" else "Ownership"
    else:
        primary   = default_mapping["primary_theme"]
        secondary = default_mapping["secondary_theme"]

    return {
        "primary_theme":  primary,
        "secondary_theme": secondary,
        "tones": default_mapping["tone_preferences"],
        "hooks": default_mapping["hooks"],
        "rule_id":   9,
        "rule_conf": 0.5,
    }


# ─────────────────────────────────────────────────────────────
# 10. CONFIDENCE SCORING (exact per spec)
# ─────────────────────────────────────────────────────────────

def compute_confidence(cluster_share: float, top_motivation_score: float,
                       motivation_spread: float, rule_conf: float) -> dict:
    """
    Compute size_conf, mot_conf, rule_conf → final confidence_score and bucket.
    """
    # A) Size confidence
    if cluster_share >= 0.30:
        size_conf = 1.0
    elif cluster_share >= 0.15:
        size_conf = 0.8
    elif cluster_share >= 0.10:
        size_conf = 0.6
    else:
        size_conf = 0.4

    # B) Motivation clarity confidence
    if top_motivation_score >= 0.55 and motivation_spread >= 0.10:
        mot_conf = 1.0
    elif top_motivation_score >= 0.50 and motivation_spread >= 0.07:
        mot_conf = 0.8
    elif top_motivation_score >= 0.45 and motivation_spread >= 0.05:
        mot_conf = 0.6
    else:
        mot_conf = 0.4

    # C) Final
    confidence_score = W_SIZE * size_conf + W_MOT * mot_conf + W_RULE * rule_conf

    if confidence_score >= CONF_HIGH_THR:
        bucket = "High"
    elif confidence_score >= CONF_MED_THR:
        bucket = "Medium"
    else:
        bucket = "Low"

    multiplier = CONF_MULT[bucket]

    return {
        "size_conf":        size_conf,
        "mot_conf":         mot_conf,
        "confidence_score": round(confidence_score, 4),
        "confidence_bucket": bucket,
        "confidence_multiplier": multiplier,
    }


# ─────────────────────────────────────────────────────────────
# 11. SAFETY OVERRIDES  (segment-level, applied last)
# ─────────────────────────────────────────────────────────────

def apply_safety_overrides(primary: str, secondary: str,
                            segment_id: str,
                            theme_scores: dict,
                            default_mapping: dict) -> tuple:
    """
    Apply all guardrail safety overrides.
    Returns (primary, secondary) after corrections.
    theme_scores: {theme -> total_score} from aggregation.
    """
    cat = get_segment_category(segment_id)

    def best_allowed(allowed_set, exclude=None):
        """Pick highest-scoring theme from allowed_set, excluding exclude."""
        candidates = {t: theme_scores.get(t, 0.0) for t in allowed_set
                      if t != exclude}
        if not candidates:
            return None
        return max(candidates, key=candidates.get)

    # ── Override 1: Risk segments ──
    if cat == "risk":
        if primary not in RISK_PRIMARY_ALLOWED:
            replacement = best_allowed(RISK_PRIMARY_ALLOWED, exclude=secondary)
            if replacement is None:
                replacement = default_mapping["primary_theme"]
            print(f"    [SAFETY] Risk override: '{primary}' → '{replacement}' (primary)")
            primary = replacement

    # ── Override 2: Lost Customers ──
    if segment_id == "Lost Customers":
        if primary != "Epic Meaning":
            print(f"    [SAFETY] Lost Customers override: forcing primary → 'Epic Meaning'")
            primary = "Epic Meaning"
        if secondary == "Loss Avoidance":
            print(f"    [SAFETY] Lost Customers override: secondary 'Loss Avoidance' → 'Empowerment'")
            secondary = "Empowerment"

    # ── Override 3: Dormant Users ──
    if segment_id == "Dormant Users":
        if primary == "Loss Avoidance":
            replacement = best_allowed({"Unpredictability", "Empowerment"}, exclude=secondary)
            if replacement is None:
                replacement = default_mapping["primary_theme"]
            print(f"    [SAFETY] Dormant override: 'Loss Avoidance' → '{replacement}' (primary)")
            primary = replacement

    # ── Override 4: Primary ≠ Secondary ──
    if primary == secondary:
        # Try next highest scored valid theme
        all_themes = sorted(theme_scores.items(), key=lambda x: x[1], reverse=True)
        new_secondary = None
        for theme, _ in all_themes:
            if theme != primary and theme in OCTALYSIS_THEMES:
                new_secondary = theme
                break
        if new_secondary is None:
            # Use default secondary if distinct
            ds = default_mapping["secondary_theme"]
            new_secondary = ds if ds != primary else list(OCTALYSIS_THEMES - {primary})[0]
        print(f"    [SAFETY] Primary==Secondary override: secondary → '{new_secondary}'")
        secondary = new_secondary

    return primary, secondary


# ─────────────────────────────────────────────────────────────
# 12. AGGREGATION LOGIC  (clusters → segment-level output)
# ─────────────────────────────────────────────────────────────

def aggregate_clusters(cluster_results: list, segment_id: str,
                       default_mapping: dict) -> dict:
    """
    Aggregate cluster-level theme/tone/hook outputs into a single segment row.

    cluster_results: list of dicts, each containing:
        profile, rulebook_output, confidence_info
    Returns: {primary_theme, secondary_theme, tone_preferences, hooks}
    """
    # ── Filter meaningful clusters ──
    meaningful = [
        cr for cr in cluster_results
        if (cr["profile"]["cluster_share"] >= CLUSTER_SHARE_MIN
            or cr["profile"]["cluster_size"] >= SMALL_CLUSTER_MIN)
    ]

    if not meaningful:
        print(f"    [WARN] No meaningful clusters for '{segment_id}'; using defaults.")
        return {
            "primary_theme":    default_mapping["primary_theme"],
            "secondary_theme":  default_mapping["secondary_theme"],
            "tone_preferences": default_mapping["tone_preferences"],
            "hooks":            default_mapping["hooks"],
        }

    # ─────────── Theme scoring ───────────
    theme_scores = {}

    for cr in meaningful:
        share  = cr["profile"]["cluster_share"]
        mult   = cr["confidence_info"]["confidence_multiplier"]
        weight = share * mult

        p_theme = cr["rulebook"]["primary_theme"]
        s_theme = cr["rulebook"]["secondary_theme"]

        theme_scores[p_theme] = theme_scores.get(p_theme, 0.0) + 2.0 * weight
        theme_scores[s_theme] = theme_scores.get(s_theme, 0.0) + 1.0 * weight

    # ─────────── Tone scoring ───────────
    tone_scores = {}

    for cr in meaningful:
        share  = cr["profile"]["cluster_share"]
        mult   = cr["confidence_info"]["confidence_multiplier"]
        weight = share * mult
        tones  = cr["rulebook"]["tones"]

        for i, tone in enumerate(tones):
            pts = 2.0 if i == 0 else 1.0
            tone_scores[tone] = tone_scores.get(tone, 0.0) + pts * weight

    # ─────────── Hook scoring ───────────
    hook_scores = {}

    for cr in meaningful:
        share  = cr["profile"]["cluster_share"]
        mult   = cr["confidence_info"]["confidence_multiplier"]
        weight = share * mult
        hooks  = cr["rulebook"]["hooks"]

        for i, hook in enumerate(hooks):
            if i < 2:
                pts = 2.0
            elif i < 4:
                pts = 1.0
            else:
                pts = 0.5
            hook_scores[hook] = hook_scores.get(hook, 0.0) + pts * weight

    # ─────────── Select primary & secondary themes ───────────
    sorted_themes = sorted(theme_scores.items(), key=lambda x: x[1], reverse=True)

    # Apply guardrails to select primary
    cat = get_segment_category(segment_id)

    def allowed_primaries():
        if cat == "risk":
            return RISK_PRIMARY_ALLOWED
        if cat == "recovery":
            return RECOVERY_PRIMARY_ALLOWED.get(segment_id,
                   {"Empowerment", "Ownership", "Unpredictability"}) - RECOVERY_BLOCKED_PRIMARY
        if cat == "growth":
            return GROWTH_PRIMARY_ALLOWED.get(segment_id, OCTALYSIS_THEMES)
        return OCTALYSIS_THEMES

    allowed_p = allowed_primaries()
    primary = None
    for theme, _ in sorted_themes:
        if theme in allowed_p:
            primary = theme
            break
    if primary is None:
        primary = default_mapping["primary_theme"]

    # Secondary: next highest, not same as primary
    secondary = None
    for theme, _ in sorted_themes:
        if theme != primary:
            secondary = theme
            break
    if secondary is None:
        # Fill from default
        ds = default_mapping["secondary_theme"]
        secondary = ds if ds != primary else list(OCTALYSIS_THEMES - {primary})[0]

    # ─────────── Apply safety overrides ───────────
    primary, secondary = apply_safety_overrides(
        primary, secondary, segment_id, theme_scores, default_mapping
    )

    # ─────────── Select top tones (top 3, min 2) ───────────
    sorted_tones = sorted(tone_scores.items(), key=lambda x: x[1], reverse=True)
    final_tones  = [t for t, _ in sorted_tones[:4]]

    # Tone safety filters
    BLOCKED_TONES_RISK     = {"aggressive", "shaming", "panic", "salesy"}
    BLOCKED_TONES_RECOVERY = {"aggressive", "shaming", "panic"}
    if cat == "risk" or cat == "recovery":
        final_tones = [t for t in final_tones if t.lower() not in
                       (BLOCKED_TONES_RISK if cat == "risk" else BLOCKED_TONES_RECOVERY)]
    if segment_id == "Lost Customers":
        AVOID_TONES = {"urgent", "urgent-but-supportive", "salesy"}
        final_tones = [t for t in final_tones if t.lower() not in AVOID_TONES]

    if len(final_tones) < 2:
        final_tones = list(default_mapping["tone_preferences"])[:3]

    final_tones = final_tones[:3]

    # ─────────── Select top hooks (top 4, min 3) ───────────
    sorted_hooks = sorted(hook_scores.items(), key=lambda x: x[1], reverse=True)
    final_hooks  = [h for h, _ in sorted_hooks[:5]]

    # Hook family balance: ≥2 primary-aligned, ≥1 secondary-aligned
    primary_hook_family   = set(THEME_HOOKS.get(primary, []))
    secondary_hook_family = set(THEME_HOOKS.get(secondary, []))

    primary_aligned   = [h for h in final_hooks if h in primary_hook_family]
    secondary_aligned = [h for h in final_hooks if h in secondary_hook_family]

    # Backfill primary hooks if needed
    if len(primary_aligned) < 2:
        for h in THEME_HOOKS.get(primary, []):
            if h not in final_hooks:
                final_hooks.append(h)
            if len([x for x in final_hooks if x in primary_hook_family]) >= 2:
                break

    # Backfill secondary hooks if needed
    if len(secondary_aligned) < 1:
        for h in THEME_HOOKS.get(secondary, []):
            if h not in final_hooks:
                final_hooks.append(h)
                break

    if len(final_hooks) < 3:
        for h in default_mapping["hooks"]:
            if h not in final_hooks:
                final_hooks.append(h)
            if len(final_hooks) >= 3:
                break

    final_hooks = final_hooks[:4]

    return {
        "primary_theme":    primary,
        "secondary_theme":  secondary,
        "tone_preferences": final_tones,
        "hooks":            final_hooks,
    }


# ─────────────────────────────────────────────────────────────
# 13. K-MEANS RUNNER
# ─────────────────────────────────────────────────────────────

def select_k(segment_size: int) -> list:
    """Return candidate K values for a given segment size."""
    if segment_size < KMEANS_MIN_SIZE:
        return []
    elif segment_size <= 80:
        return [2]
    elif segment_size <= 200:
        return [2, 3]
    else:
        return [3, 4]


def run_kmeans_for_segment(seg_df: pd.DataFrame, features: list,
                            segment_id: str) -> tuple:
    """
    Run K-means inside a segment and return (labels, chosen_k, silhouette, success).
    Returns (None, None, None, False) if any eligibility check fails.
    """
    n = len(seg_df)
    candidate_ks = select_k(n)

    if not candidate_ks:
        print(f"    [SKIP] Segment too small ({n} users) for K-means.")
        return None, None, None, False

    # Build the feature matrix
    sub = seg_df[features].copy()

    # Median imputation within segment (log it)
    for col in features:
        missing_count = sub[col].isna().sum()
        if missing_count > 0:
            median_val = sub[col].median()
            print(f"    [IMPUTE] Median imputing {missing_count} missing values in '{col}' "
                  f"(median={median_val:.4f})")
            sub[col] = sub[col].fillna(median_val)

    # Drop rows still NaN (shouldn't happen after median impute, but safety net)
    before = len(sub)
    sub = sub.dropna()
    after  = len(sub)
    if after < before:
        print(f"    [WARN] Dropped {before - after} rows still NaN after imputation.")

    if after < KMEANS_MIN_SIZE:
        print(f"    [SKIP] After NaN drop, only {after} rows remain — skipping K-means.")
        return None, None, None, False

    # Remove zero-variance features in this slice (again, after imputation)
    valid_features = [c for c in features if sub[c].std() > 1e-9]
    if len(valid_features) < 2:
        print(f"    [SKIP] <2 valid features after imputation — skipping K-means.")
        return None, None, None, False

    # Scale
    scaler = StandardScaler()
    X = scaler.fit_transform(sub[valid_features])

    best_k     = None
    best_score = -1.0
    best_labels = None

    for k in candidate_ks:
        if k >= after:
            continue
        try:
            km = KMeans(n_clusters=k, n_init=KMEANS_N_INIT, random_state=RANDOM_STATE)
            labels = km.fit_predict(X)

            # Check cluster sizes
            sizes = pd.Series(labels).value_counts().values
            min_cluster = int(sizes.min())
            max_share_ok = all(s / after >= 0.10 for s in sizes) if k > 1 else True

            if min_cluster < max(SMALL_CLUSTER_MIN, int(0.10 * after)) and k > 1:
                print(f"    [K={k}] Smallest cluster={min_cluster}; too small — penalizing.")

            # Silhouette
            if k > 1 and len(set(labels)) > 1:
                try:
                    sil = silhouette_score(X, labels)
                except Exception:
                    sil = -1.0
            else:
                sil = 0.0

            print(f"    [K={k}] cluster_sizes={sorted(sizes, reverse=True)}, silhouette={sil:.3f}")

            # Selection priority: silhouette >= 0.25, prefer simpler K
            # If score difference < 0.03, prefer smaller K
            if best_k is None:
                best_k, best_score, best_labels = k, sil, labels
            else:
                if sil - best_score >= 0.03:
                    best_k, best_score, best_labels = k, sil, labels
                elif abs(sil - best_score) < 0.03:
                    # Keep smaller K (already best_k <= k if iterating ascending)
                    pass  # no update → prefer smaller K

        except Exception as e:
            print(f"    [ERROR] K-means K={k} failed: {e}")
            continue

    if best_labels is None:
        print(f"    [SKIP] No valid K found.")
        return None, None, None, False

    # Map labels back to original seg_df index
    # sub may have fewer rows if some were dropped
    return best_labels, best_k, best_score, True, sub.index


def process_segment_with_kmeans(seg_df: pd.DataFrame, segment_id: str,
                                 default_mapping: dict) -> tuple:
    """
    Full Layer 2 pipeline for a single segment.
    Returns (final_output_dict, cluster_debug_rows, used_kmeans: bool).
    """
    n = len(seg_df)
    print(f"\n  ── Segment: '{segment_id}' | Size: {n} ──")

    # Resolve features
    resolved_cols = resolve_feature_columns(seg_df.columns.tolist())
    features = get_clustering_features(seg_df)

    if len(features) < 2 or n < KMEANS_MIN_SIZE:
        reason = "too few features" if len(features) < 2 else "too small"
        print(f"    [SKIP] {reason} → using default mapping.")
        return (
            {
                "primary_theme":    default_mapping["primary_theme"],
                "secondary_theme":  default_mapping["secondary_theme"],
                "tone_preferences": default_mapping["tone_preferences"],
                "hooks":            default_mapping["hooks"],
            },
            [],
            False,
        )

    print(f"    [FEATURES] Using: {features}")

    # Run K-means
    result = run_kmeans_for_segment(seg_df, features, segment_id)

    # Handle different return signatures
    if result[3] is False:
        print(f"    [FALLBACK] K-means failed → using default mapping.")
        return (
            {
                "primary_theme":    default_mapping["primary_theme"],
                "secondary_theme":  default_mapping["secondary_theme"],
                "tone_preferences": default_mapping["tone_preferences"],
                "hooks":            default_mapping["hooks"],
            },
            [],
            False,
        )

    labels, chosen_k, silhouette, _, valid_index = result
    print(f"    [CHOSEN K={chosen_k}] silhouette={silhouette:.3f}")

    # Align labels back to the valid sub-set rows
    # Create a series indexed like the original seg_df (fill unmatched with -1)
    label_series = pd.Series(-1, index=seg_df.index)
    for idx, lbl in zip(valid_index, labels):
        label_series[idx] = lbl

    cluster_results  = []
    cluster_debug_rows = []

    for cluster_id in sorted(set(labels)):
        cluster_idx = label_series[label_series == cluster_id].index
        cluster_df  = seg_df.loc[cluster_idx]

        # Profile
        profile = profile_cluster(
            cluster_df, resolved_cols, cluster_id, segment_id, n
        )

        # Rulebook
        rulebook = apply_rulebook(profile, segment_id, default_mapping)

        # Confidence
        conf_info = compute_confidence(
            cluster_share       = profile["cluster_share"],
            top_motivation_score = profile["top_motivation_score"],
            motivation_spread   = profile["motivation_spread"],
            rule_conf           = rulebook["rule_conf"],
        )

        print(
            f"      Cluster {cluster_id}: size={profile['cluster_size']}, "
            f"share={profile['cluster_share']:.2f}, "
            f"energy={profile['energy_state']}, risk={profile['risk_state']}, "
            f"mot={profile['dominant_motivation']}, "
            f"rule={rulebook['rule_id']}, conf={conf_info['confidence_bucket']}"
        )
        print(
            f"        → primary='{rulebook['primary_theme']}', "
            f"secondary='{rulebook['secondary_theme']}'"
        )

        cluster_results.append({
            "profile":       profile,
            "rulebook":      rulebook,
            "confidence_info": conf_info,
        })

        # Debug row
        debug_row = {**profile,
                     "primary_theme_cluster":   rulebook["primary_theme"],
                     "secondary_theme_cluster":  rulebook["secondary_theme"],
                     "tones_cluster":            " | ".join(rulebook["tones"]),
                     "hooks_cluster":            " | ".join(rulebook["hooks"]),
                     "rule_id":                  rulebook["rule_id"],
                     "rule_conf":                rulebook["rule_conf"],
                     **conf_info}
        cluster_debug_rows.append(debug_row)

    # Aggregate
    final_output = aggregate_clusters(cluster_results, segment_id, default_mapping)
    return final_output, cluster_debug_rows, True


# ─────────────────────────────────────────────────────────────
# 14. VALIDATORS
# ─────────────────────────────────────────────────────────────

def validate_output(df: pd.DataFrame) -> None:
    """
    Run all required validations on the final output DataFrame.
    Raises ValueError on hard failures.
    """
    errors   = []
    warnings = []

    # A) Theme validations
    for _, row in df.iterrows():
        seg = row["segment_id"]
        pt  = row["primary_theme"]
        st  = row["secondary_theme"]

        if pt not in OCTALYSIS_THEMES:
            errors.append(f"[{seg}] primary_theme '{pt}' not in Octalysis set.")
        if st not in OCTALYSIS_THEMES:
            errors.append(f"[{seg}] secondary_theme '{st}' not in Octalysis set.")
        if pt == st:
            errors.append(f"[{seg}] primary_theme == secondary_theme ('{pt}').")

    # B) Coverage validations
    if df["segment_id"].duplicated().any():
        errors.append("Duplicate segment_id rows found in output.")

    # C) Tone / hook validations
    for _, row in df.iterrows():
        seg   = row["segment_id"]
        tones = [t.strip() for t in row["tone_preferences"].split("|") if t.strip()]
        hooks = [h.strip() for h in row["hooks"].split("|") if h.strip()]
        if len(tones) < 2:
            errors.append(f"[{seg}] tone_preferences has <2 items: {tones}")
        if len(hooks) < 3:
            errors.append(f"[{seg}] hooks has <3 items: {hooks}")

    # D) Consistency validations
    if "Lost Customers" in df["segment_id"].values:
        lc_row = df[df["segment_id"] == "Lost Customers"].iloc[0]
        if lc_row["primary_theme"] != "Epic Meaning":
            warnings.append(f"Lost Customers primary_theme != 'Epic Meaning' "
                            f"(got '{lc_row['primary_theme']}').")

    for seg in RISK_SEGMENTS:
        if seg in df["segment_id"].values:
            row = df[df["segment_id"] == seg].iloc[0]
            if row["primary_theme"] not in RISK_PRIMARY_ALLOWED:
                warnings.append(f"[{seg}] primary_theme '{row['primary_theme']}' "
                                 f"not in risk-allowed set.")

    if warnings:
        print("\n[VALIDATION WARNINGS]")
        for w in warnings:
            print(f"  ⚠  {w}")

    if errors:
        print("\n[VALIDATION ERRORS]")
        for e in errors:
            print(f"  ✗  {e}")
        raise ValueError(f"Output validation failed with {len(errors)} error(s). See above.")

    print("\n All validations passed.")


# ─────────────────────────────────────────────────────────────
# 15. MAIN PIPELINE RUNNER
# ─────────────────────────────────────────────────────────────

def run_theme_engine(
    input_path:     str = "/content/iteration_0_before_learning/user_segments.csv",
    output_path:    str = "/content/iteration_0_before_learning/communication_themes.csv",
    debug_path:     str = "/content/theme_engine_debug_clusters.csv",
) -> pd.DataFrame:
    """
    End-to-end Theme Engine pipeline.
    Returns the final communication_themes DataFrame.
    """
    print("=" * 65)
    print("THEME ENGINE — START")
    print("=" * 65)

    # ── Load data ──
    print(f"\n[1] Loading data from: {input_path}")
    df = pd.read_csv(input_path)
    print(f"    Loaded {len(df)} rows, {len(df.columns)} columns.")
    print(f"    Columns: {df.columns.tolist()}")

    if "segment_id" not in df.columns:
        raise ValueError("Input data must contain a 'segment_id' column.")

    segments = df["segment_id"].dropna().unique().tolist()
    print(f"\n[2] Unique segments found ({len(segments)}): {segments}")

    final_rows     = []
    all_debug_rows = []

    print("\n[3] Processing segments...")

    for segment_id in segments:
        seg_df = df[df["segment_id"] == segment_id].copy().reset_index(drop=True)
        seg_n  = len(seg_df)

        # Get default mapping (Layer 1)
        default_mapping = get_default_mapping(segment_id)

        # Run Layer 2 (K-means) — may fall back to Layer 1 internally
        final_output, cluster_debug_rows, used_kmeans = process_segment_with_kmeans(
            seg_df, segment_id, default_mapping
        )

        # Format output columns
        tones_str = " | ".join(final_output["tone_preferences"])
        hooks_str = " | ".join(final_output["hooks"])

        print(f"\n  ✔ [{segment_id}] ({'K-means' if used_kmeans else 'defaults'})")
        print(f"      primary_theme:    {final_output['primary_theme']}")
        print(f"      secondary_theme:  {final_output['secondary_theme']}")
        print(f"      tone_preferences: {tones_str}")
        print(f"      hooks:            {hooks_str}")

        final_rows.append({
            "segment_id":       segment_id,
            "primary_theme":    final_output["primary_theme"],
            "secondary_theme":  final_output["secondary_theme"],
            "tone_preferences": tones_str,
            "hooks":            hooks_str,
        })

        # Attach segment metadata to debug rows
        for dr in cluster_debug_rows:
            dr["used_kmeans"] = used_kmeans
        all_debug_rows.extend(cluster_debug_rows)

    # ── Build output DataFrames ──
    output_df = pd.DataFrame(final_rows, columns=[
        "segment_id", "primary_theme", "secondary_theme", "tone_preferences", "hooks"
    ])

    debug_df = pd.DataFrame(all_debug_rows) if all_debug_rows else pd.DataFrame()

    # ── Validate ──
    print("\n[4] Running validations...")
    validate_output(output_df)

    # ── Save outputs ──
    print(f"\n[5] Saving outputs...")
    output_df.to_csv(output_path, index=False)
    print(f"    ✓ {output_path}")

    if not debug_df.empty:
        debug_df.to_csv(debug_path, index=False)
        print(f"    ✓ {debug_path}")
    else:
        print(f"    (No cluster debug rows — all segments used defaults.)")

    # ── Print summary table ──
    print("\n[6] FINAL THEME ASSIGNMENTS SUMMARY")
    print("=" * 65)
    print(output_df.to_string(index=False))
    print("=" * 65)
    print("\nTHEME ENGINE — COMPLETE ")

    return output_df


# ─────────────────────────────────────────────────────────────
# 16. ENTRY POINT
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ── Configure paths ──
    INPUT_PATH  = "/content/iteration_0_before_learning/user_segments.csv"
    OUTPUT_PATH = "/content/iteration_0_before_learning/communication_themes.csv"
    DEBUG_PATH  = "/content/theme_engine_debug_clusters.csv"

    result_df = run_theme_engine(
        input_path  = INPUT_PATH,
        output_path = OUTPUT_PATH,
        debug_path  = DEBUG_PATH,
    )

THEME ENGINE — START

[1] Loading data from: /content/iteration_0_before_learning/user_segments.csv
    Loaded 61 rows, 30 columns.
    Columns: ['user_id', 'lifecycle_stage', 'days_since_signup', 'age_band', 'region', 'sessions_last_7d', 'exercises_completed_7d', 'streak_current', 'coins_balance', 'feature_ai_tutor_used', 'feature_leaderboard_viewed', 'feature_progress_checked', 'preferred_hour', 'notif_open_rate_30d', 'motivation_score', 'gender', 'location', 'feature_leaderboard_used', 'feature_social_used', 'activeness_score', 'engagement_score', 'churn_risk_score', 'gamification_score', 'ai_tutor_score', 'leaderboard_score', 'social_score', 'user_profile', 'engagement_band', 'churn_band', 'segment_id']

[2] Unique segments found (10): ['Flight-Risk Subscribers', 'Lost Customers', 'High-Intent Trialists', 'Slipping Subscribers', 'Dormant Users', 'At-Risk Trialists', 'Community Builders', 'AI-Powered Learners', 'Gamified Competitors', 'Core Curriculum Users']

[3] Processing segment

In [ ]:
!pip install groq -q

import os, json, re, time, hashlib, warnings
import pandas as pd, numpy as np
warnings.filterwarnings('ignore')

from google.colab import userdata
from groq import Groq as GroqClient

# ── Load 3 keys ───────────────────────────────────────────────
# GROQ_KEYS = [
#     userdata.get("GROQ_API_KEY_1"),
#     userdata.get("GROQ_API_KEY_2"),
#     userdata.get("GROQ_API_KEY_3"),
# ]

GROQ_KEYS = ["gsk_placeholder","gsk_placeholder","gsk_placeholder"]

GROQ_MODEL   = "llama-3.3-70b-versatile"
RPM_SAFE     = 28          # use 28 of 30 allowed per key per minute

# ── Build one client per key ──────────────────────────────────
groq_clients = [GroqClient(api_key=k) for k in GROQ_KEYS]

# ── Per-key rate tracking ─────────────────────────────────────
_key_calls  = [0, 0, 0]                          # calls made this window
_key_resets = [time.time() + 60] * 3             # when each window resets
_key_ok     = [True, True, True]                 # which keys are working

# ── Connection tests ──────────────────────────────────────────
print("Testing 3 Groq keys...")
for i, client in enumerate(groq_clients):
    try:
        client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": "Reply: OK"}],
            max_tokens=5,
        )
        print(f"  ✓ Key {i+1} OK")
    except Exception as e:
        print(f"  ✗ Key {i+1} FAILED: {e}")
        _key_ok[i] = False

active = sum(_key_ok)
print(f"\n{active}/3 keys active — combined capacity: {active * RPM_SAFE} RPM")
print(f"150 templates at {active * RPM_SAFE} RPM = ~{150 // (active * RPM_SAFE) + 1} min estimated")





In [ ]:
THEMES_PATH = '/content/iteration_0_before_learning/communication_themes.csv'

REQUIRED_COLS = {'segment_id','primary_theme','secondary_theme',
                 'tone_preferences','hooks'}

VALID_THEMES = {
    'Epic Meaning','Accomplishment','Empowerment','Ownership',
    'Social Influence','Scarcity','Unpredictability','Loss Avoidance'
}

EXPECTED_SEGMENTS = {
    'AI-Powered Learners','Flight-Risk Subscribers','Gamified Competitors',
    'Lost Customers','At-Risk Trialists','Slipping Subscribers',
    'Dormant Users','High-Intent Trialists','Community Builders',
    'Core Curriculum Users'
}

themes_raw = pd.read_csv(THEMES_PATH)

# Structural assertions
missing_cols = REQUIRED_COLS - set(themes_raw.columns)
assert not missing_cols, f'Missing columns: {missing_cols}'
assert len(themes_raw) == 10, f'Expected 10 rows, got {len(themes_raw)}'
assert themes_raw['segment_id'].nunique() == 10, 'Duplicate segment_id found'
missing_segs = EXPECTED_SEGMENTS - set(themes_raw['segment_id'])
assert not missing_segs, f'Missing segments: {missing_segs}'

# Theme value assertions
for col in ['primary_theme','secondary_theme']:
    bad = set(themes_raw[col]) - VALID_THEMES
    assert not bad, f'Unknown theme in {col}: {bad}'
dup = themes_raw[themes_raw['primary_theme'] == themes_raw['secondary_theme']]
assert len(dup) == 0, f'primary==secondary for: {dup["segment_id"].tolist()}'

# Parse pipe-separated strings into lists
def parse_pipe(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [x.strip() for x in str(val).split('|') if x.strip()]

themes_raw['tones_list'] = themes_raw['tone_preferences'].apply(parse_pipe)
themes_raw['hooks_list'] = themes_raw['hooks'].apply(parse_pipe)

bad_tones = themes_raw[themes_raw['tones_list'].apply(len)==0]['segment_id'].tolist()
bad_hooks  = themes_raw[themes_raw['hooks_list'].apply(len)==0]['segment_id'].tolist()
assert not bad_tones, f'Segments with empty tones: {bad_tones}'
assert not bad_hooks,  f'Segments with empty hooks: {bad_hooks}'

themes_df = themes_raw.copy()
print('communication_themes.csv validated -- all 10 segments OK')
print(themes_df[['segment_id','primary_theme','secondary_theme']].to_string(index=False))


In [ ]:
# CELL 3 | Journey KB -- loaded from segment_goals.csv (Task 1 output)

SEGMENT_GOALS_PATH = '/content/iteration_0_before_learning/segment_goals.csv'

journey_df = pd.read_csv(SEGMENT_GOALS_PATH)

# Normalise column names defensively
journey_df.columns = journey_df.columns.str.strip().str.lower()

REQUIRED_JOURNEY_COLS = {
    'segment_id', 'lifecycle_stage', 'day_range',
    'primary_goal', 'sub_goals', 'daily_theme', 'notification_cta'
}
missing = REQUIRED_JOURNEY_COLS - set(journey_df.columns)
assert not missing, (
    f"segment_goals.csv is missing columns: {missing}\n"
    f"Re-run Task 1 with notification_cta in FINAL_COLS."
)

assert len(journey_df) == 30, f"Expected 30 journey rows, got {len(journey_df)}"
assert journey_df['notification_cta'].notna().all(), "Null notification_cta found"

# Every segment in themes_df must exist in journey_df
missing_in_journey = set(themes_df['segment_id']) - set(journey_df['segment_id'])
assert not missing_in_journey, (
    f"Segments in themes_df missing from segment_goals.csv: {missing_in_journey}"
)

print(f"Journey KB loaded from CSV -- {len(journey_df)} rows, "
      f"{journey_df['segment_id'].nunique()} segments")
print(journey_df.groupby('segment_id')['day_range'].count()
      .rename('phases').to_string())

In [ ]:
# CELL 4 | Merge themes x journey -> 30-row master table

MERGE_COLS = ['segment_id','day_range','primary_goal','sub_goals',
              'daily_theme','notification_cta','lifecycle_stage']

master_df = themes_df.merge(
    journey_df[MERGE_COLS],
    on='segment_id',
    how='inner'
)

# Hard assertions
assert len(master_df) == 30, f'Expected 30 rows, got {len(master_df)}'
for col in ['primary_theme','secondary_theme','tone_preferences','hooks',
            'notification_cta','day_range','lifecycle_stage','tones_list','hooks_list']:
    nulls = master_df[col].isna().sum()
    assert nulls == 0, f'Column {col} has {nulls} null(s) after merge'

print(f'master_df ready -- {len(master_df)} rows x {len(master_df.columns)} columns')
print(master_df[['segment_id','day_range','primary_theme','secondary_theme']].to_string(index=False))


In [ ]:
# CELL 5 | 5 Creative Slots with strict theme ownership
# Slots 1-3 -> PRIMARY theme  (from communication_themes.csv primary_theme)
# Slots 4-5 -> SECONDARY theme (from communication_themes.csv secondary_theme)

SLOT_CONFIGS = [
    {
        'slot': 1, 'theme_role': 'primary',
        'name': "Sia's Observation",
        'mechanism': (
            'Sia makes one specific personal observation about this user -- '
            'something that feels like she genuinely noticed a quiet pattern. '
            'NOT generic praise. Uses I naturally. '
            'Sounds like a perceptive friend who has been watching, not an algorithm.'
        ),
        'en_ex': 'You always come back. Even after missing 3 days. That is rarer than you think.',
        'hi_ex': 'Tum hamesha wapas aate ho yaar. Teen din miss ke baad bhi. Seriously, yeh common nahi hai.',
        'safe_for_all': True,
    },
    {
        'slot': 2, 'theme_role': 'primary',
        'name': 'The Honest Number',
        'mechanism': (
            'One hyper-specific odd number (47 not 50, 13 not 10, 23 not 25) '
            'that makes progress feel discovered not manufactured. '
            'Contrast it against what the average learner achieves. '
            'NEVER use round numbers. NEVER vague words like many or a lot.'
        ),
        'en_ex': '47 words this week. The average SpeakX learner manages 12.',
        'hi_ex': '47 words. Sirf is hafte mein. Average learner yaahan 12 tak pohunchta hai.',
        'safe_for_all': True,
    },
    {
        'slot': 3, 'theme_role': 'primary',
        'name': 'Soft Claim',
        'mechanism': (
            'The reward or next step is ALREADY theirs -- they just need to collect it. '
            'Frame it as belonging to them, not as something they must earn. '
            'Gentle time-awareness if relevant -- no panic, no hard sell. '
            'Like a friend reminding you that you left something valuable on the table.'
        ),
        'en_ex': '847 coins in your account right now. One session keeps them safe till next week.',
        'hi_ex': '847 coins tere account mein baithe hain. Ek session kar, safe ho jaayenge.',
        'safe_for_all': False,
        'blocked_for': [
            ('Dormant Users',  'D1-D5'),
            ('Dormant Users',  'D6-D14'),
            ('Lost Customers', 'D1-D7'),
            ('Lost Customers', 'D8-D15'),
        ],
        'fallback': {
            'name': 'The Memory',
            'mechanism': (
                'Remind the user of their own single best moment on SpeakX -- '
                'their peak streak, a specific word count, a day they felt proud. '
                'Pure positive recall. Zero pressure. Zero sell. No future ask.'
            ),
            'en_ex': 'You once built a 14-day streak. Nobody took that from you. It is still there.',
            'hi_ex': '14 din ka streak banaya tha tumne. Kisi ne nahi chhina. Abhi bhi wahin hai.',
        },
    },
    {
        'slot': 4, 'theme_role': 'secondary',
        'name': 'Half Story',
        'mechanism': (
            'Begin a sentence and cut it off with an em dash or ellipsis '
            'at the exact moment of maximum tension -- right before the payoff. '
            'The human brain cannot resist completing it. '
            'Cut when curiosity peaks: not too early (no context), not too late (gives answer). '
            'The user opens the app to finish the thought.'
        ),
        'en_ex': 'You were 3 exercises away from--',
        'hi_ex': 'Bas 3 aur exercises aur tum--',
        'safe_for_all': True,
    },
    {
        'slot': 5, 'theme_role': 'secondary',
        'name': 'Real India',
        'mechanism': (
            'Reference one specific real Indian life moment -- NOT vague professional success. '
            'Concrete: salary negotiation, office email in English, MNC interview, '
            'zoom call with foreign client, family function where relatives ask about job. '
            'Make them feel: Sia has been preparing me for exactly this moment.'
        ),
        'en_ex': 'Next salary negotiation? Sia has been building you for exactly that conversation.',
        'hi_ex': 'Agli salary negotiation mein akad ke baat karna hai? Yahi toh Sia seekha rahi hai.',
        'safe_for_all': False,
        'blocked_for': [
            ('Dormant Users',  'D1-D5'),
            ('Lost Customers', 'D1-D7'),
        ],
        'fallback': {
            'name': 'The Memory',
            'mechanism': (
                'Remind the user of their own single best moment on SpeakX -- '
                'their peak streak, a specific word count, a day they felt proud. '
                'Pure positive recall. Zero pressure. Zero sell. No future ask.'
            ),
            'en_ex': '500 words. You learned 500 words here. That is yours forever.',
            'hi_ex': '500 words seekhe the tumne yahan. Koi nahi le sakta yeh tum se.',
        },
    },
]

def resolve_slot(segment_id, day_range, slot_cfg):
    if not slot_cfg.get('safe_for_all', True):
        for (blocked_seg, blocked_day) in slot_cfg.get('blocked_for', []):
            if segment_id == blocked_seg and blocked_day in day_range:
                fb = slot_cfg['fallback'].copy()
                fb['slot']       = slot_cfg['slot']
                fb['theme_role'] = slot_cfg['theme_role']
                fb['swapped']    = True
                return fb
    out = slot_cfg.copy()
    out['swapped'] = False
    return out

print('5 creative slots defined')
print('  Slot 1: Sias Observation -> PRIMARY')
print('  Slot 2: The Honest Number -> PRIMARY')
print('  Slot 3: Soft Claim -> PRIMARY (swaps to Memory when blocked)')
print('  Slot 4: Half Story -> SECONDARY')
print('  Slot 5: Real India -> SECONDARY (swaps to Memory when blocked)')


In [ ]:
# CELL 6 | Hinglish voice bank + few-shot examples
# Hinglish is written INDEPENDENTLY from the same emotional core as English.
# It is NOT a translation. Own rhythm, humor, cultural anchors.

HINGLISH_VOICE = {
    'trial': {
        'register': 'Curious, encouraging new friend. Slightly excited. Ek baar try karo energy.',
        'phrases':  ['ek baar dekho', 'pehla step le lo', 'bas try karo na'],
        'avoid':    ['bahut accha', 'excellent', 'please aayein'],
    },
    'paid': {
        'register': 'Close friend who knows them. Casual, direct, slightly teasing. Yaar and chal na energy.',
        'phrases':  ['streak tod mat', 'rank dekha?', 'chal na'],
        'avoid':    ['kripya', 'dhanyawaad', 'aap bahut accha kar rahe hain'],
    },
    'inactive': {
        'register': 'Warm, no-pressure. Friend who checks in without expectations. Koi baat nahi energy.',
        'phrases':  ['koi baat nahi', 'jab mann kare', 'Sia wait kar rahi hai'],
        'avoid':    ['aapko chahiye', 'zaroor aana', 'mat bhoolna'],
    },
    'churned': {
        'register': 'Gentle nostalgia. Friend who misses you but will not chase. Yaad hai energy.',
        'phrases':  ['yaad hai jab', 'sab kuch abhi bhi wahin hai', 'kabhi aao toh'],
        'avoid':    ['wapas aao', 'please return', 'miss kar rahe hain'],
    },
}

INDIA_MOMENTS = {
    'trial':    ['pehle interview mein', 'boss ke saamne presentation', 'college mein group discussion'],
    'paid':     ['salary negotiation', 'office mein English email', 'zoom call pe foreign client'],
    'inactive': ['kitne time se plan kar rahe the', 'woh din yaad hai jab shuru kiya'],
    'churned':  ['jab pehli baar Sia se baat ki', 'woh 14-din wala streak', '500 words seekhe the'],
}

HINGLISH_FEW_SHOT = """
STUDY THESE BEFORE WRITING -- BAD (translated) vs GOOD (native Hinglish):

Core: You built something, do not lose it now
  EN  : Your 14-day streak is on the line. One session keeps it alive.
  BAD : Aapka 14 din ka streak khatre mein hai. Ek session usse bachayega.
  GOOD: 14 din ki mehnat hai yaar. Aaj ek lesson aur -- bas itna.

Core: Small action, real progress
  EN  : 2 minutes with Sia. That is literally all it takes today.
  BAD : Sia ke saath 2 minute. Aaj itna hi chahiye.
  GOOD: Bhai sirf 2 min de Sia ko. Chai bhi nahi poge itne mein.

Core: You are better than you realise
  EN  : 47 words in 6 days. You do not even notice you are getting good.
  BAD : 6 din mein 47 words. Aap realize nahi karte ki aap better ho rahe hain.
  GOOD: 47 words seekhe is hafte. Khud pata nahi chala na? Yahi toh progress hai.

Core: Come back without pressure
  EN  : No rush. Sia is here when you are ready.
  BAD : Koi jaldi nahi. Sia tab yahan hai jab aap ready hain.
  GOOD: Jab mann kare aa jaana. Sia kahin nahi gayi.

RULES:
1. Think in Hindi first. Add English words where they naturally fit.
2. Use aap not tum or tu. Tone respectful but friendly.
3. Indian sentence rhythm: verb at end, subject often dropped.
4. Real Indian life references: chai, ghar, office break, bus ride, dinner time — not “your journey”.
5. Humor from understatement, not exclamation marks.
6. NEVER write lines like “aap bahut accha kar rahe hain” — sounds like a school report.
7. NEVER force English words into Hindi grammar — that becomes translation, not natural Hinglish.
8. Sound like talking to a friend, not a teacher or motivational speaker.
9. Keep notifications short (8–14 words ideally).
10. Use everyday conversational phrasing Indians actually speak.
11. Avoid formal Hindi words like “kripya”, “kripya karke”, “anurodh hai”.
12. Prefer relatable micro-moments: chai break, lunch ke baad, TV start hone se pehle, raat ko phone scroll.
13. Avoid overly corporate words like “journey”, “productivity”, “milestone”.
14. Keep tone polite, light, and casual — respectful but never preachy.

"""


In [ ]:
# CELL 7 | Prompt builder
# Themes read ENTIRELY from master_df (which came from communication_themes.csv).
# No hardcoded theme values anywhere in this function.

BANNED_PHRASES = [
    'keep up the great work', 'do not give up', 'stay consistent',
    'practice makes perfect', 'learning journey', 'you got this',
    'never stop learning', 'keep going strong', 'believe in yourself',
    'great job', 'amazing work', 'stay motivated', 'every day counts',
    'you can do it', 'proud of you', 'small steps', 'fantastic',
]

def sanitize_daily_theme(text):
    text = re.sub(r'\[feature\]', 'their favourite feature', text)
    text = re.sub(r'\[topic\]',   'their chosen topic',      text)
    text = re.sub(r'\[user\]',    'a fellow learner',        text)
    text = re.sub(r'\[name\]',    'the user',                text)
    return text

def build_prompt(row, slot):
    lifecycle  = str(row.get('lifecycle_stage', 'paid'))
    hi_voice   = HINGLISH_VOICE.get(lifecycle, HINGLISH_VOICE['paid'])
    india_moms = INDIA_MOMENTS.get(lifecycle, INDIA_MOMENTS['paid'])

    # Theme this slot must express -- read from CSV columns, never hardcoded
    owned_theme = (row['primary_theme']   if slot['theme_role'] == 'primary'
                   else row['secondary_theme'])
    other_theme = (row['secondary_theme'] if slot['theme_role'] == 'primary'
                   else row['primary_theme'])

    # Tones and hooks -- parsed from communication_themes.csv
    tones_str = ', '.join(row.get('tones_list', []))
    hooks_str = ', '.join(row.get('hooks_list',  []))

    daily_clean = sanitize_daily_theme(str(row.get('daily_theme', '')))
    india_ref   = '; '.join(india_moms[:3])

    swap_note = (
        'NOTE: Gentle recall only -- NO selling, NO offers, NO social pressure.\n'
        if slot.get('swapped') else ''
    )

    return (
        f'You are a copywriter for SpeakX -- English learning app for Indian users aged 22-35.\n'
        f'Write notifications that feel like a close friend texted you, not an app.\n'
        f'{swap_note}\n'
        f'=== USER CONTEXT ===\n'
        f'Segment         : {row["segment_id"]}\n'
        f'Lifecycle       : {lifecycle}\n'
        f'Journey Phase   : {row["day_range"]}\n'
        f'Phase Goal      : {row["primary_goal"]}\n'
        f'Supporting Goal : {row["sub_goals"]}\n'
        f'Daily Theme     : {daily_clean}\n'
        f'CTA Direction   : {row["notification_cta"]}\n\n'
        f'=== COMMUNICATION THEMES (from Theme Engine output -- do NOT override) ===\n'
        f'Primary theme for this segment   : {row["primary_theme"]}\n'
        f'Secondary theme for this segment : {row["secondary_theme"]}\n'
        f'Tone preferences (use these)     : {tones_str}\n'
        f'Psychological hooks (use these)  : {hooks_str}\n\n'
        f'THIS SLOT expresses the {slot["theme_role"].upper()} theme: "{owned_theme}"\n'
        f'The theme "{other_theme}" is covered by other slots -- do NOT mix it here.\n\n'
        f'=== CREATIVE BRIEF ===\n'
        f'Slot Name   : {slot["name"]}\n'
        f'Mechanism   : {slot["mechanism"]}\n\n'
        f'Reference example (inspiration only -- do not copy):\n'
        f'  EN: {slot["en_ex"]}\n'
        f'  HI: {slot["hi_ex"]}\n\n'
        f'=== HINGLISH GUIDANCE ===\n'
        f'{HINGLISH_FEW_SHOT}\n'
        f'Voice register for {lifecycle}: {hi_voice["register"]}\n'
        f'Natural phrases: {chr(44).join(hi_voice["phrases"][:3])}\n'
        f'Avoid: {chr(44).join(hi_voice["avoid"][:2])}\n'
        f'Indian life moments for this lifecycle: {india_ref}\n\n'
        f'=== OUTPUT RULES ===\n'
        f'message_title_en : Max 7 words. Punchy. No generic opener. No forced exclamation.\n'
        f'message_body_en  : Max 20 words. Real human. Specific. Not corporate.\n'
        f'cta_text_en      : Max 4 words. Strong action verb first.\n'
        f'message_title_hi : WRITE FROM EMOTIONAL CORE IN HINGLISH -- not a translation.\n'
        f'message_body_hi  : WRITE FROM EMOTIONAL CORE IN HINGLISH -- not a translation.\n'
        f'cta_text_hi      : Max 4 words. Natural Hinglish action phrase.\n\n'
        f'BANNED (instant fail): {chr(44).join(BANNED_PHRASES[:8])}\n\n'
        f'Return ONLY valid JSON -- no markdown, no explanation, no extra keys:\n'
        f'{{\n'
        f'  "emotional_core": "one-line: what this message is really saying at its core",\n'
        f'  "message_title_en": "...",\n'
        f'  "message_body_en": "...",\n'
        f'  "cta_text_en": "...",\n'
        f'  "message_title_hi": "...",\n'
        f'  "message_body_hi": "...",\n'
        f'  "cta_text_hi": "..."\n'
        f'}}'
    )

print('Prompt builder ready -- themes read from master_df (communication_themes.csv source)')


In [ ]:
EXPECTED_KEYS = {
    'message_title_en', 'message_body_en', 'cta_text_en',
    'message_title_hi', 'message_body_hi', 'cta_text_hi',
}

SYSTEM_HEADER = (
    'You are a world-class behavioral copywriter specialising in Indian EdTech. '
    'Return ONLY valid JSON -- no markdown, no preamble, no extra text.\n\n'
)

_stats = {"key1": 0, "key2": 0, "key3": 0, "errors": 0}


def _refresh(i):
    """Reset call counter for key i if its 60s window has passed."""
    if time.time() >= _key_resets[i]:
        _key_calls[i]  = 0
        _key_resets[i] = time.time() + 60


def _get_best_key():
    """
    Return index of key with most remaining RPM budget.
    Returns -1 if ALL keys are at limit right now.
    """
    best_i   = -1
    best_rem = -1
    for i in range(3):
        if not _key_ok[i]:
            continue
        _refresh(i)
        remaining = RPM_SAFE - _key_calls[i]
        if remaining > best_rem:
            best_rem = remaining
            best_i   = i
    return best_i if best_rem > 0 else -1


def call_llm(prompt: str) -> str:
    """
    Picks the Groq key with most remaining budget.
    If ALL 3 keys are at their 28 RPM limit simultaneously,
    waits until the earliest reset and retries.
    """
    full_prompt = SYSTEM_HEADER + prompt

    for attempt in range(3):
        idx = _get_best_key()

        # All keys exhausted — wait for the earliest reset
        if idx == -1:
            earliest_reset = min(_key_resets[i] for i in range(3) if _key_ok[i])
            wait           = max(1, int(earliest_reset - time.time()) + 1)
            print(f"    [All 3 keys at RPM limit — waiting {wait}s for reset]")
            time.sleep(wait)
            for i in range(3):
                _refresh(i)
            idx = _get_best_key()
            if idx == -1:
                raise RuntimeError("All 3 Groq keys still exhausted after wait.")

        try:
            resp = groq_clients[idx].chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": full_prompt}],
                temperature=0.85,
                max_tokens=1024,
            )
            _key_calls[idx] += 1
            _stats[f"key{idx+1}"] += 1
            return resp.choices[0].message.content

        except Exception as e:
            err = str(e).lower()
            if '429' in err or 'rate' in err:
                print(f"    [Key {idx+1} hit 429 → marking exhausted, trying another]")
                _key_calls[idx] = RPM_SAFE    # treat as full for this window
            else:
                print(f"    [Key {idx+1} error: {e} → trying another key]")
                _key_ok[idx] = False          # mark key as broken
            _stats["errors"] += 1

    raise RuntimeError("call_llm failed after 3 internal attempts across all keys.")


def robust_parse(raw: str) -> dict:
    if not raw or not isinstance(raw, str):
        return {}
    try:
        clean = re.sub(r'```json|```', '', raw).strip()
        s = clean.find('{')
        e = clean.rfind('}') + 1
        if s == -1 or e == 0:
            return {}
        clean  = clean[s:e]
        clean  = re.sub(r',\s*([}\]])', r'\1', clean)
        parsed = json.loads(clean)
        if not EXPECTED_KEYS.intersection(parsed.keys()):
            for v in parsed.values():
                if isinstance(v, dict) and EXPECTED_KEYS.intersection(v.keys()):
                    return v
            return {}
        return parsed
    except Exception:
        return {}


def phash(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()[:8]


# Sanity test
try:
    _r = call_llm('Return JSON: {"message_title_en":"ok","message_body_en":"ok",'
                  '"cta_text_en":"go","message_title_hi":"ok","message_body_hi":"ok","cta_text_hi":"ja"}')
    _p = robust_parse(_r)
    print("LLM caller ready — 3-key rotation active.")
    print(f"  Model : {GROQ_MODEL}")
    print(f"  Capacity: {sum(_key_ok) * RPM_SAFE} RPM combined")
    if not _p:
        print(f"  WARNING: parse test empty — raw: {_r[:80]}")
except Exception as e:
    print(f"LLM caller test FAILED: {e}")


In [ ]:
# CELL 9 | Post-processing validators

BANNED_PHRASES_CHECK = [
    'keep up the great work', 'do not give up', 'stay consistent',
    'practice makes perfect', 'learning journey', 'you got this',
    'never stop learning', 'great job', 'amazing work', 'stay motivated',
    'you can do it', 'proud of you', 'fantastic', 'every day counts',
]

def enforce_limits(data):
    flags  = []
    limits = {
        'message_title_en': 7, 'message_title_hi': 7,
        'message_body_en': 20, 'message_body_hi': 20,
        'cta_text_en': 4,      'cta_text_hi': 4,
    }
    for col, wmax in limits.items():
        words = str(data.get(col, '')).split()
        if len(words) > wmax:
            suffix = '' if 'cta' in col else '...'
            data[col] = ' '.join(words[:wmax]) + suffix
            flags.append(f'TRUNCATED_{col}')
    return data, flags

def check_hinglish(data):
    flags = []
    for col in ['message_title_hi', 'message_body_hi', 'cta_text_hi']:
        text = str(data.get(col, ''))
        has_dev   = bool(re.search(r'[\u0900-\u097F]', text))
        has_latin = bool(re.search(r'[a-zA-Z]', text))
        if has_dev and not has_latin:
            flags.append(f'PURE_HINDI_{col}')
    return flags

def check_banned(data):
    flags = []
    haystack = (
        str(data.get('message_title_en', '')) + ' ' +
        str(data.get('message_body_en',  ''))
    ).lower()
    for phrase in BANNED_PHRASES_CHECK:
        if phrase in haystack:
            flags.append(f'BANNED:{phrase[:20]}')
    return flags

print('Post-processing validators ready')
print('  enforce_limits -- hard truncates, not just flags')
print('  check_hinglish -- catches pure Devanagari (likely translation)')
print('  check_banned   -- catches generic motivational copy')


In [ ]:
template_rows  = []
generation_log = []
tpl_counter    = 1
total          = len(master_df) * 5

print('=' * 65)
print('TEMPLATE GENERATOR -- START')
print(f'{total} templates ({len(master_df)} combos x 5 slots)')
print(f'Model    : {GROQ_MODEL}')
print(f'Keys     : {sum(_key_ok)}/3 active  |  {sum(_key_ok)*RPM_SAFE} RPM combined')
print('Slots 1-3 -> PRIMARY theme | Slots 4-5 -> SECONDARY theme')
print('=' * 65)

for _, row in master_df.iterrows():
    combo_label = f"{row['segment_id']} | {row['day_range']}"

    for slot_base in SLOT_CONFIGS:
        tpl_id   = f'TPL_{tpl_counter:04d}'
        slot     = resolve_slot(row['segment_id'], row['day_range'], slot_base)
        prompt   = build_prompt(row, slot)
        ph       = phash(prompt)
        data     = {}
        last_raw = ''

        for attempt in range(3):
            try:
                raw      = call_llm(prompt)
                last_raw = raw
                data     = robust_parse(raw)
                if data:
                    break
                wait = 2 + attempt * 2        # 2s → 4s → 6s
                print(f'  [{tpl_id}] Empty parse attempt {attempt+1}/3 -- retry in {wait}s')
                time.sleep(wait)
            except Exception as err:
                wait = 3 + attempt * 3        # 3s → 6s → 9s
                print(f'  [{tpl_id}] Error attempt {attempt+1}/3: {err} -- retry in {wait}s')
                time.sleep(wait)

        if not data:
            print(f'  PARSE FAILED [{tpl_id}] -- {combo_label} S{slot_base["slot"]}')
            if last_raw:
                print(f'    Raw: {last_raw[:120].replace(chr(10), " ")}')
            template_rows.append({
                'template_id':      tpl_id,
                'segment_id':       row['segment_id'],
                'lifecycle_stage':  row.get('lifecycle_stage', ''),
                'day_range':        row['day_range'],
                'slot_num':         slot_base['slot'],
                'slot_name':        slot['name'],
                'theme_role':       slot['theme_role'],
                'theme_expressed':  '',
                'slot_swapped':     slot.get('swapped', False),
                'primary_theme':    row['primary_theme'],
                'secondary_theme':  row['secondary_theme'],
                'tones_used':       row['tone_preferences'],
                'hooks_used':       row['hooks'],
                'emotional_core':   'GENERATION_ERROR',
                'message_title_en': 'GENERATION_ERROR',
                'message_body_en':  'Failed after 3 retries',
                'cta_text_en':      '',
                'message_title_hi': '',
                'message_body_hi':  '',
                'cta_text_hi':      '',
                'quality_flag':     'GENERATION_ERROR',
            })
            tpl_counter += 1
            time.sleep(1)
            continue

        # Post-processing
        data, trunc_flags = enforce_limits(data)
        hi_flags          = check_hinglish(data)
        ban_flags         = check_banned(data)
        all_flags         = trunc_flags + hi_flags + ban_flags
        qflag             = '|'.join(all_flags) if all_flags else 'OK'

        theme_expressed = (row['primary_theme'] if slot['theme_role'] == 'primary'
                           else row['secondary_theme'])

        template_rows.append({
            'template_id':      tpl_id,
            'segment_id':       row['segment_id'],
            'lifecycle_stage':  row.get('lifecycle_stage', ''),
            'day_range':        row['day_range'],
            'slot_num':         slot_base['slot'],
            'slot_name':        slot['name'],
            'theme_role':       slot['theme_role'],
            'theme_expressed':  theme_expressed,
            'slot_swapped':     slot.get('swapped', False),
            'primary_theme':    row['primary_theme'],
            'secondary_theme':  row['secondary_theme'],
            'tones_used':       row['tone_preferences'],
            'hooks_used':       row['hooks'],
            'emotional_core':   data.get('emotional_core',   ''),
            'message_title_en': data.get('message_title_en', ''),
            'message_body_en':  data.get('message_body_en',  ''),
            'cta_text_en':      data.get('cta_text_en',      ''),
            'message_title_hi': data.get('message_title_hi', ''),
            'message_body_hi':  data.get('message_body_hi',  ''),
            'cta_text_hi':      data.get('cta_text_hi',      ''),
            'quality_flag':     qflag,
        })

        generation_log.append({
            'template_id':     tpl_id,
            'segment_id':      row['segment_id'],
            'day_range':       row['day_range'],
            'slot_num':        slot_base['slot'],
            'slot_name':       slot['name'],
            'theme_role':      slot['theme_role'],
            'theme_expressed': theme_expressed,
            'swapped':         slot.get('swapped', False),
            'prompt_hash':     ph,
            'quality_flag':    qflag,
        })

        swap_tag = ' SWAPPED->Memory' if slot.get('swapped') else ''
        flag_tag = f' [{qflag}]'       if qflag != 'OK'       else ''
        print(f'  OK [{tpl_id}] {combo_label} '
              f'S{slot_base["slot"]}:{slot["name"]} '
              f'({slot["theme_role"]}/{theme_expressed}){swap_tag}{flag_tag}')

        tpl_counter += 1
        time.sleep(0.35)    # 0.35s = ~171 calls/min max, well within 84 RPM combined

# ── Summary ───────────────────────────────────────────────────
print()
print('=' * 65)
ok_c  = sum(1 for r in template_rows if r['quality_flag'] == 'OK')
err_c = sum(1 for r in template_rows if 'ERROR' in r['quality_flag'])
flg_c = len(template_rows) - ok_c - err_c
print(f'Generation complete -- {len(template_rows)} templates')
print(f'  OK          : {ok_c}')
print(f'  Flagged     : {flg_c}')
print(f'  Parse error : {err_c}')
print(f'\nGroq key usage:')
print(f'  Key 1 : {_stats["key1"]} calls')
print(f'  Key 2 : {_stats["key2"]} calls')
print(f'  Key 3 : {_stats["key3"]} calls')
print(f'  Errors: {_stats["errors"]} total')
print('=' * 65)

In [ ]:
# CELL 11 | Save outputs + validation report

FINAL_COLS = [
    'template_id','segment_id','lifecycle_stage','day_range',
    'slot_num','slot_name','theme_role','theme_expressed','slot_swapped',
    'primary_theme','secondary_theme','tones_used','hooks_used',
    'emotional_core',
    'message_title_en','message_body_en','cta_text_en',
    'message_title_hi','message_body_hi','cta_text_hi',
    'quality_flag',
]

templates_df = pd.DataFrame(template_rows)[FINAL_COLS]
log_df       = pd.DataFrame(generation_log)

templates_df.to_csv('/content/iteration_0_before_learning/message_templates.csv', index=False)
log_df.to_csv('/content/generation_log.csv',          index=False)

print(f'message_templates.csv saved -- {len(templates_df)} rows')
print(f'generation_log.csv saved    -- {len(log_df)} rows')

# Theme distribution check: should be 3 primary : 2 secondary per combo
print()
print('Theme role distribution (should be 3 primary : 2 secondary per combo):')
dist = (templates_df.groupby(['segment_id','theme_role'])
        .size().unstack(fill_value=0))
print(dist.to_string())

# Quality flag breakdown
print()
print('Quality flags breakdown:')
print(templates_df['quality_flag'].value_counts().head(20).to_string())

# Guard: check for empty EN titles
empty = templates_df[templates_df['message_title_en'].isin(['','GENERATION_ERROR'])]
if len(empty):
    print(f'\nWARNING: {len(empty)} template(s) with empty/error EN title')
    print(empty[['template_id','segment_id','day_range','slot_num']].to_string(index=False))
else:
    print('\nAll templates have non-empty EN titles')


In [33]:
"""
TIMING OPTIMIZER — EdTech Communication System (Task 2)
Input:  segmented DataFrame (from segmentation_engine)
Output: timing_recommendations.csv
"""

import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np


class TimingOptimizer:

    CONFIG = {
        "windows": {
            "early_morning":  (6,  9),
            "mid_morning":    (9,  12),
            "afternoon":      (12, 15),
            "late_afternoon": (15, 18),
            "evening":        (18, 21),
            "night":          (21, 24),
        },
        "default_window":                "early_morning",
        "ctr_weight":                    0.6,
        "engagement_weight":             0.4,
        "coverage_weight":               0.35,
        "consistency_weight":            0.30,
        "streak_weight":                 0.35,
        "streak_risk_session_threshold": 2,
        "habit_windows":                 ["early_morning", "evening", "night"],
        "top_n_windows":                 4,
        "required_cols": [
            "user_id", "segment_id",
            "preferred_hour", "notif_open_rate_30d", "activeness_score",
        ],
    }

    # ============================================================
    # HELPERS
    # ============================================================

    @classmethod
    def get_window(cls, hour):
        try:
            h = int(hour)
        except (ValueError, TypeError):
            return cls.CONFIG["default_window"]
        for name, (start, end) in cls.CONFIG["windows"].items():
            if start <= h < end:
                return name
        return cls.CONFIG["default_window"]

    @staticmethod
    def _safe_mean(series):
        vals = series.dropna()
        return float(vals.mean()) if len(vals) else 0.0

    @staticmethod
    def _safe_std(series):
        vals = series.dropna()
        return float(vals.std()) if len(vals) > 1 else 0.0

    @classmethod
    def assign_windows(cls, df):
        df = df.copy()
        missing = [c for c in cls.CONFIG["required_cols"] if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        df["preferred_window"] = df["preferred_hour"].apply(cls.get_window)
        return df

    # ============================================================
    # BEHAVIORAL SCORING
    # ============================================================

    @classmethod
    def score_windows(cls, seg_group):
        ctr_w = cls.CONFIG["ctr_weight"]
        eng_w = cls.CONFIG["engagement_weight"]
        cov_w = cls.CONFIG["coverage_weight"]
        con_w = cls.CONFIG["consistency_weight"]
        str_w = cls.CONFIG["streak_weight"]

        habit_windows = set(cls.CONFIG["habit_windows"])
        streak_thr    = cls.CONFIG["streak_risk_session_threshold"]

        has_streak   = "streak_current"   in seg_group.columns
        has_sessions = "sessions_last_7d" in seg_group.columns

        max_size = seg_group.groupby("preferred_window").size().max() or 1

        window_scores = {}

        for window, group in seg_group.groupby("preferred_window"):
            if group.empty:
                continue

            ctr        = cls._safe_mean(group["notif_open_rate_30d"])
            engagement = cls._safe_mean(group["activeness_score"])
            base_score = ctr_w * ctr + eng_w * engagement

            if base_score <= 0:
                continue

            coverage    = len(group) / max_size
            consistency = 1.0 / (1.0 + cls._safe_std(group["notif_open_rate_30d"]))

            if has_streak and has_sessions:
                at_risk    = group[
                    (group["streak_current"] > 0) &
                    (group["sessions_last_7d"] < streak_thr)
                ]
                risk_ratio = len(at_risk) / len(group)
            else:
                risk_ratio = 0.0

            streak_factor = risk_ratio if window in habit_windows else 0.5 * risk_ratio
            adjustment    = cov_w * coverage + con_w * consistency + str_w * streak_factor
            final_score   = base_score * (0.5 + adjustment)

            window_scores[window] = {
                "ctr":         round(ctr, 4),
                "engagement":  round(engagement, 4),
                "final_score": round(final_score, 4),
            }

        return window_scores

    # ============================================================
    # EXPORT
    # ============================================================

    @classmethod
    def export_csv(cls, df, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        records = []

        for segment_id, seg_group in df.groupby("segment_id"):
            window_scores = cls.score_windows(seg_group)

            if not window_scores:
                window_scores = {
                    cls.CONFIG["default_window"]: {
                        "ctr": None, "engagement": None, "final_score": 0.0
                    }
                }

            top_windows = sorted(
                window_scores,
                key=lambda w: window_scores[w]["final_score"],
                reverse=True
            )[:cls.CONFIG["top_n_windows"]]

            row = {"segment_id": segment_id}
            for i, window in enumerate(top_windows, 1):
                row[f"preferred_window_{i}"]    = window
                row[f"expected_ctr_{i}"]        = window_scores[window]["ctr"]
                row[f"expected_engagement_{i}"] = window_scores[window]["engagement"]

            records.append(row)

        result = pd.DataFrame(records)
        out_path = os.path.join(output_dir, "iteration_0_before_learning/timing_recommendations.csv")
        result.to_csv(out_path, index=False)
        print(f"  ✓ Saved: {out_path}")
        return result

    # ============================================================
    # ENTRY POINT
    # ============================================================

    @classmethod
    def run(cls, df_or_path, output_dir="."):
        """
        Usage:
            timing_df = TimingOptimizer.run(df, "iteration_0_before_learning")
        """
        print("=" * 65)
        print("TIMING OPTIMIZER — START")
        print("=" * 65)

        if isinstance(df_or_path, str):
            print(f"\n[1] Loading from: {df_or_path}")
            df = pd.read_csv(df_or_path)
        else:
            print(f"\n[1] Received DataFrame: {len(df_or_path)} rows.")
            df = df_or_path.copy()

        print(f"\n[2] Assigning time windows...")
        df = cls.assign_windows(df)

        print(f"\n[3] Scoring windows per segment...")
        result = cls.export_csv(df, output_dir)

        print(f"\n[4] Results:")
        print(result.to_string(index=False))

        print("\nTIMING OPTIMIZER — COMPLETE")
        return result

In [34]:
TimingOptimizer.run("../content/iteration_0_before_learning/user_segments.csv")

TIMING OPTIMIZER — START

[1] Loading from: ../content/iteration_0_before_learning/user_segments.csv

[2] Assigning time windows...

[3] Scoring windows per segment...
  ✓ Saved: ./iteration_0_before_learning/timing_recommendations.csv

[4] Results:
             segment_id preferred_window_1  expected_ctr_1  expected_engagement_1 preferred_window_2  expected_ctr_2  expected_engagement_2 preferred_window_3  expected_ctr_3  expected_engagement_3 preferred_window_4  expected_ctr_4  expected_engagement_4
    AI-Powered Learners      early_morning          0.3990                 0.5698            evening          0.2640                 0.7613                NaN             NaN                    NaN                NaN             NaN                    NaN
      At-Risk Trialists          afternoon          0.3800                 0.3184        mid_morning          0.3230                 0.2024            evening          0.2453                 0.2247                NaN             NaN      

,segment_id,preferred_window_1,expected_ctr_1,expected_engagement_1,preferred_window_2,expected_ctr_2,expected_engagement_2,preferred_window_3,expected_ctr_3,expected_engagement_3,preferred_window_4,expected_ctr_4,expected_engagement_4
0,AI-Powered Learners,early_morning,0.3990,0.5698,evening,0.2640,0.7613,NaN,NaN,NaN,NaN,NaN,NaN
1,At-Risk Trialists,afternoon,0.3800,0.3184,mid_morning,0.3230,0.2024,evening,0.2453,0.2247,NaN,NaN,NaN
2,Community Builders,night,0.5700,0.6658,early_morning,0.3900,0.5528,NaN,NaN,NaN,NaN,NaN,NaN
3,Core Curriculum Users,late_afternoon,0.5210,0.6726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dormant Users,early_morning,0.0563,0.0000,evening,0.0690,0.0000,afternoon,0.0330,0.0100,night,0.0355,0.0000
5,Flight-Risk Subscribers,evening,0.3808,0.4192,afternoon,0.3320,0.6042,night,0.3800,0.4999,early_morning,0.3575,0.4754
6,Gamified Competitors,evening,0.3760,0.8301,night,0.4250,0.5448,NaN,NaN,NaN,NaN,NaN,NaN
7,High-Intent Trialists,evening,0.3563,0.3191,night,0.4810,0.3551,mid_morning,0.3945,0.3189,early_morning,0.3483,0.2408
8,Lost Customers,mid_morning,0.1335,0.0000,night,0.0717,0.0256,evening,0.0790,0.0083,early_morning,0.0350,0.0367
9,Slipping Subscribers,evening,0.5172,0.4758,afternoon,0.5370,0.4654,late_afternoon,0.4830,0.4529,NaN,NaN,NaN


In [35]:
import os, math, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np


class FrequencyOptimizer:

    CONFIG = {
        "bands": [
            (0.7, 8, "high_active"),
            (0.4, 5, "moderate_active"),
            (0.0, 3, "low_active"),
        ],
        "main_fraction":       0.5,
        "guardrail_uninstall": 0.02,
        "guardrail_reduce":    2,
        "freq_floor":          1,
        "explore_threshold":   5,
        "ctr_weight":          0.5,
        "eng_weight":          0.5,
        "hour_map": [
            (range(6,  9),  "early_morning"),
            (range(9,  12), "mid_morning"),
            (range(12, 15), "afternoon"),
            (range(15, 18), "late_afternoon"),
            (range(18, 21), "evening"),
            (range(21, 24), "night"),
            (range(0,  6),  "early_morning"),
        ],
    }

    # ============================================================
    # HELPERS
    # ============================================================

    @classmethod
    def hour_to_window(cls, hour):
        try:
            h = int(hour)
            for r, w in cls.CONFIG["hour_map"]:
                if h in r:
                    return w
        except (TypeError, ValueError):
            pass
        return "early_morning"

    @classmethod
    def assign_frequency(cls, activeness):
        score = float(activeness or 0.0)
        for threshold, freq, band in cls.CONFIG["bands"]:
            if score >= threshold:
                return band, freq
        return "low_active", 3

    @classmethod
    def load_timing_map(cls, timing_path):
        """
        Returns {segment_id -> [{window, expected_ctr, expected_engagement}, ...]}
        Reads up to 4 preferred windows per segment.
        """
        df = pd.read_csv(timing_path)
        timing = {}
        for _, row in df.iterrows():
            seg = str(row["segment_id"])
            windows = []
            for i in range(1, 5):
                w_col   = f"preferred_window_{i}"
                ctr_col = f"expected_ctr_{i}"
                eng_col = f"expected_engagement_{i}"
                if w_col not in row or pd.isna(row.get(w_col)):
                    continue
                windows.append({
                    "window":     str(row[w_col]),
                    "ctr":        float(row.get(ctr_col, 0.0) or 0.0),
                    "engagement": float(row.get(eng_col, 0.0) or 0.0),
                })
            timing[seg] = windows
        return timing

    @classmethod
    def load_uninstall_rates(cls, experiment_path):
        """Returns {segment_id -> avg_uninstall_rate}"""
        if not experiment_path or not os.path.exists(experiment_path):
            return {}
        try:
            exp = pd.read_csv(experiment_path)
            seg_col = next((c for c in ["segment_id", "segment"] if c in exp.columns), None)
            uns_col = next((c for c in ["uninstall_rate", "uninstall"] if c in exp.columns), None)
            if not seg_col or not uns_col:
                return {}
            return exp.groupby(seg_col)[uns_col].mean().to_dict()
        except Exception:
            return {}

    # ============================================================
    # DISTRIBUTION HELPERS
    # ============================================================

    @classmethod
    def distribute_equal(cls, total, main_window, seg_windows):
        """
        iter0: equal split among timing windows.
        main_window → ceil(total * 0.5)
        others      → remaining split evenly (by rank order)
        """
        dist = {}
        main_count = min(math.ceil(total * cls.CONFIG["main_fraction"]), total)
        dist[main_window] = main_count
        remaining = total - main_count
        if remaining <= 0:
            return dist

        others = [w["window"] for w in seg_windows if w["window"] != main_window][:remaining]
        if not others:
            dist[main_window] = total
            return dist

        base, extra = divmod(remaining, len(others))
        for i, w in enumerate(others):
            dist[w] = base + (1 if i < extra else 0)
        return dist

    @classmethod
    def distribute_weighted(cls, total, main_window, seg_windows, explore):
        """
        iter1: CTR+engagement weighted distribution.
        main_window → ceil(total * 0.5)  [always fixed]
        others      → weighted by combined score from real experiment data

        Undiscovered windows (ctr=0, engagement=0):
            → get 1 notif if total > explore_threshold AND budget allows
        """
        dist = {}
        main_count = min(math.ceil(total * cls.CONFIG["main_fraction"]), total)
        dist[main_window] = main_count
        remaining = total - main_count
        if remaining <= 0:
            return dist

        others = [w for w in seg_windows if w["window"] != main_window]
        if not others:
            dist[main_window] = total
            return dist

        discovered   = [w for w in others if w["ctr"] > 0 or w["engagement"] > 0]
        undiscovered = [w for w in others if w["ctr"] == 0 and w["engagement"] == 0]

        explore_count = 0
        if explore and undiscovered:
            explore_count = min(len(undiscovered), remaining)
            for w in undiscovered[:explore_count]:
                dist[w["window"]] = 1
            remaining -= explore_count

        if remaining <= 0:
            return dist

        if discovered:
            scores = np.array([
                cls.CONFIG["ctr_weight"] * w["ctr"] + cls.CONFIG["eng_weight"] * w["engagement"]
                for w in discovered
            ])
            total_score = scores.sum()

            if total_score > 0:
                raw_alloc = (scores / total_score) * remaining
                alloc     = np.floor(raw_alloc).astype(int)
                leftover  = remaining - alloc.sum()
                fractions = raw_alloc - alloc
                top_idx   = np.argsort(-fractions)[:leftover]
                alloc[top_idx] += 1
            else:
                base, extra = divmod(remaining, len(discovered))
                alloc = np.array([base + (1 if i < extra else 0) for i in range(len(discovered))])

            for i, w in enumerate(discovered):
                dist[w["window"]] = int(alloc[i])
        else:
            dist[main_window] = dist.get(main_window, 0) + remaining

        return dist

    @classmethod
    def build_output_row(cls, base, dist, guardrail_applied):
        """
        Convert dist dict → 4 window_N / window_N_count column pairs.
        Sorted by count descending (main window always first).
        """
        row = {**base, "guardrail_applied": guardrail_applied}

        sorted_windows = sorted(dist.items(), key=lambda x: x[1], reverse=True)
        while len(sorted_windows) < 4:
            sorted_windows.append(("", 0))

        for i, (window, count) in enumerate(sorted_windows[:4], start=1):
            row[f"window_{i}"]       = window
            row[f"window_{i}_count"] = count

        return row

    # ============================================================
    # ENTRY POINTS
    # ============================================================

    @classmethod
    def run(cls, df_or_path, output_dir="./iteration_0_before_learning", timing_path=None):
        """
        iter0 initialization — equal distribution, no guardrail.

        Usage:
            freq_df = FrequencyOptimizer.run(
                df,
                output_dir  = "iteration_0_before_learning",
                timing_path = "iteration_0_before_learning/timing_recommendations.csv",
            )
        """
        df = pd.read_csv(df_or_path) if isinstance(df_or_path, str) else df_or_path.copy()

        if timing_path is None:
            timing_path = os.path.join(output_dir, "timing_recommendations.csv")
        timing_map = cls.load_timing_map(timing_path)

        records = []
        for _, row in df.iterrows():
            seg         = str(row["segment_id"])
            activeness  = float(row.get("activeness_score") or 0.0)
            band, total = cls.assign_frequency(activeness)
            main_window = cls.hour_to_window(row.get("preferred_hour", 7))
            seg_windows = timing_map.get(seg, [])
            dist        = cls.distribute_equal(total, main_window, seg_windows)

            base = {
                "user_id":          str(row["user_id"]),
                "segment_id":       seg,
                "lifecycle_stage":  str(row.get("lifecycle_stage", "paid")),
                "activeness_score": round(activeness, 4),
                "band":             band,
                "main_window":      main_window,
                "total_notifs":     total,
            }
            records.append(cls.build_output_row(base, dist, guardrail_applied=False))

        out_df = pd.DataFrame(records)
        os.makedirs(output_dir, exist_ok=True)
        out_df.to_csv(os.path.join(output_dir, "user_notification_schedule_iter0.csv"), index=False)
        print(f"[FrequencyOptimizer] {len(out_df)} users → {output_dir}/user_notification_schedule_iter0.csv")
        return out_df

    @classmethod
    def update(cls, df_or_path, output_dir="./output", timing_path=None, experiment_path=None):
        """
        iter1 update — guardrail + CTR/engagement weighted redistribution.

        Usage:
            freq_df = FrequencyOptimizer.update(
                df,
                output_dir      = "iteration_1_after_learning",
                timing_path     = "iteration_1_after_learning/timing_recommendations.csv",
                experiment_path = "experiment_results.csv",
            )
        """
        df = pd.read_csv(df_or_path) if isinstance(df_or_path, str) else df_or_path.copy()

        if timing_path is None:
            timing_path = os.path.join(output_dir, "timing_recommendations.csv")
        timing_map      = cls.load_timing_map(timing_path)
        uninstall_rates = cls.load_uninstall_rates(experiment_path)

        records = []
        for _, row in df.iterrows():
            seg         = str(row["segment_id"])
            activeness  = float(row.get("activeness_score") or 0.0)
            band, total = cls.assign_frequency(activeness)

            guardrail = False
            if uninstall_rates.get(seg, 0.0) > cls.CONFIG["guardrail_uninstall"]:
                total     = max(cls.CONFIG["freq_floor"], total - cls.CONFIG["guardrail_reduce"])
                guardrail = True

            main_window = cls.hour_to_window(row.get("preferred_hour", 7))
            seg_windows = timing_map.get(seg, [])
            explore     = total > cls.CONFIG["explore_threshold"]
            dist        = cls.distribute_weighted(total, main_window, seg_windows, explore)

            base = {
                "user_id":          str(row["user_id"]),
                "segment_id":       seg,
                "lifecycle_stage":  str(row.get("lifecycle_stage", "paid")),
                "activeness_score": round(activeness, 4),
                "band":             band,
                "main_window":      main_window,
                "total_notifs":     total,
            }
            records.append(cls.build_output_row(base, dist, guardrail_applied=guardrail))

        out_df = pd.DataFrame(records)
        os.makedirs(output_dir, exist_ok=True)
        out_df.to_csv(os.path.join(output_dir, "user_notification_schedule.csv"), index=False)
        print(f"[FrequencyOptimizer.update] {len(out_df)} users → {output_dir}/user_notification_schedule.csv")
        return out_df

In [36]:
FrequencyOptimizer.run(
    "../content/iteration_0_before_learning/user_segments.csv",
    "/content/iteration_0_before_learning", # Corrected output_dir
    "../content/iteration_0_before_learning/timing_recommendations.csv"
)

[FrequencyOptimizer] 61 users → /content/iteration_0_before_learning/user_notification_schedule_iter0.csv


,user_id,segment_id,lifecycle_stage,activeness_score,band,main_window,total_notifs,guardrail_applied,window_1,window_1_count,window_2,window_2_count,window_3,window_3_count,window_4,window_4_count
0,US_1,Flight-Risk Subscribers,paid,0.5796,moderate_active,early_morning,5,False,early_morning,3,evening,1,afternoon,1,,0
1,US_2,Lost Customers,churned,0.0000,low_active,evening,3,False,evening,2,mid_morning,1,,0,,0
2,US_3,High-Intent Trialists,trial,0.3784,low_active,mid_morning,3,False,mid_morning,2,evening,1,,0,,0
3,US_4,Slipping Subscribers,paid,0.4529,moderate_active,late_afternoon,5,False,late_afternoon,3,evening,1,afternoon,1,,0
4,US_5,Lost Customers,churned,0.0000,low_active,mid_morning,3,False,mid_morning,2,night,1,,0,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,US_57,Dormant Users,inactive,0.0000,low_active,night,3,False,night,2,early_morning,1,,0,,0
57,US_58,Dormant Users,inactive,0.0000,low_active,early_morning,3,False,early_morning,2,evening,1,,0,,0
58,US_59,Gamified Competitors,paid,0.5401,moderate_active,night,5,False,night,3,evening,2,,0,,0
59,US_60,Flight-Risk Subscribers,paid,0.4999,moderate_active,night,5,False,night,3,evening,1,afternoon,1,,0


# **Task 3**

In [41]:
dfff=pd.read_csv("/content/iteration_0_before_learning/message_templates.csv")

In [42]:
dfff

,template_id,segment_id,slot_num,performance_label,best_window,best_ctr,best_engagement,posterior_mean,suppressed
0,TPL_0001,AI-Powered Learners,1,GOOD,late_afternoon,0.6484,0.8644,0.5833,False
1,TPL_0002,AI-Powered Learners,2,GOOD,late_afternoon,0.5604,0.7647,0.5833,False
2,TPL_0003,AI-Powered Learners,3,GOOD,late_afternoon,0.4945,0.6000,0.5833,False
3,TPL_0004,AI-Powered Learners,4,GOOD,late_afternoon,0.4396,0.5750,0.5833,False
4,TPL_0005,AI-Powered Learners,5,GOOD,late_afternoon,0.3077,0.4643,0.5833,False
5,TPL_0006,At-Risk Trialists,1,GOOD,afternoon,0.4891,0.4179,0.5278,False
6,TPL_0007,At-Risk Trialists,2,NEUTRAL,afternoon,0.4307,0.3559,0.5000,False
7,TPL_0008,At-Risk Trialists,3,NEUTRAL,afternoon,0.3577,0.3061,0.5000,False
8,TPL_0009,At-Risk Trialists,4,NEUTRAL,afternoon,0.2628,0.2500,0.5000,False
9,TPL_0010,At-Risk Trialists,5,NEUTRAL,afternoon,0.2263,0.1935,0.4722,False


## -- Schedule Generator --

In [44]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

USERS_PATH     = "/content/iteration_0_before_learning/user_segments.csv"
SCHEDULE_PATH  = "/content/iteration_0_before_learning/user_notification_schedule_iter0.csv"
TEMPLATES_PATH = "/content/iteration_0_before_learning/message_templates.csv"
TIMING_PATH    = "/content/iteration_0_before_learning/timing_recommendations.csv"
OUTPUT_DIR     = "/content/iteration_0_before_learning"

MAX_SLOTS = 9
LIFECYCLE_CHANNEL = {
    "trial": "push", "paid": "push",
    "inactive": "push", "churned": "email",
}
ALL_WINDOWS = [
    "early_morning", "mid_morning", "afternoon",
    "late_afternoon", "evening", "night",
]


def load_data(users_path, schedule_path, templates_path, timing_path):
    print("[1] Loading input files...")
    users_df = pd.read_csv(users_path)
    users_df.columns = users_df.columns.str.strip().str.lower()

    sched_df = pd.read_csv(schedule_path)
    sched_df.columns = sched_df.columns.str.strip().str.lower()
    want = ["user_id", "total_notifs",
            "window_1", "window_1_count", "window_2", "window_2_count",
            "window_3", "window_3_count", "window_4", "window_4_count"]
    sched_cols = [c for c in want if c in sched_df.columns]
    users_df = users_df.merge(sched_df[sched_cols], on="user_id", how="left")
    users_df["total_notifs"] = users_df["total_notifs"].fillna(3).astype(int)

    tmpls_df = pd.read_csv(templates_path)
    tmpls_df.columns = tmpls_df.columns.str.strip().str.lower()
    before = len(tmpls_df)
    if "quality_flag" in tmpls_df.columns:
        tmpls_df = tmpls_df[
            ~tmpls_df["quality_flag"].str.upper().str.contains("ERROR", na=False)
        ].reset_index(drop=True)
    after = len(tmpls_df)
    if before != after:
        print(f"    Templates: {before} → {after} (removed {before-after} errors)")
    else:
        print(f"    Templates: {after}")

    timing_df = pd.read_csv(timing_path)
    timing_df.columns = timing_df.columns.str.strip().str.lower()

    print(f"    Users    : {len(users_df)}")
    print(f"    Segments with timing: {len(timing_df)}")
    print(f"    Template columns: {tmpls_df.columns.tolist()}")
    return users_df, tmpls_df, timing_df


def resolve_lifecycle_day(lifecycle_stage, days_since_signup, available_ranges):
    if not available_ranges:
        return None
    d  = int(days_since_signup) if pd.notna(days_since_signup) else 0
    lc = str(lifecycle_stage).lower().strip()

    def pick(keywords):
        for kw in keywords:
            for r in available_ranges:
                if kw in str(r): return r
        return available_ranges[0]

    if lc in ("churned", "inactive"):
        if d <= 7:   return pick(["D1-D7", "D0-D7"])
        elif d <= 15: return pick(["D8-D15"])
        else:         return pick(["D16-D30", "D16"])
    elif lc == "trial":
        if d <= 2:   return pick(["D0-D2"])
        elif d <= 5: return pick(["D3-D5"])
        else:        return pick(["D6-D7", "D5-D7", "D6"])
    else:
        if d <= 15:  return pick(["D8-D15", "D0-D15"])
        elif d <= 23: return pick(["D16-D23"])
        else:         return pick(["D24-D30", "D23-D30"])


def get_window_sequence(user_row, timing_df, total_notifs):
    sequence = []
    for i in range(1, 5):
        window = user_row.get(f"window_{i}", None)
        count  = user_row.get(f"window_{i}_count", 0)
        if pd.isna(window) or not str(window).strip():
            continue
        try:
            count = int(count) if pd.notna(count) else 0
        except (ValueError, TypeError):
            count = 0
        if count > 0:
            sequence.extend([str(window).strip()] * count)

    if len(sequence) == total_notifs:
        return sequence

    if len(sequence) < total_notifs:
        seg = str(user_row.get("segment_id", ""))
        seg_row = timing_df[timing_df["segment_id"] == seg]
        fallback = []
        if not seg_row.empty:
            r = seg_row.iloc[0]
            for i in range(1, 5):
                col = f"preferred_window_{i}"
                if col in r and pd.notna(r[col]) and str(r[col]).strip():
                    w = str(r[col]).strip()
                    if w not in fallback:
                        fallback.append(w)
        for w in ALL_WINDOWS:
            if w not in fallback:
                fallback.append(w)
        while len(sequence) < total_notifs:
            sequence.append(fallback[len(sequence) % len(fallback)])

    return sequence[:total_notifs]


def build_schedule_row(user_row, tmpls_df, timing_df):
    user_id    = str(user_row["user_id"])
    segment_id = str(user_row.get("segment_id", ""))
    lifecycle  = str(user_row.get("lifecycle_stage", "paid")).lower().strip()
    n_slots    = int(user_row.get("total_notifs", 3))
    days       = user_row.get("days_since_signup", 0)
    channel    = LIFECYCLE_CHANNEL.get(lifecycle, "push")

    # ── Resolve lifecycle_day — safe: day_range may not exist ──
    has_day_range = "day_range" in tmpls_df.columns
    if has_day_range:
        seg_ranges    = tmpls_df[tmpls_df["segment_id"]==segment_id]["day_range"].dropna().unique().tolist()
        lifecycle_day = resolve_lifecycle_day(lifecycle, days, seg_ranges)
        candidates    = tmpls_df[
            (tmpls_df["segment_id"]==segment_id) &
            (tmpls_df["day_range"]==lifecycle_day)
        ].copy() if lifecycle_day else pd.DataFrame()
        # fallback: all templates for this segment if phase match fails
        if candidates.empty:
            candidates = tmpls_df[tmpls_df["segment_id"]==segment_id].copy()
    else:
        # No day_range column — use all templates for this segment
        lifecycle_day = None
        candidates    = tmpls_df[tmpls_df["segment_id"]==segment_id].copy()

    window_sequence = get_window_sequence(user_row, timing_df, n_slots)

    row = {
        "user_id":       user_id,
        "segment_id":    segment_id,
        "lifecycle_day": lifecycle_day if lifecycle_day else "unknown",
    }

    for slot_pos in range(1, MAX_SLOTS + 1):
        if slot_pos > n_slots:
            row[f"notif_{slot_pos}_template"] = None
            row[f"notif_{slot_pos}_time"]     = None
            row[f"notif_{slot_pos}_channel"]  = None
            continue

        # cycle slot_num for variety — safe if slot_num missing
        if "slot_num" in candidates.columns:
            target_slot_num = ((slot_pos - 1) % 5) + 1
            slot_tmpls = candidates[candidates["slot_num"] == target_slot_num]
            if slot_tmpls.empty:
                slot_tmpls = candidates
        else:
            slot_tmpls = candidates

        if slot_tmpls.empty:
            row[f"notif_{slot_pos}_template"] = None
            row[f"notif_{slot_pos}_time"]     = None
            row[f"notif_{slot_pos}_channel"]  = None
            continue

        template_id   = slot_tmpls.iloc[0]["template_id"]
        chosen_window = window_sequence[slot_pos - 1]

        row[f"notif_{slot_pos}_template"] = template_id
        row[f"notif_{slot_pos}_time"]     = chosen_window
        row[f"notif_{slot_pos}_channel"]  = channel

    return row


def validate(output_df, users_df):
    print("\n[4] Validating output...")
    ok = True
    if len(output_df) == len(users_df):
        print(f"   Row count: {len(output_df)} (1 per user)")
    else:
        print(f"  ⚠  Row count: expected {len(users_df)}, got {len(output_df)}")
        ok = False

    bad_ch = [c for c in output_df["notif_1_channel"].dropna().unique() if "+" in str(c)]
    if bad_ch:
        print(f"  ⚠  Compound channel values: {bad_ch}")
    else:
        print("   Channel values valid")

    unknown = (output_df["lifecycle_day"] == "unknown").sum()
    if unknown:
        print(f"  ⚠  {unknown} users have lifecycle_day='unknown' (no day_range in templates)")
    else:
        print("   lifecycle_day resolved for all users")

    null_n1 = output_df["notif_1_template"].isna().sum()
    if null_n1:
        print(f"  ⚠  {null_n1} users missing notif_1_template")
        ok = False
    else:
        print("   notif_1_template populated for all users")

    print("   Slot columns structurally consistent")
    return ok


def print_summary(output_df):
    print("\n[5] Summary:")
    print(f"    Rows    : {len(output_df)}")
    print(f"    Columns : {len(output_df.columns)}")

    print("\n  Users per segment:")
    for seg, cnt in output_df["segment_id"].value_counts().items():
        print(f"    {seg:<30}  {cnt:>4} users")

    print("\n  lifecycle_day distribution:")
    for day, cnt in output_df["lifecycle_day"].value_counts().items():
        print(f"    {day:<30}  {cnt:>4} users")

    print("\n  notif_1_time distribution:")
    if "notif_1_time" in output_df.columns:
        for w, cnt in output_df["notif_1_time"].value_counts().items():
            print(f"    {w:<20}  {cnt:>4}")

    print("\n  Slot fill rate:")
    for i in range(1, 10):
        col = f"notif_{i}_template"
        if col in output_df.columns:
            filled = output_df[col].notna().sum()
            pct    = 100 * filled / len(output_df)
            bar    = "█" * int(pct / 5)
            print(f"    notif_{i}: {filled:>4}/{len(output_df)}  ({pct:5.1f}%)  {bar}")

    print("\n  Sample rows (3 users):")
    sample_cols = ["user_id", "segment_id", "lifecycle_day",
                   "notif_1_template", "notif_1_time", "notif_1_channel",
                   "notif_2_template", "notif_2_time"]
    print(output_df[[c for c in sample_cols if c in output_df.columns]]
          .head(3).to_string(index=False))


def run(
    users_path     = USERS_PATH,
    schedule_path  = SCHEDULE_PATH,
    templates_path = TEMPLATES_PATH,
    timing_path    = TIMING_PATH,
    output_dir     = OUTPUT_DIR,
):
    print("=" * 65)
    print("TASK 3 — PART 1: SCHEDULE GENERATOR (Iteration 0)")
    print("=" * 65)

    out_path = os.path.join(output_dir, "user_notification_schedule_iter0.csv")
    os.makedirs(output_dir, exist_ok=True)

    users_df, tmpls_df, timing_df = load_data(
        users_path, schedule_path, templates_path, timing_path)

    print(f"\n[2] Building schedules for {len(users_df)} users...")
    rows = [build_schedule_row(r, tmpls_df, timing_df) for _, r in users_df.iterrows()]
    output_df = pd.DataFrame(rows)

    validate(output_df, users_df)

    output_df.to_csv(out_path, index=False)
    print(f"\n[3] Saved → {out_path}")
    print(f"    {len(output_df)} rows  ×  {len(output_df.columns)} columns")

    print_summary(output_df)

    print("\n" + "=" * 65)
    print("  PART 1 COMPLETE — iter0 schedule ready")
    print(f"   {out_path}")
    print("=" * 65)

    return output_df


if __name__ == "__main__":
    run()

TASK 3 — PART 1: SCHEDULE GENERATOR (Iteration 0)
[1] Loading input files...
    Templates: 50
    Users    : 61
    Segments with timing: 10
    Template columns: ['template_id', 'segment_id', 'slot_num', 'performance_label', 'best_window', 'best_ctr', 'best_engagement', 'posterior_mean', 'suppressed']

[2] Building schedules for 61 users...

[4] Validating output...
   Row count: 61 (1 per user)
   Channel values valid
  ⚠  61 users have lifecycle_day='unknown' (no day_range in templates)
   notif_1_template populated for all users
   Slot columns structurally consistent

[3] Saved → /content/iteration_0_before_learning/user_notification_schedule_iter0.csv
    61 rows  ×  30 columns

[5] Summary:
    Rows    : 61
    Columns : 30

  Users per segment:
    Flight-Risk Subscribers           12 users
    High-Intent Trialists             10 users
    Lost Customers                     8 users
    Dormant Users                      8 users
    Slipping Subscribers               6 users
 

## -- Classifier --

In [45]:
import os
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

# PATHS

EXPERIMENT_PATH = "/content/experiment_results.csv"
OUTPUT_PATH     = "/content/classified_results.csv"

# THRESHOLDS  (from PS)

CTR_GOOD          = 0.15
CTR_BAD           = 0.05
ENGAGEMENT_GOOD   = 0.40
ENGAGEMENT_BAD    = 0.20
UNINSTALL_VETO    = 0.05
MIN_SENDS         = 30


# CLASSIFICATION LOGIC

def classify_row(row):
    """
    Decision tree per row:

    1. total_sends < 30        → INSUFFICIENT_DATA, reward=0.0
    2. uninstall_rate > 0.05   → BAD,               reward=-1.0
    3. CTR>15% AND eng>40%     → GOOD,              reward=+1.0
    4. CTR<5%  AND eng<20%     → BAD,               reward=-1.0
    5. everything else         → NEUTRAL,            reward= 0.0

    Mixed signals (e.g. CTR=18% but eng=25%) → NEUTRAL
    Only label GOOD if BOTH metrics clear their threshold.
    Only label BAD  if BOTH metrics fall below their threshold.
    """
    sends      = int(row.get("total_sends", 0))
    ctr        = float(row.get("ctr", 0))
    engagement = float(row.get("engagement_rate", 0))
    uninstall  = float(row.get("uninstall_rate", 0))

    # Guard: insufficient data
    if sends < MIN_SENDS:
        return "INSUFFICIENT_DATA", 0.0

    # Override: uninstall veto
    if uninstall > UNINSTALL_VETO:
        return "BAD", -1.0

    # PS thresholds
    if ctr > CTR_GOOD and engagement > ENGAGEMENT_GOOD:
        return "GOOD", 1.0

    if ctr < CTR_BAD and engagement < ENGAGEMENT_BAD:
        return "BAD", -1.0

    return "NEUTRAL", 0.0



# VALIDATE INPUT SCHEMA

REQUIRED_COLS = [
    "template_id", "segment_id", "notification_window",
    "total_sends", "ctr", "engagement_rate", "uninstall_rate",
]

def validate_input(df):
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"experiment_results.csv is missing required columns: {missing}\n"
            f"Available: {df.columns.tolist()}"
        )
    nulls = {c: df[c].isna().sum() for c in REQUIRED_COLS if df[c].isna().sum() > 0}
    if nulls:
        print(f"  [WARN] Null values in: {nulls}  — treating as 0")
        for c in nulls:
            df[c] = df[c].fillna(0)
    return df


# SUMMARY

def print_summary(classified_df):
    print("\n[3] Classification Summary:")
    print(f"    Total rows : {len(classified_df)}")

    print("\n  Label distribution:")
    for label, cnt in classified_df["label"].value_counts().items():
        pct = 100 * cnt / len(classified_df)
        bar = "█" * int(pct / 3)
        print(f"    {label:<20}  {cnt:>4}  ({pct:5.1f}%)  {bar}")

    print("\n  Reward distribution:")
    print(f"    {classified_df['reward'].value_counts().sort_index().to_dict()}")

    print("\n  Per segment breakdown:")
    for seg, grp in classified_df.groupby("segment_id"):
        dist = grp["label"].value_counts().to_dict()
        n    = len(grp)
        good_pct = 100 * dist.get("GOOD", 0) / n
        bad_pct  = 100 * dist.get("BAD",  0) / n
        print(f"    {seg:<30}  "
              f"GOOD={dist.get('GOOD',0):>3}({good_pct:4.1f}%)  "
              f"BAD={dist.get('BAD',0):>3}({bad_pct:4.1f}%)  "
              f"NEUTRAL={dist.get('NEUTRAL',0):>3}")

    # Flag rows where our label conflicts with SpeakX's performance_status
    if "performance_status" in classified_df.columns:
        conflicts = classified_df[
            classified_df["label"].isin(["GOOD","BAD","NEUTRAL"]) &
            classified_df["performance_status"].isin(["GOOD","BAD","NEUTRAL"]) &
            (classified_df["label"] != classified_df["performance_status"])
        ]
        if len(conflicts) > 0:
            print(f"\n  [INFO] {len(conflicts)} rows where our label differs "
                  f"from SpeakX's performance_status.")
            print("         Both columns kept — Part 3 uses 'label' (ours).")

    print("\n  CTR range by label:")
    for label in ["GOOD", "NEUTRAL", "BAD"]:
        grp = classified_df[classified_df["label"] == label]["ctr"]
        if len(grp) > 0:
            print(f"    {label:<10}  "
                  f"mean={grp.mean():.3f}  "
                  f"min={grp.min():.3f}  "
                  f"max={grp.max():.3f}")

    print("\n  Sample rows (5):")
    cols = ["template_id", "segment_id", "notification_window",
            "ctr", "engagement_rate", "uninstall_rate", "label", "reward"]
    print(classified_df[[c for c in cols if c in classified_df.columns]]
          .head(5).to_string(index=False))


# ENTRY POINT

def run(
    experiment_path = EXPERIMENT_PATH,
    output_path     = OUTPUT_PATH,
):
    """
    Classify experiment_results.csv → classified_results.csv.
    Works with both SpeakX's real file and the demo-generated file.
    """
    print("=" * 65)
    print("TASK 3 — PART 2: CLASSIFICATION ENGINE")
    print("=" * 65)

    # Load
    print(f"\n[1] Loading {experiment_path}...")
    df = pd.read_csv(experiment_path)
    df.columns = df.columns.str.strip().str.lower()
    print(f"    {len(df)} rows × {len(df.columns)} columns")

    # Validate
    print("\n[2] Validating schema...")
    df = validate_input(df)
    print("    Schema valid")

    # Classify every row
    labels, rewards = zip(*df.apply(classify_row, axis=1))
    df["label"]  = list(labels)
    df["reward"] = list(rewards)

    # Save
    df.to_csv(output_path, index=False)
    print(f"\n    Saved → {output_path}")
    print(f"    {len(df)} rows × {len(df.columns)} columns")

    # Summary
    print_summary(df)

    print("\n" + "=" * 65)
    print(" PART 2 COMPLETE — classified_results.csv ready for Part 3")
    print(f"   {output_path}")
    print("=" * 65)

    return df


if __name__ == "__main__":
    run()

TASK 3 — PART 2: CLASSIFICATION ENGINE

[1] Loading /content/experiment_results.csv...


FileNotFoundError: [Errno 2] No such file or directory: '/content/experiment_results.csv'

## -- Learning Engine --

In [46]:
import os
import json
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime

warnings.filterwarnings("ignore")
np.random.seed(42)

# PATHS

CLASSIFIED_PATH  = "/content/classified_results.csv"
USERS_PATH       = "/content/iteration_0_before_learning/user_segments.csv"
SCHEDULE_PATH    = "/content/iteration_0_before_learning/user_notification_schedule_iter0.csv"
TIMING_PATH      = "/content/iteration_0_before_learning/timing_recommendations.csv"
OUTPUT_DIR       = "/content/iteration_1_after_learning"

# CONSTANTS

MAX_SLOTS = 9

ALL_WINDOWS = [
    "early_morning", "mid_morning", "afternoon",
    "late_afternoon", "evening", "night",
]

LIFECYCLE_CHANNEL = {
    "trial":    "push",
    "paid":     "push",
    "inactive": "push",
    "churned":  "email",
}

# Guardrail: if segment avg uninstall > 2% → reduce notifs by 2
GUARDRAIL_UNINSTALL_THRESHOLD = 0.02
GUARDRAIL_REDUCE_BY           = 2
FREQ_FLOOR                    = 1   # never go below 1 notif/day


# ══════════════════════════════════════════════════════════════════
# STEP 1 — LOAD
# ══════════════════════════════════════════════════════════════════
def load_inputs(classified_path, users_path, schedule_path, timing_path):
    print("[1] Loading inputs...")

    clf_df  = pd.read_csv(classified_path)
    clf_df.columns = clf_df.columns.str.strip().str.lower()

    users_df = pd.read_csv(users_path)
    users_df.columns = users_df.columns.str.strip().str.lower()

    sched_df = pd.read_csv(schedule_path)
    sched_df.columns = sched_df.columns.str.strip().str.lower()

    timing_df = pd.read_csv(timing_path)
    timing_df.columns = timing_df.columns.str.strip().str.lower()

    # Merge total_notifs + window counts onto users
    want = ["user_id", "total_notifs",
            "window_1", "window_1_count", "window_2", "window_2_count",
            "window_3", "window_3_count", "window_4", "window_4_count"]
    sched_cols = [c for c in want if c in sched_df.columns]
    users_df = users_df.merge(sched_df[sched_cols], on="user_id", how="left")
    users_df["total_notifs"] = users_df["total_notifs"].fillna(3).astype(int)

    print(f"    classified rows : {len(clf_df)}")
    print(f"    users           : {len(users_df)}")
    print(f"    label dist      : {clf_df['label'].value_counts().to_dict()}")
    return clf_df, users_df, timing_df


# ══════════════════════════════════════════════════════════════════
# STEP 2 — BUILD BETA TABLE
#
# Beta(α, β) per arm  where arm = "template_id|notification_window"
#
# Update rules (from classified_results):
#   label == GOOD              → α += 1  (success)
#   label == BAD               → β += 1  (failure)
#   label == NEUTRAL           → no update (no evidence either way)
#   label == INSUFFICIENT_DATA → no update (too few sends to trust)
#
# Posterior mean of Beta(α,β) = α / (α+β)
#   → represents our learned belief that this arm succeeds
#   → used to rerank time windows per segment
# ══════════════════════════════════════════════════════════════════
def build_beta_table(clf_df):
    print("[2] Building Thompson Sampling Beta table...")

    # All arms start at Beta(1,1) — uninformative prior
    # Meaning: equal probability of success and failure before any data
    alpha = defaultdict(lambda: 1.0)
    beta  = defaultdict(lambda: 1.0)

    updates = defaultdict(int)

    for _, row in clf_df.iterrows():
        label  = str(row.get("label", "")).strip().upper()
        tmpl   = str(row["template_id"]).strip()
        window = str(row.get("notification_window", "")).strip()

        if not window or not tmpl:
            continue

        arm = f"{tmpl}|{window}"

        if label == "GOOD":
            alpha[arm] += 1.0
            updates["good"] += 1
        elif label == "BAD":
            beta[arm] += 1.0
            updates["bad"] += 1
        # NEUTRAL and INSUFFICIENT_DATA: no update

    total_arms = len(set(list(alpha.keys()) + list(beta.keys())))
    print(f"    Arms updated : GOOD={updates['good']}  BAD={updates['bad']}")
    print(f"    Unique arms  : {total_arms}  "
          f"(untested arms keep Beta(1,1) for exploration)")

    return alpha, beta


# ══════════════════════════════════════════════════════════════════
# STEP 3 — IDENTIFY SUPPRESSED ARMS AND TEMPLATES
#
# Arm-level suppression:
#   An arm (template|window) is suppressed if its label == BAD
#   → excluded from Thompson Sampling pool
#
# Template-level suppression (fully suppressed):
#   A template is fully suppressed if ALL its observed arms are BAD
#   AND it has no GOOD or NEUTRAL arms anywhere
#   → never assigned in iter1 schedule
# ══════════════════════════════════════════════════════════════════
def build_suppression_sets(clf_df):
    print("[3] Building suppression sets...")

    # Arm-level: individual (template, window) pairs labelled BAD
    bad_arms = set()
    for _, row in clf_df.iterrows():
        if str(row.get("label","")).upper() == "BAD":
            arm = f"{row['template_id']}|{row['notification_window']}"
            bad_arms.add(arm)

    # Template-level: any template that has at least one non-BAD observed arm
    # is kept active. Only fully-BAD templates are suppressed entirely.
    template_labels = clf_df.groupby("template_id")["label"].apply(list).to_dict()
    fully_suppressed = set()
    for tmpl, labels in template_labels.items():
        non_bad = [l for l in labels if l.upper() not in ("BAD", "INSUFFICIENT_DATA")]
        if len(non_bad) == 0:
            fully_suppressed.add(str(tmpl))

    print(f"    Bad arms (template×window)   : {len(bad_arms)}")
    print(f"    Fully suppressed templates   : {len(fully_suppressed)}  "
          f"→ {sorted(fully_suppressed)}")
    return bad_arms, fully_suppressed


# ══════════════════════════════════════════════════════════════════
# STEP 4 — GUARDRAIL CHECK
#
# Per PS: if segment avg uninstall_rate > 2% → reduce notifs by 2
# ══════════════════════════════════════════════════════════════════
def get_guardrail_segments(clf_df):
    if "uninstall_rate" not in clf_df.columns:
        return set()

    seg_col = "segment_id" if "segment_id" in clf_df.columns else None
    if not seg_col:
        return set()

    flagged = set(
        clf_df.groupby(seg_col)["uninstall_rate"].mean()
        .pipe(lambda s: s[s > GUARDRAIL_UNINSTALL_THRESHOLD])
        .index.astype(str)
    )
    if flagged:
        print(f"    Guardrail triggered for: {flagged}")
    else:
        print("    No guardrail triggers (all segment uninstall rates < 2%)")
    return flagged


# ══════════════════════════════════════════════════════════════════
# STEP 5 — INFER slot_num PER TEMPLATE
#
# We don't have the raw message_templates.csv, but template IDs
# within each segment are numbered sequentially (e.g. TPL_0001..0005
# for AI-Powered Learners). Sorting by ID within segment gives slot_nums 1..5.
# This matches Part 1's cycling logic: slot_pos → slot_num = ((slot_pos-1)%5)+1
# ══════════════════════════════════════════════════════════════════
def infer_slot_nums(clf_df):
    slot_map = {}
    for seg, grp in clf_df.groupby("segment_id"):
        sorted_tmpls = sorted(grp["template_id"].unique())
        for i, tmpl in enumerate(sorted_tmpls):
            slot_map[str(tmpl)] = (i % 5) + 1
    return slot_map


# ══════════════════════════════════════════════════════════════════
# STEP 6 — UPDATE TIMING RECOMMENDATIONS
#
# For each segment × window:
#   Collect all arms (template|window) for templates in this segment
#   posterior_mean = mean(α/(α+β)) across those arms
#   → this is our learned belief that this window works for this segment
#
# Re-rank windows per segment by posterior_mean descending.
# Windows with BAD-heavy arms rank lower. GOOD-heavy arms rank higher.
#
# Example — At-Risk Trialists:
#   early_morning arms: α=1 (prior), β=2 (3 BAD updates) → mean=0.33
#   afternoon arms:     α=2 (1 GOOD), β=1 (prior)       → mean=0.67
#   → afternoon moves to window_1 in iter1 timing
# ══════════════════════════════════════════════════════════════════
def update_timing_recommendations(clf_df, timing_df, alpha, beta):
    print("[5] Updating timing recommendations via Beta posteriors...")

    # Build segment → templates mapping
    seg_templates = clf_df.groupby("segment_id")["template_id"].apply(
        lambda x: list(x.astype(str).unique())
    ).to_dict()

    rows = []
    for _, orig_row in timing_df.iterrows():
        seg   = str(orig_row["segment_id"])
        tmpls = seg_templates.get(seg, [])

        # Compute posterior mean per window for this segment
        window_scores = {}
        for w in ALL_WINDOWS:
            arm_means = []
            for t in tmpls:
                arm = f"{t}|{w}"
                a = alpha[arm]
                b = beta[arm]
                arm_means.append(a / (a + b))
            window_scores[w] = float(np.mean(arm_means)) if arm_means else 0.5

        # Observed engagement rates for context
        obs_eng = {}
        for w in ALL_WINDOWS:
            w_rows = clf_df[
                (clf_df["segment_id"] == seg) &
                (clf_df["notification_window"] == w)
            ]
            if len(w_rows) > 0:
                obs_eng[w] = float(w_rows["engagement_rate"].mean())

        # Re-rank windows by learned posterior mean
        ranked = sorted(window_scores.items(), key=lambda x: x[1], reverse=True)

        new_row = {"segment_id": seg}
        for rank, (w, score) in enumerate(ranked[:4], 1):
            # Get original engagement for this window if observed
            orig_eng = obs_eng.get(w, None)
            if orig_eng is None:
                # Fallback: find original timing engagement for this window
                for i in range(1, 5):
                    if str(orig_row.get(f"preferred_window_{i}", "")) == w:
                        orig_eng = float(orig_row.get(f"expected_engagement_{i}", 0.3) or 0.3)
                        break
                if orig_eng is None:
                    orig_eng = 0.3

            new_row[f"preferred_window_{rank}"]    = w
            new_row[f"expected_ctr_{rank}"]        = round(score, 4)
            new_row[f"expected_engagement_{rank}"] = round(float(orig_eng), 4)

        # Fill any empty rank slots with empty strings (match schema)
        for rank in range(len(ranked[:4]) + 1, 5):
            new_row[f"preferred_window_{rank}"]    = None
            new_row[f"expected_ctr_{rank}"]        = None
            new_row[f"expected_engagement_{rank}"] = None

        rows.append(new_row)

    timing_iter1 = pd.DataFrame(rows)

    # Print timing changes for transparency
    print("\n  Window ranking changes (iter0 → iter1):")
    for _, r1 in timing_iter1.iterrows():
        seg    = r1["segment_id"]
        orig   = timing_df[timing_df["segment_id"] == seg]
        if orig.empty:
            continue
        o = orig.iloc[0]
        w0 = str(o.get("preferred_window_1", ""))
        w1 = str(r1.get("preferred_window_1", ""))
        changed = "←CHANGED" if w0 != w1 else ""
        print(f"    {seg:<30}  iter0 top={w0:<15}  iter1 top={w1:<15}  {changed}")

    return timing_iter1


# ══════════════════════════════════════════════════════════════════
# STEP 7 — UPDATE MESSAGE TEMPLATES TABLE
#
# Adds performance_label + best_window + posterior_mean columns.
# Fully suppressed templates flagged as SUPPRESSED.
# GOOD templates flagged for promotion.
# ══════════════════════════════════════════════════════════════════
def update_message_templates(clf_df, alpha, beta, slot_map, fully_suppressed):
    print("\n[6] Updating message templates table...")

    rows = []
    # One row per template (summarised across all observed windows)
    for tmpl_id, grp in clf_df.groupby("template_id"):
        tmpl_id = str(tmpl_id)
        seg     = str(grp["segment_id"].iloc[0])

        # Best observed window (highest CTR among non-BAD)
        non_bad = grp[grp["label"] != "BAD"]
        if len(non_bad) > 0:
            best_row = non_bad.loc[non_bad["ctr"].idxmax()]
            best_win = best_row["notification_window"]
            best_ctr = round(float(best_row["ctr"]), 4)
            best_eng = round(float(best_row["engagement_rate"]), 4)
        else:
            # All arms BAD → use the least-bad window
            best_row = grp.loc[grp["ctr"].idxmax()]
            best_win = best_row["notification_window"]
            best_ctr = round(float(best_row["ctr"]), 4)
            best_eng = round(float(best_row["engagement_rate"]), 4)

        # Overall performance label for this template
        if tmpl_id in fully_suppressed:
            perf_label = "SUPPRESSED"
        elif "GOOD" in grp["label"].values:
            perf_label = "GOOD"
        elif "NEUTRAL" in grp["label"].values:
            perf_label = "NEUTRAL"
        else:
            perf_label = "BAD"

        # Posterior mean across all windows (overall template quality)
        arm_means = []
        for w in ALL_WINDOWS:
            arm = f"{tmpl_id}|{w}"
            a, b = alpha[arm], beta[arm]
            arm_means.append(a / (a + b))
        posterior_mean = round(float(np.mean(arm_means)), 4)

        rows.append({
            "template_id":      tmpl_id,
            "segment_id":       seg,
            "slot_num":         slot_map.get(tmpl_id, 0),
            "performance_label":perf_label,
            "best_window":      best_win,
            "best_ctr":         best_ctr,
            "best_engagement":  best_eng,
            "posterior_mean":   posterior_mean,
            "suppressed":       tmpl_id in fully_suppressed,
        })

    tmpls_iter1 = pd.DataFrame(rows).sort_values(
        ["segment_id", "slot_num"]
    ).reset_index(drop=True)

    counts = tmpls_iter1["performance_label"].value_counts().to_dict()
    print(f"    Template labels: {counts}")
    return tmpls_iter1


# ══════════════════════════════════════════════════════════════════
# STEP 8 — THOMPSON SAMPLING SELECT
#
# Given a list of (template, window) arm candidates:
#   1. Exclude BAD arms (suppressed at arm level)
#   2. For each remaining arm: sample from Beta(α, β)
#   3. Return the arm with the highest sample
#
# Why this works:
#   - A GOOD arm has α=2, β=1 → Beta(2,1) → samples mostly 0.5-0.9
#   - A BAD arm is excluded
#   - An untested arm has α=1, β=1 → Beta(1,1) = Uniform → sometimes
#     samples high → gets explored naturally
#   - Arms with many GOODs (α>>β) sample near 1.0 consistently → win
# ══════════════════════════════════════════════════════════════════
def thompson_select(candidate_templates, windows, alpha, beta,
                    bad_arms, rng):
    """
    Returns: (chosen_template_id, chosen_window)
    """
    best_sample = -1.0
    best_tmpl   = None
    best_window = None

    for tmpl in candidate_templates:
        for w in windows:
            arm = f"{tmpl}|{w}"
            if arm in bad_arms:
                continue   # never sample from a suppressed arm

            a = alpha[arm]
            b = beta[arm]
            sample = rng.beta(a, b)   # sample from Beta(α, β)

            if sample > best_sample:
                best_sample = sample
                best_tmpl   = tmpl
                best_window = w

    return best_tmpl, best_window


# ══════════════════════════════════════════════════════════════════
# STEP 9 — RESOLVE lifecycle_day
# (Identical to Part 1 — must stay in sync)
# ══════════════════════════════════════════════════════════════════
def resolve_lifecycle_day(lifecycle_stage, days_since_signup, available_ranges):
    if not available_ranges:
        return None
    d  = int(days_since_signup) if pd.notna(days_since_signup) else 0
    lc = str(lifecycle_stage).lower().strip()

    def pick(keywords):
        for kw in keywords:
            for r in available_ranges:
                if kw in str(r):
                    return r
        return available_ranges[0]

    if lc in ("churned", "inactive"):
        if d <= 7:  return pick(["D1-D7", "D0-D7"])
        elif d <= 15: return pick(["D8-D15"])
        else:         return pick(["D16-D30", "D16"])
    elif lc == "trial":
        if d <= 2:  return pick(["D0-D2"])
        elif d <= 5: return pick(["D3-D5"])
        else:        return pick(["D6-D7", "D5-D7", "D6"])
    else:
        if d <= 15: return pick(["D8-D15", "D0-D15"])
        elif d <= 23: return pick(["D16-D23"])
        else:         return pick(["D24-D30", "D23-D30"])


# ══════════════════════════════════════════════════════════════════
# STEP 10 — BUILD ITER1 SCHEDULE
#
# Core change from Part 1:
#   Part 1: window = window_sequence[slot_pos - 1]  (Task 2 heuristic)
#   Part 3: (template, window) = thompson_select()  (learned from data)
#
# This is where timing patterns are applied:
#   If At-Risk Trialists' early_morning arms all have high β →
#   thompson_select rarely picks early_morning →
#   iter1 schedule shows afternoon/evening instead.
# ══════════════════════════════════════════════════════════════════
def build_schedule_iter1(users_df, clf_df, alpha, beta,
                          bad_arms, fully_suppressed, slot_map,
                          guardrail_segs, rng):
    print("\n[7] Building iter1 schedule with Thompson Sampling...")

    # All templates per segment (sorted, for slot_num cycling)
    seg_templates = {}
    for seg, grp in clf_df.groupby("segment_id"):
        sorted_tmpls = sorted(grp["template_id"].astype(str).unique())
        seg_templates[str(seg)] = sorted_tmpls

    # day_range values available per segment (for lifecycle_day resolution)
    # Since we don't have message_templates.csv, we infer from clf_df if possible
    # Fallback: use the standard ranges based on lifecycle_stage
    STANDARD_RANGES = {
        "trial":    ["D0-D2", "D3-D5", "D6-D7"],
        "paid":     ["D8-D15", "D16-D23", "D24-D30"],
        "churned":  ["D1-D7 (Post-Churn)", "D8-D15 (Post-Churn)",
                     "D16-D30 (Post-Churn)"],
        "inactive": ["D1-D7 (Post-Churn)", "D8-D15 (Post-Churn)",
                     "D16-D30 (Post-Churn)"],
    }

    records  = []
    skipped  = []

    for _, user in users_df.iterrows():
        uid       = str(user["user_id"])
        seg       = str(user.get("segment_id", ""))
        lifecycle = str(user.get("lifecycle_stage", "paid")).lower().strip()
        days      = user.get("days_since_signup", 0)
        n_slots   = int(user.get("total_notifs", 3))
        channel   = LIFECYCLE_CHANNEL.get(lifecycle, "push")

        # Guardrail: reduce frequency if segment has high uninstall
        if seg in guardrail_segs:
            n_slots = max(FREQ_FLOOR, n_slots - GUARDRAIL_REDUCE_BY)

        n_slots = min(n_slots, MAX_SLOTS)

        # Resolve lifecycle day
        available_ranges = STANDARD_RANGES.get(lifecycle, STANDARD_RANGES["paid"])
        lifecycle_day = resolve_lifecycle_day(lifecycle, days, available_ranges)

        # Get all active templates for this segment (exclude fully suppressed)
        all_seg_tmpls = seg_templates.get(seg, [])
        active_tmpls  = [t for t in all_seg_tmpls if t not in fully_suppressed]

        if not active_tmpls:
            # All templates suppressed → use any (last resort fallback)
            active_tmpls = all_seg_tmpls
            if not active_tmpls:
                skipped.append(uid)
                continue

        rec = {
            "user_id":      uid,
            "segment_id":   seg,
            "lifecycle_day": lifecycle_day if lifecycle_day else "unknown",
        }

        for slot_pos in range(1, MAX_SLOTS + 1):

            if slot_pos > n_slots:
                rec[f"notif_{slot_pos}_template"] = None
                rec[f"notif_{slot_pos}_time"]     = None
                rec[f"notif_{slot_pos}_channel"]  = None
                continue

            # Which slot_num style does this position target?
            target_slot_num = ((slot_pos - 1) % 5) + 1

            # Get templates for this slot_num (maintain style variety)
            slot_tmpls = [t for t in active_tmpls
                          if slot_map.get(t, 0) == target_slot_num]
            if not slot_tmpls:
                slot_tmpls = active_tmpls   # fallback: any active template

            # ── THOMPSON SAMPLING: pick (template, window) jointly ──
            chosen_tmpl, chosen_window = thompson_select(
                slot_tmpls, ALL_WINDOWS, alpha, beta, bad_arms, rng
            )

            # Absolute fallback if all arms suppressed
            if chosen_tmpl is None:
                chosen_tmpl   = slot_tmpls[0]
                chosen_window = ALL_WINDOWS[slot_pos % len(ALL_WINDOWS)]

            rec[f"notif_{slot_pos}_template"] = chosen_tmpl
            rec[f"notif_{slot_pos}_time"]     = chosen_window
            rec[f"notif_{slot_pos}_channel"]  = channel

        records.append(rec)

    if skipped:
        print(f"    Skipped (no templates): {len(skipped)} users")
    print(f"    Scheduled: {len(records)} users")

    return pd.DataFrame(records)


# ══════════════════════════════════════════════════════════════════
# STEP 11 — SAVE BANDIT STATE (human-readable JSON)
# ══════════════════════════════════════════════════════════════════
def save_bandit_state(alpha, beta, clf_df, output_dir):
    arms_data = {}
    all_arms  = set(list(alpha.keys()) + list(beta.keys()))

    # Only save arms that were actually updated (not just defaultdict defaults)
    observed_arms = set()
    for _, row in clf_df.iterrows():
        arm = f"{row['template_id']}|{row['notification_window']}"
        if str(row.get("label","")).upper() in ("GOOD", "BAD"):
            observed_arms.add(arm)

    for arm in sorted(observed_arms):
        a = alpha[arm]
        b = beta[arm]
        arms_data[arm] = {
            "alpha":          round(a, 2),
            "beta":           round(b, 2),
            "posterior_mean": round(a / (a + b), 4),
            "n_updates":      int(a + b - 2),   # subtract the prior Beta(1,1)
            "status":         ("GOOD" if a > b else "BAD" if b > a else "NEUTRAL"),
        }

    state = {
        "created_at":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "algorithm":        "Thompson Sampling (Beta-Bernoulli)",
        "arm_definition":   "template_id|notification_window",
        "prior":            "Beta(1,1) — uninformative",
        "update_rule":      "GOOD→alpha+=1  BAD→beta+=1  NEUTRAL→no update",
        "total_arms_saved": len(arms_data),
        "arms":             arms_data,
    }

    path = os.path.join(output_dir, "bandit_state.json")
    with open(path, "w") as f:
        json.dump(state, f, indent=2)
    print(f"    bandit_state.json → {path}  ({len(arms_data)} arms)")
    return path


# ══════════════════════════════════════════════════════════════════
# STEP 12 — VALIDATE OUTPUTS
# ══════════════════════════════════════════════════════════════════
def validate(schedule_df, users_df, timing_df, timing_iter1):
    print("\n[8] Validating outputs...")
    ok = True

    # Row count
    if len(schedule_df) == len(users_df):
        print(f"  ✅ Schedule rows: {len(schedule_df)} (1 per user)")
    else:
        print(f"  ⚠  Schedule rows: expected {len(users_df)}, got {len(schedule_df)}")
        ok = False

    # No compound channel strings
    ch_vals = schedule_df["notif_1_channel"].dropna().unique()
    bad_ch  = [c for c in ch_vals if "+" in str(c)]
    if bad_ch:
        print(f"  ⚠  Compound channel values: {bad_ch}")
        ok = False
    else:
        print("  ✅ Channel values valid")

    # notif_1 always populated
    null_n1 = schedule_df["notif_1_template"].isna().sum()
    if null_n1:
        print(f"  ⚠  {null_n1} users missing notif_1_template")
        ok = False
    else:
        print("  ✅ notif_1_template populated for all users")

    # Timing schema matches original
    expected_cols = [c for c in timing_df.columns if c in timing_iter1.columns]
    if len(expected_cols) >= 5:
        print("  ✅ timing_recommendations_iter1 schema valid")
    else:
        print(f"  ⚠  timing schema mismatch: {timing_iter1.columns.tolist()}")

    if ok:
        print("  ✅ All validation checks passed")
    return ok


# ══════════════════════════════════════════════════════════════════
# STEP 13 — PRINT LEARNING SUMMARY
# ══════════════════════════════════════════════════════════════════
def print_learning_summary(clf_df, schedule_df, timing_df,
                            timing_iter1, tmpls_iter1,
                            bad_arms, fully_suppressed, guardrail_segs):
    print("\n" + "═"*65)
    print("LEARNING SUMMARY — What changed from iter0 → iter1")
    print("═"*65)

    print(f"\n  Templates fully suppressed : {len(fully_suppressed)}")
    for t in sorted(fully_suppressed):
        seg = clf_df[clf_df["template_id"]==t]["segment_id"].iloc[0] if len(clf_df[clf_df["template_id"]==t]) else "?"
        print(f"    {t} ({seg}) — all observed arms were BAD")

    print(f"\n  Arm-level suppressions     : {len(bad_arms)}")
    print(f"  (template×window combos that will no longer be scheduled)")

    print(f"\n  Guardrail segments         : {len(guardrail_segs)}")
    if guardrail_segs:
        print(f"    {guardrail_segs} → notifs reduced by {GUARDRAIL_REDUCE_BY}/day")

    print("\n  Timing window shifts (top window per segment):")
    for _, r1 in timing_iter1.iterrows():
        seg  = r1["segment_id"]
        orig = timing_df[timing_df["segment_id"] == seg]
        if orig.empty: continue
        o    = orig.iloc[0]
        w0   = str(o.get("preferred_window_1",""))
        w1   = str(r1.get("preferred_window_1",""))
        ctr0 = float(o.get("expected_ctr_1", 0))
        ctr1 = float(r1.get("expected_ctr_1", 0))
        changed = " ← TIMING SHIFT" if w0 != w1 else ""
        print(f"    {seg:<30}  {w0:<15} ({ctr0:.3f}) → "
              f"{w1:<15} ({ctr1:.3f}){changed}")

    print("\n  Template performance labels (iter1):")
    for label, cnt in tmpls_iter1["performance_label"].value_counts().items():
        print(f"    {label:<12}  {cnt:>3} templates")

    print("\n  Schedule channel distribution (iter1):")
    print(f"    {schedule_df['notif_1_channel'].value_counts().to_dict()}")

    print("\n  notif_1_time distribution (iter1):")
    for w, cnt in schedule_df["notif_1_time"].value_counts().items():
        pct = 100 * cnt / len(schedule_df)
        bar = "█" * int(pct / 4)
        print(f"    {w:<15}  {cnt:>4}  ({pct:4.1f}%)  {bar}")


# ══════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════
def run(
    classified_path = CLASSIFIED_PATH,
    users_path      = USERS_PATH,
    schedule_path   = SCHEDULE_PATH,
    timing_path     = TIMING_PATH,
    output_dir      = OUTPUT_DIR,
    seed            = 42,
):
    """
    Run the learning engine. Accepts custom paths for demo day runtime.
    Returns dict of all output dataframes.
    """
    print("=" * 65)
    print("TASK 3 — PART 3: LEARNING ENGINE (Iteration 1)")
    print("=" * 65)

    rng = np.random.default_rng(seed)
    os.makedirs(output_dir, exist_ok=True)

    # Load
    clf_df, users_df, timing_df = load_inputs(
        classified_path, users_path, schedule_path, timing_path
    )

    # Build Beta table
    alpha, beta = build_beta_table(clf_df)

    # Suppression sets
    print()
    bad_arms, fully_suppressed = build_suppression_sets(clf_df)

    # Guardrail
    print()
    print("[4] Guardrail check...")
    guardrail_segs = get_guardrail_segments(clf_df)

    # Infer slot_nums
    slot_map = infer_slot_nums(clf_df)

    # Update timing
    print()
    timing_iter1 = update_timing_recommendations(clf_df, timing_df, alpha, beta)

    # Update templates
    tmpls_iter1 = update_message_templates(
        clf_df, alpha, beta, slot_map, fully_suppressed
    )

    # Build schedule
    schedule_iter1 = build_schedule_iter1(
        users_df, clf_df, alpha, beta,
        bad_arms, fully_suppressed, slot_map,
        guardrail_segs, rng
    )

    # Validate
    validate(schedule_iter1, users_df, timing_df, timing_iter1)

    # Save outputs
    print("\n[9] Saving outputs...")
    outputs = {
        "user_notification_schedule_iter1.csv": schedule_iter1,
        "timing_recommendations_iter1.csv":     timing_iter1,
        "message_templates_iter1.csv":          tmpls_iter1,
    }
    for fname, df in outputs.items():
        path = os.path.join(output_dir, fname)
        df.to_csv(path, index=False)
        print(f"    ✅ {fname}  ({len(df)} rows × {len(df.columns)} cols)")

    save_bandit_state(alpha, beta, clf_df, output_dir)

    # Learning summary
    print_learning_summary(
        clf_df, schedule_iter1, timing_df, timing_iter1,
        tmpls_iter1, bad_arms, fully_suppressed, guardrail_segs
    )

    print("\n" + "=" * 65)
    print("✅ PART 3 COMPLETE")
    print(f"   {output_dir}/")
    print("=" * 65)

    return {
        "schedule":      schedule_iter1,
        "timing":        timing_iter1,
        "templates":     tmpls_iter1,
        "alpha":         dict(alpha),
        "beta":          dict(beta),
        "bad_arms":      bad_arms,
        "fully_suppressed": fully_suppressed,
        "guardrail_segs": guardrail_segs,
    }


if __name__ == "__main__":
    run()


TASK 3 — PART 3: LEARNING ENGINE (Iteration 1)
[1] Loading inputs...


FileNotFoundError: [Errno 2] No such file or directory: '/content/classified_results.csv'

In [47]:
import os
import json
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────
# PATHS (override via run())
# ──────────────────────────────────────────────────────────────────
ITER0_SCHEDULE_PATH  = "/content/iteration_0_before_learning/user_notification_schedule_iter0.csv"
ITER1_SCHEDULE_PATH  = "/content/iteration_1_after_learning/user_notification_schedule_iter1.csv"
CLASSIFIED_PATH      = "/content/classified_results.csv"
TIMING0_PATH         = "/content/timing_recommendations.csv"
TIMING1_PATH         = "/content/iteration_1_after_learning/timing_recommendations_iter1.csv"
TEMPLATES_ITER1_PATH = "/content/iteration_1_after_learning/message_templates_iter1.csv"
BANDIT_STATE_PATH    = "/content/iteration_1_after_learning/bandit_state.json"
OUTPUT_DIR           = "/content/iteration_1_after_learning"

MAX_SLOTS               = 9
GUARDRAIL_THRESHOLD     = 0.02    # uninstall_rate > 2% triggers frequency guardrail


# ══════════════════════════════════════════════════════════════════
# STEP 1 — LOAD ALL INPUTS
# ══════════════════════════════════════════════════════════════════
def load_all(iter0_path, iter1_path, clf_path,
             timing0_path, timing1_path, tmpls_path, bandit_path):
    print("[1] Loading inputs...")

    def read(path, label):
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip().str.lower()
        print(f"    {label:<40} {df.shape[0]:>5} rows × {df.shape[1]} cols")
        return df

    s0      = read(iter0_path,    "iter0 schedule")
    s1      = read(iter1_path,    "iter1 schedule")
    clf     = read(clf_path,      "classified_results")
    t0      = read(timing0_path,  "timing_recommendations (iter0)")
    t1      = read(timing1_path,  "timing_recommendations (iter1)")
    tmpls   = read(tmpls_path,    "message_templates_iter1")

    with open(bandit_path) as f:
        bandit = json.load(f)
    print(f"    {'bandit_state.json':<40} {len(bandit['arms'])} arms")

    return s0, s1, clf, t0, t1, tmpls, bandit


# ══════════════════════════════════════════════════════════════════
# STEP 2 — BUILD LOOKUP TABLES
#
# Fast O(1) lookups for metrics, posteriors, and suppression status
# referenced when building explanation strings.
# ══════════════════════════════════════════════════════════════════
def build_lookups(clf, tmpls, bandit):
    print("\n[2] Building lookup tables...")

    # arm_metrics[(template_id, window)] → {ctr, engagement_rate, label, ...}
    arm_metrics = {}
    for _, row in clf.iterrows():
        key = (str(row["template_id"]), str(row["notification_window"]))
        arm_metrics[key] = {
            "ctr":             float(row.get("ctr", 0)),
            "engagement_rate": float(row.get("engagement_rate", 0)),
            "uninstall_rate":  float(row.get("uninstall_rate", 0)),
            "label":           str(row.get("label", "")),
            "reward":          float(row.get("reward", 0)),
            "total_sends":     int(row.get("total_sends", 0)),
        }

    # template_info[template_id] → {segment_id, performance_label, posterior_mean, suppressed, ...}
    template_info = {}
    for _, row in tmpls.iterrows():
        template_info[str(row["template_id"])] = {
            "segment_id":        str(row.get("segment_id", "")),
            "performance_label": str(row.get("performance_label", "")),
            "posterior_mean":    float(row.get("posterior_mean", 0.5)),
            "suppressed":        bool(row.get("suppressed", False)),
            "best_window":       str(row.get("best_window", "")),
            "best_ctr":          float(row.get("best_ctr", 0)),
        }

    # posterior[(template_id, window)] → posterior_mean from bandit_state
    posterior = {}
    for arm_key, arm_data in bandit["arms"].items():
        if "|" in arm_key:
            tmpl, window = arm_key.split("|", 1)
            posterior[(tmpl, window)] = float(arm_data.get("posterior_mean", 0.5))

    # fully_suppressed: set of template_ids where suppressed==True
    fully_suppressed = set(
        row["template_id"] for _, row in tmpls.iterrows()
        if bool(row.get("suppressed", False))
    )

    # segment_uninstall_rate[segment_id] → mean uninstall rate across all arms
    seg_uninstall = clf.groupby("segment_id")["uninstall_rate"].mean().to_dict()

    print(f"    arm_metrics keys    : {len(arm_metrics)}")
    print(f"    template_info keys  : {len(template_info)}")
    print(f"    posterior keys      : {len(posterior)}")
    print(f"    fully_suppressed    : {sorted(fully_suppressed)}")
    print(f"    guardrail segments  : {[s for s,r in seg_uninstall.items() if r > GUARDRAIL_THRESHOLD]}")

    return arm_metrics, template_info, posterior, fully_suppressed, seg_uninstall


# ══════════════════════════════════════════════════════════════════
# STEP 3 — CLASSIFY SLOT CHANGE TYPE
#
# Change type decision tree per (user, slot_pos):
#
#  Both null               → skip (no change)
#  Only iter1 is null      → slot_dropped  (frequency reduced mid-slot)
#  Only iter0 is null      → slot_added    (frequency increased)
#  template changed
#    AND window changed     → template_and_window_updated
#    ONLY template changed  → template_suppressed  (if old was fully suppressed)
#                             template_replaced     (if new is better posterior)
#  ONLY window changed      → window_shifted
#  Neither changed          → skip (no change)
# ══════════════════════════════════════════════════════════════════
def classify_slot_change(t0, w0, t1, w1, fully_suppressed):
    """Returns change_type string or None (if no change / skip)."""
    t0_null = pd.isna(t0) or str(t0).strip() in ("", "nan", "None")
    t1_null = pd.isna(t1) or str(t1).strip() in ("", "nan", "None")

    if t0_null and t1_null:
        return None          # both empty — inactive slot, no change

    if t0_null and not t1_null:
        return "slot_added"  # new slot appeared (frequency increased)

    if not t0_null and t1_null:
        return "slot_dropped"  # slot removed (frequency reduced)

    tmpl_changed = (str(t0).strip() != str(t1).strip())
    win_changed  = (str(w0).strip() != str(w1).strip())

    if tmpl_changed and win_changed:
        return "template_and_window_updated"

    if tmpl_changed:
        # Was old template fully suppressed?
        if str(t0) in fully_suppressed:
            return "template_suppressed"
        return "template_replaced"

    if win_changed:
        return "window_shifted"

    return None   # no change in this slot


# ══════════════════════════════════════════════════════════════════
# STEP 4 — BUILD EXPLANATION STRINGS
#
# All values come from real data — no hardcoding.
# Each change_type has its own template string filled with metrics.
# ══════════════════════════════════════════════════════════════════
def build_slot_explanation(change_type, t0, w0, t1, w1,
                            segment_id, arm_metrics, template_info, posterior):
    """Returns (metric_trigger, explanation) for a slot-level change row."""

    def arm_key(tmpl, window):
        return (str(tmpl).strip(), str(window).strip())

    def get_metric(tmpl, window, field, default=0.0):
        return arm_metrics.get(arm_key(tmpl, window), {}).get(field, default)

    def get_pm(tmpl, window):
        return posterior.get(arm_key(tmpl, window), 0.5)

    def get_label(tmpl, window):
        return arm_metrics.get(arm_key(tmpl, window), {}).get("label", "UNKNOWN")

    t0s, w0s = str(t0).strip(), str(w0).strip()
    t1s, w1s = str(t1).strip(), str(w1).strip()
    ti_new   = template_info.get(t1s, {})
    ti_old   = template_info.get(t0s, {})

    if change_type == "window_shifted":
        old_ctr   = get_metric(t0s, w0s, "ctr")
        old_label = get_label(t0s, w0s)
        new_pm    = get_pm(t0s, w1s)
        new_ctr   = get_metric(t0s, w1s, "ctr")
        metric    = f"old_arm_label={old_label};old_ctr={old_ctr:.3f};new_posterior={new_pm:.3f}"
        expl      = (
            f"{t0s} at {w0s} had CTR={old_ctr:.1%} "
            f"(label={old_label} in {segment_id}). "
            f"Thompson Sampling shifted to {w1s} "
            f"(posterior_mean={new_pm:.3f}, CTR={new_ctr:.1%})."
        )

    elif change_type == "template_and_window_updated":
        old_ctr   = get_metric(t0s, w0s, "ctr")
        old_label = get_label(t0s, w0s)
        new_pm    = ti_new.get("posterior_mean", 0.5)
        new_label = ti_new.get("performance_label", "")
        metric    = f"old_arm_label={old_label};old_ctr={old_ctr:.3f};new_pm={new_pm:.3f}"
        expl      = (
            f"{t0s} at {w0s} had CTR={old_ctr:.1%} "
            f"(label={old_label} in {segment_id}). "
            f"Replaced by {t1s} at {w1s} "
            f"(posterior_mean={new_pm:.3f}, label={new_label})."
        )

    elif change_type == "template_suppressed":
        old_ctr  = get_metric(t0s, w0s, "ctr")
        new_pm   = ti_new.get("posterior_mean", 0.5)
        metric   = f"old_template_suppressed=True;old_ctr={old_ctr:.3f}"
        expl     = (
            f"{t0s} was fully suppressed (all observed arms BAD in {segment_id}). "
            f"Replaced by {t1s} at {w1s} "
            f"(posterior_mean={new_pm:.3f})."
        )

    elif change_type == "template_replaced":
        old_pm   = ti_old.get("posterior_mean", 0.5)
        new_pm   = ti_new.get("posterior_mean", 0.5)
        old_label = ti_old.get("performance_label", "")
        new_label = ti_new.get("performance_label", "")
        metric   = f"old_pm={old_pm:.3f};new_pm={new_pm:.3f}"
        expl     = (
            f"{t0s} (posterior_mean={old_pm:.3f}, label={old_label}) "
            f"replaced by {t1s} "
            f"(posterior_mean={new_pm:.3f}, label={new_label}) "
            f"in {segment_id}."
        )

    elif change_type == "slot_dropped":
        metric = "slot_dropped_by_frequency_guardrail"
        expl   = (
            f"Slot was active in iter0 ({t0s} at {w0s}) but removed in iter1. "
            f"Frequency guardrail reduced total notifications for {segment_id}."
        )

    elif change_type == "slot_added":
        new_pm  = ti_new.get("posterior_mean", 0.5)
        metric  = f"new_pm={new_pm:.3f}"
        expl    = (
            f"New slot added in iter1: {t1s} at {w1s} "
            f"(posterior_mean={new_pm:.3f} in {segment_id})."
        )

    else:
        metric = ""
        expl   = f"Change type: {change_type}"

    return metric, expl


# ══════════════════════════════════════════════════════════════════
# STEP 5 — DIFF SCHEDULES: BUILD SLOT ROWS
#
# For each of 1000 users × 9 slots:
#   Compare (template, window) in iter0 vs iter1
#   If changed → build one delta row with explanation
#   If unchanged or both-null → skip
# ══════════════════════════════════════════════════════════════════
def build_slot_rows(s0, s1, arm_metrics, template_info,
                    posterior, fully_suppressed):
    print("\n[3] Building slot-level delta rows...")

    rows = []
    change_type_counts = defaultdict(int)

    # Build per-user segment lookup from iter0 (iter1 has the same values)
    user_segment = s0.set_index("user_id")["segment_id"].to_dict()

    merged = s0.merge(s1, on="user_id", suffixes=("_0", "_1"))

    for _, row in merged.iterrows():
        uid     = str(row["user_id"])
        seg     = str(user_segment.get(uid, ""))

        for slot in range(1, MAX_SLOTS + 1):
            t0_val = row.get(f"notif_{slot}_template_0")
            w0_val = row.get(f"notif_{slot}_time_0")
            t1_val = row.get(f"notif_{slot}_template_1")
            w1_val = row.get(f"notif_{slot}_time_1")

            change_type = classify_slot_change(
                t0_val, w0_val, t1_val, w1_val, fully_suppressed
            )

            if change_type is None:
                continue

            metric, expl = build_slot_explanation(
                change_type, t0_val, w0_val, t1_val, w1_val,
                seg, arm_metrics, template_info, posterior
            )

            before = (
                f"{str(t0_val).strip()}|{str(w0_val).strip()}"
                if not pd.isna(t0_val) else "null"
            )
            after = (
                f"{str(t1_val).strip()}|{str(w1_val).strip()}"
                if not pd.isna(t1_val) else "null"
            )

            rows.append({
                "entity_type":    "slot",
                "entity_id":      f"{uid}_slot{slot}",
                "segment_id":     seg,
                "slot_pos":       slot,
                "change_type":    change_type,
                "metric_trigger": metric,
                "before_value":   before,
                "after_value":    after,
                "explanation":    expl,
            })
            change_type_counts[change_type] += 1

    print(f"    Total slot rows      : {len(rows)}")
    for ct, cnt in sorted(change_type_counts.items()):
        print(f"      {ct:<35}: {cnt}")
    return rows


# ══════════════════════════════════════════════════════════════════
# STEP 6 — FREQUENCY CHANGE ROWS (per user)
#
# Count active slots (non-null notif_N_template) in iter0 vs iter1.
# If they differ → frequency_reduced or frequency_increased row.
# ══════════════════════════════════════════════════════════════════
def count_active_slots(df):
    counts = pd.Series(0, index=df.index)
    for slot in range(1, MAX_SLOTS + 1):
        col = f"notif_{slot}_template"
        if col in df.columns:
            counts += df[col].notna().astype(int)
    df = df.copy()
    df["_active"] = counts
    return df


def build_user_freq_rows(s0, s1, seg_uninstall):
    print("\n[4] Building user-level frequency change rows...")

    s0c = count_active_slots(s0).set_index("user_id")
    s1c = count_active_slots(s1).set_index("user_id")

    rows = []
    for uid in s0c.index:
        n0  = int(s0c.loc[uid, "_active"])
        n1  = int(s1c.loc[uid, "_active"]) if uid in s1c.index else n0
        if n0 == n1:
            continue

        seg = str(s0c.loc[uid, "segment_id"])
        uninstall_rate = seg_uninstall.get(seg, 0.0)

        if n1 < n0:
            change_type = "frequency_reduced"
            metric      = (
                f"segment_avg_uninstall_rate={uninstall_rate:.4f}"
                f";threshold={GUARDRAIL_THRESHOLD}"
            )
            expl = (
                f"Guardrail triggered: {seg} average uninstall_rate="
                f"{uninstall_rate:.1%} exceeded {GUARDRAIL_THRESHOLD:.0%} threshold. "
                f"Notifications reduced from {n0} to {n1}/day."
            )
        else:
            change_type = "frequency_increased"
            metric      = f"n0={n0};n1={n1}"
            expl        = (
                f"User {uid} had more active slots in iter1 ({n1}) than iter0 ({n0})."
            )

        rows.append({
            "entity_type":    "user",
            "entity_id":      uid,
            "segment_id":     seg,
            "slot_pos":       None,
            "change_type":    change_type,
            "metric_trigger": metric,
            "before_value":   str(n0),
            "after_value":    str(n1),
            "explanation":    expl,
        })

    print(f"    User frequency rows  : {len(rows)}")
    return rows


# ══════════════════════════════════════════════════════════════════
# STEP 7 — TEMPLATE SUPPRESSION ROWS (per fully suppressed template)
#
# One row per template that was fully suppressed (all arms BAD).
# Pulls min/max CTR from classified_results for the explanation.
# ══════════════════════════════════════════════════════════════════
def build_template_rows(clf, fully_suppressed):
    print("\n[5] Building template-level suppression rows...")

    rows = []
    for tmpl in sorted(fully_suppressed):
        tmpl_data = clf[clf["template_id"] == tmpl]
        if tmpl_data.empty:
            continue

        seg      = str(tmpl_data["segment_id"].iloc[0])
        n_arms   = len(tmpl_data)
        min_ctr  = float(tmpl_data["ctr"].min())
        max_ctr  = float(tmpl_data["ctr"].max())
        windows  = ", ".join(sorted(tmpl_data["notification_window"].unique()))

        metric = (
            f"all_arms_BAD=True;n_arms={n_arms}"
            f";min_ctr={min_ctr:.4f};max_ctr={max_ctr:.4f}"
        )
        expl = (
            f"{tmpl} fully suppressed in {seg}: "
            f"all {n_arms} observed arms (windows: {windows}) were BAD. "
            f"CTR range: {min_ctr:.1%}–{max_ctr:.1%}. "
            f"Template will not appear in iter1 schedule."
        )

        rows.append({
            "entity_type":    "template",
            "entity_id":      tmpl,
            "segment_id":     seg,
            "slot_pos":       None,
            "change_type":    "template_fully_suppressed",
            "metric_trigger": metric,
            "before_value":   "active",
            "after_value":    "suppressed",
            "explanation":    expl,
        })

    print(f"    Template suppression rows: {len(rows)}")
    return rows


# ══════════════════════════════════════════════════════════════════
# STEP 8 — TIMING SHIFT ROWS (per segment where top window changed)
#
# Compare preferred_window_1 in iter0 vs iter1 timing tables.
# If different → one timing row per segment.
# ══════════════════════════════════════════════════════════════════
def build_timing_rows(t0, t1):
    print("\n[6] Building timing-level shift rows...")

    t0_idx = t0.set_index("segment_id")
    t1_idx = t1.set_index("segment_id")

    rows = []
    for seg in t0_idx.index:
        if seg not in t1_idx.index:
            continue

        w0_top  = str(t0_idx.loc[seg, "preferred_window_1"])
        w1_top  = str(t1_idx.loc[seg, "preferred_window_1"])
        ctr0    = float(t0_idx.loc[seg, "expected_ctr_1"])
        ctr1    = float(t1_idx.loc[seg, "expected_ctr_1"])

        if w0_top == w1_top:
            continue  # no shift in top window

        direction = "improved" if ctr1 >= ctr0 else "changed"
        pct_delta = (ctr1 - ctr0) * 100

        metric = (
            f"iter0_top_window={w0_top};iter0_ctr={ctr0:.4f}"
            f";iter1_top_window={w1_top};iter1_ctr={ctr1:.4f}"
        )
        expl = (
            f"{seg} top notification window shifted from {w0_top} to {w1_top}. "
            f"Expected CTR {direction}: {ctr0:.1%} → {ctr1:.1%} "
            f"({pct_delta:+.1f}pp). "
            f"Beta posteriors confirmed {w1_top} as higher-performing window."
        )

        rows.append({
            "entity_type":    "timing",
            "entity_id":      f"timing_{seg.replace(' ', '_')}",
            "segment_id":     seg,
            "slot_pos":       None,
            "change_type":    "top_window_shifted",
            "metric_trigger": metric,
            "before_value":   f"{w0_top}|ctr={ctr0:.4f}",
            "after_value":    f"{w1_top}|ctr={ctr1:.4f}",
            "explanation":    expl,
        })

    print(f"    Timing shift rows    : {len(rows)}")
    return rows


# ══════════════════════════════════════════════════════════════════
# STEP 9 — SEGMENT SUMMARY ROWS (one per segment)
#
# Aggregate all changes for each segment:
#   - total slot changes
#   - breakdown by change_type
#   - CTR direction from timing shift
# ══════════════════════════════════════════════════════════════════
def build_segment_rows(all_rows_so_far, t0, t1):
    print("\n[7] Building segment-level summary rows...")

    df = pd.DataFrame(all_rows_so_far)
    t0_idx = t0.set_index("segment_id")
    t1_idx = t1.set_index("segment_id")

    segments = t0["segment_id"].unique()
    rows = []

    for seg in sorted(segments):
        seg_df = df[df["segment_id"] == seg]

        n_slot_changes = len(seg_df[seg_df["entity_type"] == "slot"])
        n_window_shifts = len(seg_df[
            (seg_df["entity_type"] == "slot") &
            (seg_df["change_type"] == "window_shifted")
        ])
        n_tmpl_updates = len(seg_df[
            (seg_df["entity_type"] == "slot") &
            (seg_df["change_type"].isin([
                "template_and_window_updated",
                "template_suppressed",
                "template_replaced"
            ]))
        ])
        n_freq_reductions = len(seg_df[
            (seg_df["entity_type"] == "user") &
            (seg_df["change_type"] == "frequency_reduced")
        ])
        n_suppressed = len(seg_df[
            (seg_df["entity_type"] == "template") &
            (seg_df["change_type"] == "template_fully_suppressed")
        ])

        # CTR change from timing tables
        ctr0 = float(t0_idx.loc[seg, "expected_ctr_1"]) if seg in t0_idx.index else 0.0
        ctr1 = float(t1_idx.loc[seg, "expected_ctr_1"]) if seg in t1_idx.index else 0.0
        pct  = (ctr1 - ctr0) * 100

        metric = (
            f"slot_changes={n_slot_changes}"
            f";window_shifts={n_window_shifts}"
            f";template_updates={n_tmpl_updates}"
            f";freq_reductions={n_freq_reductions}"
            f";templates_suppressed={n_suppressed}"
            f";ctr_delta_pp={pct:+.2f}"
        )

        direction = "improved" if pct > 0 else "declined" if pct < 0 else "unchanged"
        expl = (
            f"{seg}: {n_slot_changes} slot-level changes "
            f"({n_window_shifts} window shifts, {n_tmpl_updates} template updates). "
            f"{n_freq_reductions} users had frequency reduced by guardrail. "
            f"{n_suppressed} templates fully suppressed. "
            f"Top-window expected CTR {direction}: "
            f"{ctr0:.1%} → {ctr1:.1%} ({pct:+.1f}pp)."
        )

        rows.append({
            "entity_type":    "segment",
            "entity_id":      seg,
            "segment_id":     seg,
            "slot_pos":       None,
            "change_type":    "segment_summary",
            "metric_trigger": metric,
            "before_value":   f"ctr={ctr0:.4f}",
            "after_value":    f"ctr={ctr1:.4f}",
            "explanation":    expl,
        })

    print(f"    Segment summary rows : {len(rows)}")
    return rows


# ══════════════════════════════════════════════════════════════════
# STEP 10 — VALIDATE OUTPUT
# ══════════════════════════════════════════════════════════════════
def validate_report(report_df, s0, s1):
    print("\n[8] Validating delta report...")
    ok = True

    # Required columns all present
    required = [
        "entity_type", "entity_id", "segment_id", "slot_pos",
        "change_type", "metric_trigger", "before_value",
        "after_value", "explanation"
    ]
    missing = [c for c in required if c not in report_df.columns]
    if missing:
        print(f"  ⚠  Missing columns: {missing}")
        ok = False
    else:
        print(f"  ✅ All {len(required)} required columns present")

    # No empty entity_ids
    null_ids = report_df["entity_id"].isna().sum()
    if null_ids:
        print(f"  ⚠  {null_ids} rows have null entity_id")
        ok = False
    else:
        print("  ✅ No null entity_id values")

    # No empty explanations
    null_expl = report_df["explanation"].isna().sum()
    empty_expl = (report_df["explanation"].astype(str).str.strip() == "").sum()
    if null_expl + empty_expl:
        print(f"  ⚠  {null_expl + empty_expl} rows have empty explanation")
        ok = False
    else:
        print("  ✅ All explanations populated")

    # Entity types valid
    valid_types = {"slot", "user", "template", "timing", "segment"}
    bad_types = set(report_df["entity_type"].unique()) - valid_types
    if bad_types:
        print(f"  ⚠  Unknown entity_types: {bad_types}")
        ok = False
    else:
        print(f"  ✅ Entity types valid: {set(report_df['entity_type'].unique())}")

    # Segment summary: one row per segment
    seg_rows = report_df[report_df["entity_type"] == "segment"]
    print(f"  ✅ Segment summary rows: {len(seg_rows)} (expected 10)")

    # Frequency reduced: matches guardrail expectation
    freq_rows = report_df[
        (report_df["entity_type"] == "user") &
        (report_df["change_type"] == "frequency_reduced")
    ]
    print(f"  ✅ Frequency_reduced rows: {len(freq_rows)}")

    if ok:
        print("  ✅ All validation checks passed")
    return ok


# ══════════════════════════════════════════════════════════════════
# STEP 11 — PRINT SUMMARY
# ══════════════════════════════════════════════════════════════════
def print_summary(report_df):
    print("\n" + "═"*65)
    print("DELTA REPORT SUMMARY")
    print("═"*65)

    print(f"\n  Total rows in report: {len(report_df)}")

    print("\n  By entity_type:")
    for et, cnt in report_df["entity_type"].value_counts().items():
        print(f"    {et:<12} {cnt:>5} rows")

    print("\n  By change_type:")
    for ct, cnt in report_df["change_type"].value_counts().items():
        bar = "█" * int(cnt / 80)
        print(f"    {ct:<40} {cnt:>5}  {bar}")

    print("\n  Segment impact summary:")
    seg_rows = report_df[report_df["entity_type"] == "segment"].sort_values("segment_id")
    for _, row in seg_rows.iterrows():
        print(f"    {row['segment_id']:<30}  "
              f"{row['before_value']} → {row['after_value']}")


# ══════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════
def run(
    iter0_schedule_path  = ITER0_SCHEDULE_PATH,
    iter1_schedule_path  = ITER1_SCHEDULE_PATH,
    classified_path      = CLASSIFIED_PATH,
    timing0_path         = TIMING0_PATH,
    timing1_path         = TIMING1_PATH,
    templates_iter1_path = TEMPLATES_ITER1_PATH,
    bandit_state_path    = BANDIT_STATE_PATH,
    output_dir           = OUTPUT_DIR,
):
    """
    Run the delta reporter. Accepts custom paths for demo day runtime.
    Returns the final report DataFrame.
    """
    print("=" * 65)
    print("TASK 3 — PART 4: DELTA REPORTER")
    print("=" * 65)

    os.makedirs(output_dir, exist_ok=True)

    # Load
    s0, s1, clf, t0, t1, tmpls, bandit = load_all(
        iter0_schedule_path, iter1_schedule_path, classified_path,
        timing0_path, timing1_path, templates_iter1_path, bandit_state_path
    )

    # Build lookups
    arm_metrics, template_info, posterior, fully_suppressed, seg_uninstall = \
        build_lookups(clf, tmpls, bandit)

    # Build each entity type
    slot_rows     = build_slot_rows(
        s0, s1, arm_metrics, template_info, posterior, fully_suppressed
    )
    user_rows     = build_user_freq_rows(s0, s1, seg_uninstall)
    template_rows = build_template_rows(clf, fully_suppressed)
    timing_rows   = build_timing_rows(t0, t1)

    # Combine first four types before building segment summaries
    # (segment summaries aggregate over the others)
    partial = slot_rows + user_rows + template_rows + timing_rows
    segment_rows = build_segment_rows(partial, t0, t1)

    # Assemble final report
    all_rows  = partial + segment_rows
    report_df = pd.DataFrame(all_rows, columns=[
        "entity_type", "entity_id", "segment_id", "slot_pos",
        "change_type", "metric_trigger", "before_value",
        "after_value", "explanation"
    ])

    # Sort: segment summaries last; within slot rows: by user → slot_pos
    type_order = {"slot": 0, "user": 1, "template": 2, "timing": 3, "segment": 4}
    report_df["_sort"] = report_df["entity_type"].map(type_order)
    report_df = report_df.sort_values(
        ["_sort", "segment_id", "entity_id", "slot_pos"]
    ).drop(columns=["_sort"]).reset_index(drop=True)

    # Validate
    validate_report(report_df, s0, s1)

    # Save
    out_path = os.path.join(output_dir, "learning_delta_report.csv")
    report_df.to_csv(out_path, index=False)

    print(f"\n[9] Saved: {out_path}")
    print(f"           {len(report_df)} rows × {len(report_df.columns)} cols")

    print_summary(report_df)

    print("\n" + "=" * 65)
    print("✅ PART 4 COMPLETE — Task 3 fully done.")
    print("=" * 65)

    return report_df


if __name__ == "__main__":
    run()

TASK 3 — PART 4: DELTA REPORTER
[1] Loading inputs...
    iter0 schedule                              61 rows × 30 cols


FileNotFoundError: [Errno 2] No such file or directory: '/content/iteration_1_after_learning/user_notification_schedule_iter1.csv'